<a href="https://colab.research.google.com/github/IhabAltekreeti/vaultify/blob/main/vaultify_compact_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
# VAULTIFY COMPACT - Cell 1: Install all runtime dependencies

%pip install -q \
    qdrant-client \
    groq \
    docling \
    "rapidocr<3.8" \
    onnxruntime \
    "mcp[cli]" \
    flask \
    flask-sqlalchemy \
    flask-login \
    flask-wtf \
    flask-migrate \
    flask-limiter \
    email-validator \
    filetype \
    python-dotenv \
    sentence-transformers

print("✅ Vaultify runtime dependencies installed.")


✅ Vaultify runtime dependencies installed.


In [44]:
# VAULTIFY COMPACT - Cell 2: Load secrets and define the central configuration

from pathlib import Path

from google.colab import userdata

REQUIRED_SECRETS = [
    "QDRANT_URL",
    "QDRANT_API_KEY",
    "GROQ_API_KEY",
]

OPTIONAL_SECRETS = [
    "NGROK_TOKEN",
]

loaded_secrets = {}
missing_secrets = []

for secret_name in REQUIRED_SECRETS + OPTIONAL_SECRETS:
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None

    if secret_value:
        loaded_secrets[secret_name] = secret_value
        print(f"✅ {secret_name}: Found")
    elif secret_name in REQUIRED_SECRETS:
        missing_secrets.append(secret_name)
        print(f"❌ {secret_name}: Missing")
    else:
        print(f"ℹ️ {secret_name}: Optional and not configured")

if missing_secrets:
    raise RuntimeError(
        "Missing required Colab Secrets: "
        + ", ".join(missing_secrets)
    )

QDRANT_URL = loaded_secrets["QDRANT_URL"]
QDRANT_API_KEY = loaded_secrets["QDRANT_API_KEY"]
GROQ_API_KEY = loaded_secrets["GROQ_API_KEY"]
NGROK_TOKEN = loaded_secrets.get("NGROK_TOKEN")

PROJECT_NAME = "Vaultify"
PROJECT_VERSION = "3.0.0"

COLLECTION_NAME = "vaultify_v3_documents"

TENANT_ID_FIELD = "tenant_id"
DOCUMENT_HASH_FIELD = "document_hash"
FILENAME_FIELD = "filename"
CHUNK_INDEX_FIELD = "chunk_index"
CHUNK_TYPE_FIELD = "chunk_type"
TEXT_FIELD = "text"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_SIZE = 384
LLM_MODEL = "llama-3.3-70b-versatile"

PROJECT_ROOT = Path("/content/vaultify_v3")
UPLOADS_DIR = PROJECT_ROOT / "uploads"
CACHE_DIR = PROJECT_ROOT / "cache"

for directory in [PROJECT_ROOT, UPLOADS_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("\n✅ Vaultify configuration loaded.")
print(f"📁 Runtime workspace: {PROJECT_ROOT}")


✅ QDRANT_URL: Found
✅ QDRANT_API_KEY: Found
✅ GROQ_API_KEY: Found
✅ NGROK_TOKEN: Found

✅ Vaultify configuration loaded.
📁 Runtime workspace: /content/vaultify_v3


In [45]:
# VAULTIFY COMPACT - Cell 3: Verify the runtime and initialize cloud clients

import platform
import sys

import torch
from groq import Groq
from qdrant_client import QdrantClient

print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🖥️ Platform: {platform.platform()}")
print(f"⚡ CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is unavailable. Select Runtime > Change runtime type > T4 GPU."
    )

print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

existing_collections = {
    collection.name
    for collection in qdrant.get_collections().collections
}

print("✅ Qdrant Cloud connection successful.")
print(f"📦 Collections: {sorted(existing_collections) or 'None'}")

groq_client = Groq(api_key=GROQ_API_KEY)

available_model_ids = {
    model.id
    for model in groq_client.models.list().data
}

if LLM_MODEL not in available_model_ids:
    raise RuntimeError(
        f"Groq model is unavailable: {LLM_MODEL}"
    )

print("✅ Groq Cloud connection successful.")
print(f"🤖 LLM model: {LLM_MODEL}")


🐍 Python: 3.12.13
🖥️ Platform: Linux-6.6.122+-x86_64-with-glibc2.35
⚡ CUDA available: True
🎮 GPU: Tesla T4
✅ Qdrant Cloud connection successful.
📦 Collections: ['vaultify_documents', 'vaultify_v3_documents']
✅ Groq Cloud connection successful.
🤖 LLM model: llama-3.3-70b-versatile


In [46]:
# VAULTIFY COMPACT - Cell 4: Load the embedding model and prepare Qdrant

from sentence_transformers import SentenceTransformer
from qdrant_client.models import (
    Distance,
    PayloadSchemaType,
    VectorParams,
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda",
)

validation_embeddings = embedding_model.encode(
    [
        "Vaultify searches private company documents.",
        "Financial reports contain tables and numerical data.",
    ],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

if validation_embeddings.shape[1] != VECTOR_SIZE:
    raise RuntimeError(
        "The embedding dimension does not match the configured vector size."
    )

existing_collections = {
    collection.name
    for collection in qdrant.get_collections().collections
}

if COLLECTION_NAME not in existing_collections:
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE,
        ),
    )
    print(f"✅ Created collection: {COLLECTION_NAME}")
else:
    print(f"ℹ️ Collection already exists: {COLLECTION_NAME}")

collection_info = qdrant.get_collection(COLLECTION_NAME)

for field_name in [TENANT_ID_FIELD, DOCUMENT_HASH_FIELD]:
    if field_name not in (collection_info.payload_schema or {}):
        qdrant.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name=field_name,
            field_schema=PayloadSchemaType.KEYWORD,
            wait=True,
        )
        print(f"✅ Created payload index: {field_name}")
    else:
        print(f"ℹ️ Payload index already exists: {field_name}")

print(f"📐 Embedding shape: {validation_embeddings.shape}")
print("✅ Embedding and Qdrant setup completed.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ℹ️ Collection already exists: vaultify_v3_documents
ℹ️ Payload index already exists: tenant_id
ℹ️ Payload index already exists: document_hash
📐 Embedding shape: (2, 384)
✅ Embedding and Qdrant setup completed.


In [47]:
# VAULTIFY COMPACT - Cell 5: Inspect existing tenant data and choose the next path

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)

DEMO_TENANTS = {
    "company_a": {
        "tenant_id": "demo_apple_tenant",
        "display_name": "Apple Financial Demo",
    },
    "company_b": {
        "tenant_id": "demo_tesla_tenant",
        "display_name": "Tesla Financial Demo",
    },
}

tenant_counts = {}

for tenant_key, tenant in DEMO_TENANTS.items():
    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant["tenant_id"]
                ),
            )
        ]
    )

    tenant_counts[tenant_key] = qdrant.count(
        collection_name=COLLECTION_NAME,
        count_filter=tenant_filter,
        exact=True,
    ).count

total_point_count = qdrant.count(
    collection_name=COLLECTION_NAME,
    exact=True,
).count

print("📊 Existing Qdrant data:")
print(f"   Apple tenant: {tenant_counts['company_a']} chunks")
print(f"   Tesla tenant: {tenant_counts['company_b']} chunks")
print(f"   Entire collection: {total_point_count} points")

SEED_DATA_READY = (
    tenant_counts["company_a"] > 0
    and tenant_counts["company_b"] > 0
)

if SEED_DATA_READY:
    print("\n✅ Existing demo vectors were found.")
    print("⏭️ Seed PDF parsing and indexing can be skipped.")
    print("➡️ Next: restore retrieval, MCP, Flask, and Cloudflare services.")
else:
    print("\n⚠️ Demo vectors are incomplete or missing.")
    print("➡️ Next: run the compact ingestion cells before starting services.")


📊 Existing Qdrant data:
   Apple tenant: 609 chunks
   Tesla tenant: 140 chunks
   Entire collection: 1042 points

✅ Existing demo vectors were found.
⏭️ Seed PDF parsing and indexing can be skipped.
➡️ Next: restore retrieval, MCP, Flask, and Cloudflare services.


In [48]:
# VAULTIFY COMPACT - Cell 6: Restore tenant-scoped retrieval and grounded answers

from qdrant_client.models import FieldCondition, Filter, MatchValue

if not SEED_DATA_READY:
    raise RuntimeError(
        "The demo vectors are missing. Run the ingestion path before retrieval."
    )

RETRIEVAL_CANDIDATE_LIMIT = 12
FINAL_CONTEXT_LIMIT = 6
MAX_ANSWER_TOKENS = 400


def search_tenant_documents(
    query,
    tenant_id,
    limit=RETRIEVAL_CANDIDATE_LIMIT,
):
    """
    Search only the document chunks that belong to one trusted tenant.
    """
    cleaned_query = query.strip()

    if not cleaned_query:
        raise ValueError("The search query cannot be empty.")

    if not tenant_id:
        raise ValueError("A tenant ID is required.")

    if limit <= 0:
        raise ValueError("The retrieval limit must be greater than zero.")

    query_vector = embedding_model.encode(
        cleaned_query,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    if query_vector.shape[0] != VECTOR_SIZE:
        raise RuntimeError(
            "The query embedding dimension does not match Qdrant."
        )

    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(value=tenant_id),
            )
        ]
    )

    response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        query_filter=tenant_filter,
        limit=limit,
        with_payload=True,
    )

    return response.points


def build_retrieval_context(
    results,
    context_limit=FINAL_CONTEXT_LIMIT,
):
    """
    Convert Qdrant results into a structured context for the language model.
    """
    context_sections = []

    for result_index, result in enumerate(
        results[:context_limit],
        start=1,
    ):
        payload = result.payload or {}
        chunk_text = str(
            payload.get(TEXT_FIELD, "")
        ).strip()

        if not chunk_text:
            continue

        context_sections.append(
            "\n".join(
                [
                    f"[Context {result_index}]",
                    f"File: {payload.get(FILENAME_FIELD, 'Unknown file')}",
                    f"Section: {payload.get('section', 'Unknown section')}",
                    f"Chunk type: {payload.get(CHUNK_TYPE_FIELD, 'unknown')}",
                    f"Similarity score: {float(result.score):.4f}",
                    "Content:",
                    chunk_text,
                ]
            )
        )

    return "\n\n---\n\n".join(context_sections)


def answer_tenant_question(
    question,
    tenant_id,
    retrieval_limit=RETRIEVAL_CANDIDATE_LIMIT,
):
    """
    Retrieve tenant-isolated evidence and generate an evidence-only answer.
    """
    cleaned_question = question.strip()

    if not cleaned_question:
        raise ValueError("The question cannot be empty.")

    results = search_tenant_documents(
        query=cleaned_question,
        tenant_id=tenant_id,
        limit=retrieval_limit,
    )

    context = build_retrieval_context(results)

    if not context:
        return {
            "answer": (
                "The requested information was not found "
                "in the tenant's indexed documents."
            ),
            "results": results,
        }

    system_prompt = """
You are Vaultify, a document-grounded business assistant.

Rules:
1. Use only the supplied document context.
2. Never use outside knowledge or guess missing facts.
3. Keep company names, reporting periods, units, and totals exact.
4. When evidence is insufficient, clearly say the information was not found.
5. Do not reveal or discuss documents that are absent from the supplied context.
6. Give a concise answer and mention the source file or section when useful.
""".strip()

    user_prompt = f"""
Question:
{cleaned_question}

Document context:
{context}
""".strip()

    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0,
        max_tokens=MAX_ANSWER_TOKENS,
    )

    answer = response.choices[0].message.content.strip()

    if not answer:
        raise RuntimeError("Groq returned an empty answer.")

    return {
        "answer": answer,
        "results": results,
    }


grounded_tests = [
    {
        "name": "Apple FY2025 net sales",
        "tenant_id": DEMO_TENANTS["company_a"]["tenant_id"],
        "question": (
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        "expected_filename": "apple_fy2025_10k.pdf",
    },
    {
        "name": "Tesla Q4 2025 revenue",
        "tenant_id": DEMO_TENANTS["company_b"]["tenant_id"],
        "question": (
            "What was Tesla's total revenue "
            "in the fourth quarter of 2025?"
        ),
        "expected_filename": "tesla_q4_2025_update.pdf",
    },
]

grounded_test_outputs = {}

for test in grounded_tests:
    result = answer_tenant_question(
        question=test["question"],
        tenant_id=test["tenant_id"],
    )

    returned_filenames = {
        item.payload.get(FILENAME_FIELD)
        for item in result["results"]
        if item.payload
    }

    unexpected_filenames = (
        returned_filenames
        - {test["expected_filename"]}
    )

    if unexpected_filenames:
        raise RuntimeError(
            "Tenant isolation failed. Unexpected files: "
            f"{sorted(unexpected_filenames)}"
        )

    grounded_test_outputs[test["name"]] = result

    print("\n" + "=" * 88)
    print(f"🧪 {test['name']}")
    print(f"🏢 Tenant: {test['tenant_id']}")
    print(f"📄 Retrieved files: {sorted(returned_filenames)}")
    print("🤖 Answer:")
    print(result["answer"])

print("\n✅ Tenant-scoped retrieval restored.")
print("✅ Apple and Tesla grounded-answer tests passed.")
print("✅ No cross-tenant document appeared in either retrieval result.")



🧪 Apple FY2025 net sales
🏢 Tenant: demo_apple_tenant
📄 Retrieved files: ['apple_fy2025_10k.pdf']
🤖 Answer:
According to the table in Section: Note 2 - Revenue of the apple_fy2025_10k.pdf file, Apple's total net sales in fiscal year 2025 were $416,161. (Source: Context 3, Section: Note 2 - Revenue)

🧪 Tesla Q4 2025 revenue
🏢 Tenant: demo_tesla_tenant
📄 Retrieved files: ['tesla_q4_2025_update.pdf']
🤖 Answer:
Tesla's total revenue in the fourth quarter of 2025 was $24,901 million. 
Source: File: tesla_q4_2025_update.pdf, Section: (Unaudited) table.

✅ Tenant-scoped retrieval restored.
✅ Apple and Tesla grounded-answer tests passed.
✅ No cross-tenant document appeared in either retrieval result.


In [49]:
# VAULTIFY COMPACT - Cell 7: Define and test the tenant-bound MCP tool in memory

import json
from importlib.metadata import version
from typing import Any

from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.exceptions import ToolError
from mcp.server.transport_security import TransportSecuritySettings
from mcp.shared.memory import (
    create_connected_server_and_client_session,
)

MCP_HOST = "127.0.0.1"
MCP_PORT = 8003
MCP_SERVER_NAME = "Vaultify"
MCP_RETRIEVAL_LIMIT = 5

ACTIVE_DEMO_TENANT_KEY = "company_a"
ACTIVE_DEMO_TENANT_ID = (
    DEMO_TENANTS[ACTIVE_DEMO_TENANT_KEY]["tenant_id"]
)
ACTIVE_DEMO_FILENAME = "apple_fy2025_10k.pdf"

LOCAL_MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

vaultify_mcp = FastMCP(
    MCP_SERVER_NAME,
    host=MCP_HOST,
    port=MCP_PORT,
    stateless_http=True,
    json_response=True,
    transport_security=TransportSecuritySettings(
        enable_dns_rebinding_protection=False,
    ),
)


def serialize_retrieval_sources(results):
    """
    Convert Qdrant results into non-sensitive MCP source metadata.
    """
    sources = []

    for result in results:
        payload = result.payload or {}

        sources.append(
            {
                "filename": payload.get(
                    FILENAME_FIELD,
                    "Unknown file",
                ),
                "section": payload.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": payload.get(
                    CHUNK_TYPE_FIELD,
                    "unknown",
                ),
                "chunk_index": payload.get(
                    CHUNK_INDEX_FIELD,
                ),
                "similarity_score": round(
                    float(result.score),
                    4,
                ),
            }
        )

    return sources


def parse_mcp_tool_result(tool_result):
    """
    Parse structured MCP output or the JSON text fallback.
    """
    if tool_result.isError:
        error_messages = [
            item.text
            for item in tool_result.content
            if hasattr(item, "text")
        ]

        raise RuntimeError(
            "The MCP tool returned an error. "
            + " ".join(error_messages)
        )

    structured = tool_result.structuredContent

    if isinstance(structured, dict):
        if isinstance(structured.get("result"), dict):
            return structured["result"]

        return structured

    text_blocks = [
        item.text.strip()
        for item in tool_result.content
        if hasattr(item, "text")
        and item.text.strip()
    ]

    for text_block in text_blocks:
        try:
            parsed = json.loads(text_block)
        except json.JSONDecodeError:
            continue

        if isinstance(parsed, dict):
            if isinstance(parsed.get("result"), dict):
                return parsed["result"]

            return parsed

    raise RuntimeError(
        "The MCP response could not be parsed as structured JSON."
    )


def validate_mcp_payload(payload):
    """
    Validate the answer, trusted tenant, and tenant-owned source files.
    """
    answer = str(payload.get("answer", "")).strip()
    tenant_id = payload.get("tenant_id")
    sources = payload.get("sources", [])

    if not answer:
        raise RuntimeError("The MCP response contained no answer.")

    if tenant_id != ACTIVE_DEMO_TENANT_ID:
        raise RuntimeError(
            "The MCP response returned the wrong tenant."
        )

    if not isinstance(sources, list):
        raise RuntimeError("The MCP sources field is invalid.")

    unexpected_files = {
        source.get("filename")
        for source in sources
        if isinstance(source, dict)
        and source.get("filename") != ACTIVE_DEMO_FILENAME
    }

    if unexpected_files:
        raise RuntimeError(
            "The MCP response leaked unexpected files: "
            f"{sorted(unexpected_files)}"
        )

    return {
        "answer": answer,
        "tenant_id": tenant_id,
        "sources": sources,
    }


@vaultify_mcp.tool()
def ask_documents(question: str) -> dict[str, Any]:
    """
    Answer using only the server-configured tenant's indexed documents.
    """
    cleaned_question = question.strip()

    if not cleaned_question:
        raise ToolError("The question cannot be empty.")

    try:
        result = answer_tenant_question(
            question=cleaned_question,
            tenant_id=ACTIVE_DEMO_TENANT_ID,
            retrieval_limit=MCP_RETRIEVAL_LIMIT,
        )

    except Exception as error:
        raise ToolError(
            "Vaultify could not process the document question."
        ) from error

    return {
        "answer": result["answer"],
        "tenant_id": ACTIVE_DEMO_TENANT_ID,
        "sources": serialize_retrieval_sources(
            result["results"]
        ),
    }


MCP_TEST_QUESTION = (
    "What were Apple's total net sales "
    "in fiscal year 2025?"
)

async with create_connected_server_and_client_session(
    vaultify_mcp,
    raise_exceptions=True,
) as session:
    tools = await session.list_tools()
    tool_names = [tool.name for tool in tools.tools]

    if "ask_documents" not in tool_names:
        raise RuntimeError("The ask_documents tool was not registered.")

    in_memory_tool_result = await session.call_tool(
        "ask_documents",
        {"question": MCP_TEST_QUESTION},
    )

in_memory_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        in_memory_tool_result
    )
)

print(f"✅ MCP SDK version: {version('mcp')}")
print(f"✅ Registered tools: {tool_names}")
print(f"✅ Bound tenant: {in_memory_payload['tenant_id']}")
print(f"✅ Returned sources: {len(in_memory_payload['sources'])}")
print("\n🤖 In-memory MCP answer:")
print(in_memory_payload["answer"])


✅ MCP SDK version: 1.28.1
✅ Registered tools: ['ask_documents']
✅ Bound tenant: demo_apple_tenant
✅ Returned sources: 5

🤖 In-memory MCP answer:
According to the table in Section: Note 2 - Revenue of the apple_fy2025_10k.pdf file, Apple's total net sales in fiscal year 2025 were $416,161. 

Source: File: apple_fy2025_10k.pdf, Section: Note 2 - Revenue.


In [50]:
# VAULTIFY COMPACT - Cell 8: Start and test the local Streamable HTTP MCP server

import socket
import threading
import time

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def is_port_open(host, port):
    """
    Return True when a TCP server is listening on the target port.
    """
    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    ) as connection_socket:
        connection_socket.settimeout(0.5)

        return (
            connection_socket.connect_ex((host, port))
            == 0
        )


def run_vaultify_mcp_server():
    """
    Run the stateless JSON Streamable HTTP server.
    """
    vaultify_mcp.run(
        transport="streamable-http"
    )


if is_port_open(MCP_HOST, MCP_PORT):
    print(
        f"ℹ️ Port {MCP_PORT} is already open. "
        "The existing local MCP server will be reused."
    )
else:
    mcp_server_thread = threading.Thread(
        target=run_vaultify_mcp_server,
        name="vaultify-mcp-server",
        daemon=True,
    )

    mcp_server_thread.start()

    startup_deadline = time.time() + 15

    while (
        time.time() < startup_deadline
        and not is_port_open(MCP_HOST, MCP_PORT)
    ):
        time.sleep(0.25)

if not is_port_open(MCP_HOST, MCP_PORT):
    raise RuntimeError(
        f"The MCP server did not start on port {MCP_PORT}."
    )


async def test_local_mcp_endpoint():
    """
    Run initialization, discovery, and one tool call through local HTTP.
    """
    async with streamablehttp_client(
        LOCAL_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization = await session.initialize()
            tools = await session.list_tools()
            tool_names = [
                tool.name
                for tool in tools.tools
            ]

            tool_result = await session.call_tool(
                "ask_documents",
                {"question": MCP_TEST_QUESTION},
            )

            return {
                "server_name": initialization.serverInfo.name,
                "server_version": initialization.serverInfo.version,
                "tool_names": tool_names,
                "tool_result": tool_result,
            }


local_test = await test_local_mcp_endpoint()

local_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        local_test["tool_result"]
    )
)

print("✅ Local Streamable HTTP MCP test passed.")
print(f"🌐 Endpoint: {LOCAL_MCP_URL}")
print(
    f"📦 Server: "
    f"{local_test['server_name']} "
    f"{local_test['server_version']}"
)
print(f"🛠️ Tools: {local_test['tool_names']}")
print(f"🔐 Tenant: {local_payload['tenant_id']}")
print(f"📚 Sources: {len(local_payload['sources'])}")


ℹ️ Port 8003 is already open. The existing local MCP server will be reused.
INFO:     127.0.0.1:56696 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:56696 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:56710 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:56710 - "POST /mcp HTTP/1.1" 200 OK
✅ Local Streamable HTTP MCP test passed.
🌐 Endpoint: http://127.0.0.1:8003/mcp
📦 Server: Vaultify 1.28.1
🛠️ Tools: ['ask_documents']
🔐 Tenant: demo_apple_tenant
📚 Sources: 5


In [51]:
# VAULTIFY COMPACT - Cell 9: Start a Cloudflare Quick Tunnel for MCP

import re
import shutil
import subprocess
import threading
import time

CLOUDFLARED_BINARY = "cloudflared"
CLOUDFLARE_STARTUP_TIMEOUT_SECONDS = 60


def install_cloudflared_if_missing():
    """
    Install the Cloudflare tunnel client in the Colab runtime.
    """
    if shutil.which(CLOUDFLARED_BINARY):
        return

    install_command = """
set -e
wget -q \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
  -O /tmp/cloudflared.deb
dpkg -i /tmp/cloudflared.deb >/dev/null
rm -f /tmp/cloudflared.deb
"""

    subprocess.run(
        ["bash", "-lc", install_command],
        check=True,
    )


def stream_cloudflared_logs(process, storage):
    """
    Store Cloudflare logs while displaying useful startup messages.
    """
    if process.stderr is None:
        return

    for line in iter(process.stderr.readline, ""):
        cleaned_line = line.strip()

        if cleaned_line:
            storage.append(cleaned_line)
            print(f"[cloudflared] {cleaned_line}")

        if process.poll() is not None:
            break


def extract_trycloudflare_url(log_lines):
    """
    Extract the generated public hostname from Cloudflare logs.
    """
    pattern = re.compile(
        r"https://[a-zA-Z0-9-]+"
        r"\.trycloudflare\.com"
    )

    for line in log_lines:
        match = pattern.search(line)

        if match:
            return match.group(0)

    return None


def tunnel_is_registered(log_lines):
    """
    Return True after Cloudflare registers the tunnel connection.
    """
    return any(
        "Registered tunnel connection" in line
        for line in log_lines
    )


install_cloudflared_if_missing()

if not is_port_open(MCP_HOST, MCP_PORT):
    raise RuntimeError(
        "The local MCP server is not running."
    )

if (
    "cloudflare_tunnel_process" in globals()
    and cloudflare_tunnel_process.poll() is None
):
    cloudflare_tunnel_process.terminate()

    try:
        cloudflare_tunnel_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        cloudflare_tunnel_process.kill()
        cloudflare_tunnel_process.wait(timeout=5)

cloudflare_tunnel_logs = []

cloudflare_tunnel_process = subprocess.Popen(
    [
        CLOUDFLARED_BINARY,
        "tunnel",
        "--url",
        f"http://{MCP_HOST}:{MCP_PORT}",
        "--no-autoupdate",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,
)

cloudflare_log_thread = threading.Thread(
    target=stream_cloudflared_logs,
    args=(
        cloudflare_tunnel_process,
        cloudflare_tunnel_logs,
    ),
    name="vaultify-cloudflare-log-reader",
    daemon=True,
)

cloudflare_log_thread.start()

startup_deadline = (
    time.time()
    + CLOUDFLARE_STARTUP_TIMEOUT_SECONDS
)

CLOUDFLARE_PUBLIC_URL = None

while time.time() < startup_deadline:
    if cloudflare_tunnel_process.poll() is not None:
        raise RuntimeError(
            "cloudflared stopped unexpectedly. "
            f"Recent logs: {cloudflare_tunnel_logs[-10:]}"
        )

    CLOUDFLARE_PUBLIC_URL = extract_trycloudflare_url(
        cloudflare_tunnel_logs
    )

    if (
        CLOUDFLARE_PUBLIC_URL
        and tunnel_is_registered(
            cloudflare_tunnel_logs
        )
    ):
        break

    time.sleep(0.25)

if not CLOUDFLARE_PUBLIC_URL:
    raise RuntimeError(
        "Cloudflare did not provide a public URL."
    )

if not tunnel_is_registered(
    cloudflare_tunnel_logs
):
    raise RuntimeError(
        "The URL was created, but the tunnel was not registered."
    )

PUBLIC_MCP_URL = (
    CLOUDFLARE_PUBLIC_URL.rstrip("/")
    + "/mcp"
)

print("\n✅ Cloudflare Quick Tunnel started.")
print(f"🌍 Public URL: {CLOUDFLARE_PUBLIC_URL}")
print(f"🔗 Public MCP endpoint: {PUBLIC_MCP_URL}")


[cloudflared] 2026-07-23T11:53:07Z INF Initiating graceful shutdown due to signal terminated ...
[cloudflared] 2026-07-23T11:53:07Z ERR failed to run the datagram handler error="context canceled" connIndex=0 event=0 ip=198.41.200.193
[cloudflared] 2026-07-23T11:53:07Z ERR failed to serve tunnel connection error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.193
[cloudflared] 2026-07-23T11:53:07Z ERR Serve tunnel error error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.193
[cloudflared] 2026-07-23T11:53:07Z INF Retrying connection in up to 1s connIndex=0 event=0 ip=198.41.200.193
[cloudflared] 2026-07-23T11:53:07Z ERR Connection terminated connIndex=0
[cloudflared] 2026-07-23T11:53:07Z ERR no more connections active and exiting
[cloudflared] 2026-07-23T11:53:07Z INF Tunnel server stopped
[cloudflared] 2026-07-23T11:53:07Z INF Metrics server stopped
[cloudflared] 2026-07-23T11:53:07Z INF Tha

In [53]:
# VAULTIFY COMPACT - Cell 10: Resolve tunnel DNS and verify the public MCP endpoint

import asyncio
import socket
from urllib.parse import urlparse

import requests

PUBLIC_TEST_ATTEMPTS = 12
PUBLIC_TEST_RETRY_SECONDS = 3


def ensure_public_hostname_resolves(public_url):
    """
    Resolve a new Quick Tunnel hostname, with DNS-over-HTTPS as a fallback.
    """
    hostname = urlparse(public_url).hostname

    if not hostname:
        raise RuntimeError("The public tunnel hostname is invalid.")

    try:
        resolved_ip = socket.gethostbyname(hostname)
        print(f"✅ DNS resolved normally: {hostname} → {resolved_ip}")
        return
    except socket.gaierror:
        print("ℹ️ Normal DNS is not ready. Trying DNS-over-HTTPS.")

    dns_response = requests.get(
        "https://dns.google/resolve",
        params={
            "name": hostname,
            "type": "A",
        },
        timeout=15,
    )

    dns_response.raise_for_status()

    answers = dns_response.json().get(
        "Answer",
        [],
    )

    ipv4_addresses = [
        answer["data"]
        for answer in answers
        if answer.get("type") == 1
    ]

    if not ipv4_addresses:
        raise RuntimeError(
            "The Quick Tunnel hostname has no IPv4 DNS answer yet."
        )

    resolved_ip = ipv4_addresses[0]
    hosts_entry = f"{resolved_ip} {hostname}\n"

    with open(
        "/etc/hosts",
        "r",
        encoding="utf-8",
    ) as hosts_file:
        existing_hosts = hosts_file.read()

    if hostname not in existing_hosts:
        with open(
            "/etc/hosts",
            "a",
            encoding="utf-8",
        ) as hosts_file:
            hosts_file.write(hosts_entry)

    print(
        f"✅ Added temporary DNS fallback: "
        f"{hostname} → {resolved_ip}"
    )


async def call_public_mcp_once():
    """
    Run one complete MCP request through the public Cloudflare endpoint.
    """
    async with streamablehttp_client(
        PUBLIC_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization = await session.initialize()
            tools = await session.list_tools()
            tool_names = [
                tool.name
                for tool in tools.tools
            ]

            if "ask_documents" not in tool_names:
                raise RuntimeError(
                    "The ask_documents tool is missing publicly."
                )

            tool_result = await session.call_tool(
                "ask_documents",
                {"question": MCP_TEST_QUESTION},
            )

            return {
                "server_name": initialization.serverInfo.name,
                "server_version": initialization.serverInfo.version,
                "tool_names": tool_names,
                "tool_result": tool_result,
            }


ensure_public_hostname_resolves(
    CLOUDFLARE_PUBLIC_URL
)

public_test = None
last_public_error = None

for attempt_number in range(
    1,
    PUBLIC_TEST_ATTEMPTS + 1,
):
    try:
        public_test = await call_public_mcp_once()
        break

    except BaseException as error:
        last_public_error = error

        print(
            f"⏳ Public MCP attempt "
            f"{attempt_number}/"
            f"{PUBLIC_TEST_ATTEMPTS} failed: "
            f"{type(error).__name__}: {error}"
        )

        if attempt_number < PUBLIC_TEST_ATTEMPTS:
            await asyncio.sleep(
                PUBLIC_TEST_RETRY_SECONDS
            )

if public_test is None:
    raise RuntimeError(
        "The public MCP endpoint remained unreachable."
    ) from last_public_error

public_payload = validate_mcp_payload(
    parse_mcp_tool_result(
        public_test["tool_result"]
    )
)

print("\n✅ Public MCP test passed.")
print(
    f"📦 Server: "
    f"{public_test['server_name']} "
    f"{public_test['server_version']}"
)
print(f"🛠️ Tools: {public_test['tool_names']}")
print(f"🔐 Tenant: {public_payload['tenant_id']}")
print(f"📚 Sources: {len(public_payload['sources'])}")
print("\n🤖 Public answer:")
print(public_payload["answer"])

print("\n🚀 Claude Custom Connector endpoint:")
print(PUBLIC_MCP_URL)
print(
    "\n⚠️ This Quick Tunnel is temporary and works only "
    "while the Colab runtime remains connected."
)


✅ DNS resolved normally: sarah-banner-refugees-sydney.trycloudflare.com → 104.16.231.132
INFO:     34.16.191.33:0 - "POST /mcp HTTP/1.1" 200 OK
INFO:     34.16.191.33:0 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     34.16.191.33:0 - "POST /mcp HTTP/1.1" 200 OK
INFO:     34.16.191.33:0 - "POST /mcp HTTP/1.1" 200 OK

✅ Public MCP test passed.
📦 Server: Vaultify 1.28.1
🛠️ Tools: ['ask_documents']
🔐 Tenant: demo_apple_tenant
📚 Sources: 5

🤖 Public answer:
According to the table in Section: Note 2 - Revenue of the apple_fy2025_10k.pdf file, Apple's total net sales in fiscal year 2025 were $416,161. (Source: Context 3, Section: Note 2 - Revenue)

🚀 Claude Custom Connector endpoint:
https://sarah-banner-refugees-sydney.trycloudflare.com/mcp

⚠️ This Quick Tunnel is temporary and works only while the Colab runtime remains connected.


In [54]:
# VAULTIFY COMPACT - Cell 11: Create the compact Flask frontend files

from pathlib import Path
import textwrap

WEB_PROJECT_ROOT = Path("/content/vaultify_compact_web")
WEB_TEMPLATES_DIR = WEB_PROJECT_ROOT / "templates"
WEB_STATIC_DIR = WEB_PROJECT_ROOT / "static"
WEB_UPLOADS_DIR = WEB_PROJECT_ROOT / "uploads"
WEB_INSTANCE_DIR = WEB_PROJECT_ROOT / "instance"

for directory in [
    WEB_TEMPLATES_DIR,
    WEB_STATIC_DIR,
    WEB_UPLOADS_DIR,
    WEB_INSTANCE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def write_text_file(path, content):
    """
    Write one UTF-8 project file after removing shared indentation.
    """
    path.write_text(
        textwrap.dedent(content).strip() + "\n",
        encoding="utf-8",
    )


write_text_file(
    WEB_STATIC_DIR / "vaultify.css",
    r"""
    :root {
        color-scheme: dark;
        --bg: #080b12;
        --panel: #101522;
        --border: #283044;
        --text: #f5f7ff;
        --muted: #9da8c2;
        --accent: #8066ff;
        --success: #39e6bd;
        --danger: #ff6f91;
        --warning: #ffc96b;
        --max: 1120px;
    }

    * {
        box-sizing: border-box;
    }

    html {
        background: var(--bg);
    }

    body {
        margin: 0;
        min-height: 100vh;
        background:
            radial-gradient(
                circle at 15% 0%,
                rgba(62, 70, 155, 0.18),
                transparent 30rem
            ),
            var(--bg);
        color: var(--text);
        font-family:
            Inter,
            ui-sans-serif,
            system-ui,
            -apple-system,
            BlinkMacSystemFont,
            "Segoe UI",
            sans-serif;
        font-size: 15px;
        line-height: 1.5;
    }

    a {
        color: #a9b5ff;
        text-decoration: none;
    }

    a:hover {
        color: #d2d8ff;
    }

    button,
    input,
    select {
        font: inherit;
    }

    .shell {
        width: min(var(--max), calc(100% - 32px));
        margin: 0 auto;
    }

    .topbar {
        position: sticky;
        top: 0;
        z-index: 20;
        border-bottom: 1px solid rgba(40, 48, 68, 0.85);
        background: rgba(8, 11, 18, 0.92);
        backdrop-filter: blur(16px);
    }

    .nav {
        min-height: 64px;
        display: flex;
        align-items: center;
        gap: 18px;
    }

    .brand {
        color: white;
        font-size: 20px;
        font-weight: 800;
        letter-spacing: -0.04em;
        margin-right: auto;
    }

    .nav-links {
        display: flex;
        align-items: center;
        gap: 6px;
        flex-wrap: wrap;
        justify-content: flex-end;
    }

    .nav-link,
    .nav-button {
        border: 0;
        background: transparent;
        color: #b5bdd2;
        border-radius: 10px;
        padding: 8px 11px;
        cursor: pointer;
    }

    .nav-link:hover,
    .nav-link.active,
    .nav-button:hover {
        color: white;
        background: rgba(128, 102, 255, 0.13);
    }

    .org-switcher {
        max-width: 190px;
        min-width: 150px;
    }

    .page {
        padding: 38px 0 60px;
    }

    .eyebrow {
        margin: 0 0 10px;
        color: #8298ff;
        font-size: 12px;
        font-weight: 800;
        letter-spacing: 0.14em;
        text-transform: uppercase;
    }

    h1,
    h2,
    h3,
    p {
        margin-top: 0;
    }

    h1 {
        margin-bottom: 8px;
        font-size: clamp(32px, 5vw, 50px);
        line-height: 1.03;
        letter-spacing: -0.045em;
    }

    h2 {
        font-size: 22px;
        letter-spacing: -0.02em;
    }

    .lead,
    .muted {
        color: var(--muted);
    }

    .lead {
        margin-bottom: 26px;
        max-width: 740px;
        font-size: 16px;
    }

    .grid {
        display: grid;
        gap: 16px;
    }

    .stats {
        grid-template-columns: repeat(4, minmax(0, 1fr));
    }

    .two-column {
        grid-template-columns:
            minmax(0, 1.35fr)
            minmax(280px, 0.65fr);
    }

    .panel,
    .stat,
    .auth-card {
        border: 1px solid var(--border);
        background:
            linear-gradient(
                180deg,
                rgba(17, 22, 36, 0.98),
                rgba(13, 18, 30, 0.98)
            );
        box-shadow: 0 18px 55px rgba(0, 0, 0, 0.2);
    }

    .panel {
        border-radius: 18px;
        padding: 24px;
    }

    .stat {
        min-height: 112px;
        border-radius: 16px;
        padding: 19px;
    }

    .stat-label {
        color: #8ea1c7;
        font-size: 12px;
        letter-spacing: 0.08em;
        text-transform: uppercase;
    }

    .stat-value {
        margin-top: 10px;
        font-size: 23px;
        font-weight: 800;
        overflow-wrap: anywhere;
    }

    .flash-stack {
        display: grid;
        gap: 10px;
        margin-bottom: 18px;
    }

    .flash {
        border: 1px solid var(--border);
        border-radius: 12px;
        padding: 12px 14px;
        background: #11192a;
    }

    .flash.success {
        border-color: rgba(57, 230, 189, 0.38);
        color: var(--success);
    }

    .flash.error {
        border-color: rgba(255, 111, 145, 0.38);
        color: var(--danger);
    }

    .flash.warning {
        border-color: rgba(255, 201, 107, 0.38);
        color: var(--warning);
    }

    label {
        display: block;
        margin-bottom: 7px;
        font-size: 13px;
        font-weight: 700;
    }

    .field {
        margin-bottom: 15px;
    }

    input,
    select {
        width: 100%;
        min-height: 44px;
        border: 1px solid #35405a;
        border-radius: 11px;
        background: #0b1020;
        color: white;
        padding: 10px 12px;
        outline: none;
    }

    input:focus,
    select:focus {
        border-color: var(--accent);
        box-shadow:
            0 0 0 3px rgba(128, 102, 255, 0.16);
    }

    input:-webkit-autofill {
        -webkit-text-fill-color: white;
        box-shadow:
            0 0 0 1000px #172036 inset;
        transition:
            background-color 9999s ease-out;
    }

    .button {
        display: inline-flex;
        min-height: 42px;
        align-items: center;
        justify-content: center;
        border: 0;
        border-radius: 11px;
        padding: 10px 16px;
        background:
            linear-gradient(
                90deg,
                #7565ff,
                #9866ff
            );
        color: white;
        font-weight: 800;
        cursor: pointer;
    }

    .button:hover {
        filter: brightness(1.08);
        color: white;
    }

    .button.secondary {
        border: 1px solid var(--border);
        background: #171e2e;
    }

    .button.danger {
        background: #542139;
    }

    .button.small {
        min-height: 34px;
        padding: 7px 11px;
        font-size: 13px;
    }

    .button-row {
        display: flex;
        flex-wrap: wrap;
        gap: 10px;
        align-items: center;
    }

    .auth-wrap {
        min-height: calc(100vh - 65px);
        display: grid;
        place-items: center;
        padding: 24px 16px;
    }

    .auth-card {
        width: min(450px, 100%);
        border-radius: 18px;
        padding: 26px;
    }

    .auth-card h1 {
        font-size: clamp(30px, 7vw, 41px);
    }

    .auth-card .button {
        width: 100%;
    }

    .table-wrap {
        overflow-x: auto;
    }

    table {
        width: 100%;
        border-collapse: collapse;
    }

    th,
    td {
        border-bottom: 1px solid var(--border);
        padding: 13px 10px;
        text-align: left;
        vertical-align: top;
    }

    th {
        color: #9caccc;
        font-size: 12px;
        letter-spacing: 0.06em;
        text-transform: uppercase;
    }

    .status {
        display: inline-flex;
        border-radius: 999px;
        padding: 4px 9px;
        background: rgba(128, 102, 255, 0.14);
        color: #bcb5ff;
        font-size: 12px;
        font-weight: 800;
    }

    .status.ready {
        background: rgba(57, 230, 189, 0.12);
        color: var(--success);
    }

    .status.failed {
        background: rgba(255, 111, 145, 0.12);
        color: var(--danger);
    }

    .answer {
        margin-top: 18px;
        border-left: 3px solid var(--accent);
        border-radius: 0 12px 12px 0;
        background: #0a1020;
        padding: 18px;
        white-space: pre-wrap;
    }

    .source-list {
        margin: 12px 0 0;
        padding-left: 20px;
        color: var(--muted);
    }

    .empty {
        border: 1px dashed #35405a;
        border-radius: 16px;
        padding: 34px 20px;
        text-align: center;
        color: var(--muted);
    }

    code {
        color: #d8dbff;
        overflow-wrap: anywhere;
    }

    @media (max-width: 900px) {
        .stats,
        .two-column {
            grid-template-columns: 1fr 1fr;
        }

        .nav {
            align-items: flex-start;
            padding: 14px 0;
            flex-wrap: wrap;
        }

        .brand {
            width: 100%;
        }

        .nav-links {
            width: 100%;
            justify-content: flex-start;
        }
    }

    @media (max-width: 620px), (max-height: 720px) {
        .page {
            padding-top: 24px;
        }

        .stats,
        .two-column {
            grid-template-columns: 1fr;
        }

        .panel,
        .auth-card {
            padding: 19px;
        }

        .auth-wrap {
            place-items: start center;
            padding-top: 20px;
        }

        h1 {
            font-size: 34px;
        }
    }
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "base.html",
    r"""
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <meta
            name="viewport"
            content="width=device-width, initial-scale=1"
        >
        <title>{{ title or "Vaultify" }}</title>
        <link
            rel="stylesheet"
            href="{{ url_for('static', filename='vaultify.css') }}"
        >
    </head>
    <body>
        <header class="topbar">
            <div class="shell nav">
                <a
                    class="brand"
                    href="{{
                        url_for('dashboard')
                        if current_user.is_authenticated
                        else url_for('login')
                    }}"
                >
                    Vaultify
                </a>

                {% if current_user.is_authenticated %}
                <form
                    method="post"
                    action="{{ url_for('switch_organization') }}"
                >
                    <input
                        type="hidden"
                        name="csrf_token"
                        value="{{ csrf_token() }}"
                    >

                    <select
                        class="org-switcher"
                        name="organization_id"
                        aria-label="Active organization"
                        onchange="this.form.submit()"
                    >
                        {% for membership in memberships %}
                        <option
                            value="{{ membership.organization.id }}"
                            {% if
                                current_org
                                and membership.organization.id
                                == current_org.id
                            %}selected{% endif %}
                        >
                            {{ membership.organization.name }}
                        </option>
                        {% endfor %}
                    </select>
                </form>

                <nav
                    class="nav-links"
                    aria-label="Primary navigation"
                >
                    <a
                        class="nav-link
                        {% if request.endpoint == 'dashboard' %}
                        active
                        {% endif %}"
                        href="{{ url_for('dashboard') }}"
                    >
                        Dashboard
                    </a>

                    <a
                        class="nav-link
                        {% if request.endpoint in
                            ['documents', 'upload_document']
                        %}
                        active
                        {% endif %}"
                        href="{{ url_for('documents') }}"
                    >
                        Documents
                    </a>

                    <a
                        class="nav-link
                        {% if request.endpoint == 'organization' %}
                        active
                        {% endif %}"
                        href="{{ url_for('organization') }}"
                    >
                        Organization
                    </a>

                    <span class="nav-link">
                        {{ current_user.display_name }}
                    </span>

                    <form
                        method="post"
                        action="{{ url_for('logout') }}"
                    >
                        <input
                            type="hidden"
                            name="csrf_token"
                            value="{{ csrf_token() }}"
                        >

                        <button
                            class="nav-button"
                            type="submit"
                        >
                            Sign out
                        </button>
                    </form>
                </nav>
                {% endif %}
            </div>
        </header>

        {% if current_user.is_authenticated %}
        <main class="shell page">
        {% else %}
        <main>
        {% endif %}

            {% with
                messages =
                get_flashed_messages(
                    with_categories=true
                )
            %}
                {% if messages %}
                <div
                    class="
                    {% if current_user.is_authenticated %}
                    flash-stack
                    {% else %}
                    shell flash-stack
                    {% endif %}
                    "
                >
                    {% for category, message in messages %}
                    <div class="flash {{ category }}">
                        {{ message }}
                    </div>
                    {% endfor %}
                </div>
                {% endif %}
            {% endwith %}

            {% block content %}{% endblock %}
        </main>
    </body>
    </html>
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "login.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <section class="auth-wrap">
        <div class="auth-card">
            <p class="eyebrow">Welcome back</p>
            <h1>Sign in to Vaultify</h1>

            <p class="lead">
                Access your organization's
                private document workspace.
            </p>

            <form method="post">
                <input
                    type="hidden"
                    name="csrf_token"
                    value="{{ csrf_token() }}"
                >

                <div class="field">
                    <label for="email">
                        Email address
                    </label>

                    <input
                        id="email"
                        name="email"
                        type="email"
                        autocomplete="email"
                        required
                    >
                </div>

                <div class="field">
                    <label for="password">
                        Password
                    </label>

                    <input
                        id="password"
                        name="password"
                        type="password"
                        autocomplete="current-password"
                        required
                    >
                </div>

                <button
                    class="button"
                    type="submit"
                >
                    Sign in
                </button>
            </form>

            <p
                class="muted"
                style="margin-top:16px"
            >
                New to Vaultify?
                <a href="{{ url_for('register') }}">
                    Create an account
                </a>
            </p>
        </div>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "register.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <section class="auth-wrap">
        <div class="auth-card">
            <p class="eyebrow">
                Start using Vaultify
            </p>

            <h1>Create your account</h1>

            <p class="lead">
                Your first secure organization
                will be created automatically.
            </p>

            <form method="post">
                <input
                    type="hidden"
                    name="csrf_token"
                    value="{{ csrf_token() }}"
                >

                <div class="field">
                    <label for="display_name">
                        Full name
                    </label>

                    <input
                        id="display_name"
                        name="display_name"
                        maxlength="100"
                        required
                    >
                </div>

                <div class="field">
                    <label for="organization_name">
                        Organization name
                    </label>

                    <input
                        id="organization_name"
                        name="organization_name"
                        maxlength="120"
                        required
                    >
                </div>

                <div class="field">
                    <label for="email">
                        Email address
                    </label>

                    <input
                        id="email"
                        name="email"
                        type="email"
                        autocomplete="email"
                        required
                    >
                </div>

                <div class="field">
                    <label for="password">
                        Password
                    </label>

                    <input
                        id="password"
                        name="password"
                        type="password"
                        minlength="10"
                        autocomplete="new-password"
                        required
                    >
                </div>

                <div class="field">
                    <label for="confirm_password">
                        Confirm password
                    </label>

                    <input
                        id="confirm_password"
                        name="confirm_password"
                        type="password"
                        minlength="10"
                        autocomplete="new-password"
                        required
                    >
                </div>

                <button
                    class="button"
                    type="submit"
                >
                    Create account
                </button>
            </form>

            <p
                class="muted"
                style="margin-top:16px"
            >
                Already have an account?
                <a href="{{ url_for('login') }}">
                    Sign in
                </a>
            </p>
        </div>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "dashboard.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <p class="eyebrow">
        Private AI workspace
    </p>

    <h1>
        Welcome, {{ current_user.display_name }}
    </h1>

    <p class="lead">
        Your tenant context is resolved
        from your authenticated organization membership.
    </p>

    <section class="grid stats">
        <article class="stat">
            <div class="stat-label">
                Organization
            </div>

            <div class="stat-value">
                {{ current_org.name }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Documents
            </div>

            <div class="stat-value">
                {{ document_count }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Ready chunks
            </div>

            <div class="stat-value">
                {{ chunk_count }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Role
            </div>

            <div class="stat-value">
                {{ current_membership.role|title }}
            </div>
        </article>
    </section>

    <section
        class="grid two-column"
        style="margin-top:18px"
    >
        <article class="panel">
            <h2>Ask your documents</h2>

            <p class="muted">
                Answers are restricted to the
                active organization's Qdrant tenant.
            </p>

            <form
                method="post"
                action="{{ url_for('ask') }}"
            >
                <input
                    type="hidden"
                    name="csrf_token"
                    value="{{ csrf_token() }}"
                >

                <div class="field">
                    <label for="question">
                        Question
                    </label>

                    <input
                        id="question"
                        name="question"
                        value="{{ question or '' }}"
                        placeholder="What was total revenue in Q4 2025?"
                        required
                    >
                </div>

                <button
                    class="button"
                    type="submit"
                >
                    Ask Vaultify
                </button>
            </form>

            {% if answer %}
            <div class="answer">
                {{ answer }}
            </div>

            {% if sources %}
            <ul class="source-list">
                {% for source in sources %}
                <li>
                    {{ source.filename }}
                    · {{ source.section }}
                    · score
                    {{ "%.4f"|format(source.score) }}
                </li>
                {% endfor %}
            </ul>
            {% endif %}
            {% endif %}
        </article>

        <aside class="panel">
            <h2>Trusted tenant</h2>

            <p class="muted">
                This value comes from the verified membership,
                not from a user-controlled form field.
            </p>

            <code>
                {{ current_org.tenant_id }}
            </code>

            <div
                class="button-row"
                style="margin-top:18px"
            >
                <a
                    class="button secondary"
                    href="{{ url_for('documents') }}"
                >
                    Open documents
                </a>
            </div>
        </aside>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "documents.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <div
        class="button-row"
        style="justify-content:space-between"
    >
        <div>
            <p class="eyebrow">
                Knowledge base
            </p>

            <h1>Documents</h1>

            <p class="lead">
                Files belonging to
                {{ current_org.name }}.
            </p>
        </div>

        {% if documents %}
        <a
            class="button"
            href="{{ url_for('upload_document') }}"
        >
            Upload PDF
        </a>
        {% endif %}
    </div>

    {% if documents %}
    <section class="panel table-wrap">
        <table>
            <thead>
                <tr>
                    <th>Document</th>
                    <th>Status</th>
                    <th>Chunks</th>
                    <th>Uploaded</th>
                    <th>Actions</th>
                </tr>
            </thead>

            <tbody>
                {% for document in documents %}
                <tr>
                    <td>
                        <strong>
                            {{ document.original_filename }}
                        </strong>
                        <br>

                        <span class="muted">
                            {{ document.size_bytes }} bytes
                        </span>
                    </td>

                    <td>
                        <span
                            class="status {{ document.status }}"
                        >
                            {{ document.status }}
                        </span>
                    </td>

                    <td>
                        {{ document.chunk_count }}
                    </td>

                    <td>
                        {{
                            document.created_at.strftime(
                                "%Y-%m-%d %H:%M"
                            )
                        }}
                    </td>

                    <td>
                        <div class="button-row">
                            {% if document.status == "failed" %}
                            <form
                                method="post"
                                action="{{
                                    url_for(
                                        'retry_document',
                                        document_id=document.id
                                    )
                                }}"
                            >
                                <input
                                    type="hidden"
                                    name="csrf_token"
                                    value="{{ csrf_token() }}"
                                >

                                <button
                                    class="button secondary small"
                                    type="submit"
                                >
                                    Retry
                                </button>
                            </form>
                            {% endif %}

                            <form
                                method="post"
                                action="{{
                                    url_for(
                                        'delete_document',
                                        document_id=document.id
                                    )
                                }}"
                            >
                                <input
                                    type="hidden"
                                    name="csrf_token"
                                    value="{{ csrf_token() }}"
                                >

                                <button
                                    class="button danger small"
                                    type="submit"
                                >
                                    Delete
                                </button>
                            </form>
                        </div>
                    </td>
                </tr>
                {% endfor %}
            </tbody>
        </table>
    </section>

    {% else %}
    <section class="empty">
        <h2>No documents yet</h2>

        <p>
            Upload the first PDF
            for this organization.
        </p>

        <a
            class="button"
            href="{{ url_for('upload_document') }}"
        >
            Upload first document
        </a>
    </section>
    {% endif %}
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "upload.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <p class="eyebrow">
        Secure ingestion
    </p>

    <h1>Upload a PDF</h1>

    <p class="lead">
        The file is validated, parsed,
        embedded, and indexed only under
        {{ current_org.name }}.
    </p>

    <section
        class="panel"
        style="max-width:720px"
    >
        <form
            method="post"
            enctype="multipart/form-data"
        >
            <input
                type="hidden"
                name="csrf_token"
                value="{{ csrf_token() }}"
            >

            <div class="field">
                <label for="file">
                    PDF document
                </label>

                <input
                    id="file"
                    name="file"
                    type="file"
                    accept=".pdf,application/pdf"
                    required
                >
            </div>

            <button
                class="button"
                type="submit"
            >
                Upload and index
            </button>
        </form>

        <p
            class="muted"
            style="margin-top:16px"
        >
            The compact version currently
            processes uploads synchronously.
            Large reports can take one or two minutes.
        </p>
    </section>
    {% endblock %}
    """,
)

write_text_file(
    WEB_TEMPLATES_DIR / "organization.html",
    r"""
    {% extends "base.html" %}

    {% block content %}
    <p class="eyebrow">
        Organization management
    </p>

    <h1>
        {{ current_org.name }}
    </h1>

    <p class="lead">
        This page shows the trusted organization
        context used by documents, retrieval, and MCP.
    </p>

    <section class="grid stats">
        <article class="stat">
            <div class="stat-label">
                Your role
            </div>

            <div class="stat-value">
                {{ current_membership.role|title }}
            </div>
        </article>

        <article class="stat">
            <div class="stat-label">
                Organization slug
            </div>

            <div class="stat-value">
                {{ current_org.slug }}
            </div>
        </article>

        <article
            class="stat"
            style="grid-column:span 2"
        >
            <div class="stat-label">
                Trusted tenant
            </div>

            <div class="stat-value">
                <code>
                    {{ current_org.tenant_id }}
                </code>
            </div>
        </article>
    </section>
    {% endblock %}
    """,
)

print(
    "✅ Compact Flask templates "
    "and responsive CSS created."
)
print(
    f"📁 Web project root: "
    f"{WEB_PROJECT_ROOT}"
)

✅ Compact Flask templates and responsive CSS created.
📁 Web project root: /content/vaultify_compact_web


In [55]:
# VAULTIFY COMPACT - Cell 12: Create the compact Flask backend

import textwrap

WEB_BACKEND_PATH = (
    WEB_PROJECT_ROOT
    / "vaultify_web_backend.py"
)

WEB_BACKEND_CODE = r"""
import hashlib
import re
import secrets
import unicodedata
import uuid
from datetime import datetime, timezone
from pathlib import Path

import filetype
import torch
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from flask import (
    Flask,
    abort,
    flash,
    redirect,
    render_template,
    request,
    session,
    url_for,
)
from flask_login import (
    LoginManager,
    UserMixin,
    current_user,
    login_required,
    login_user,
    logout_user,
)
from flask_sqlalchemy import SQLAlchemy
from flask_wtf.csrf import CSRFProtect
from qdrant_client.models import (
    FieldCondition,
    Filter,
    FilterSelector,
    MatchValue,
    PointStruct,
)
from sqlalchemy import UniqueConstraint
from werkzeug.security import (
    check_password_hash,
    generate_password_hash,
)
from werkzeug.utils import secure_filename


db = SQLAlchemy()
login_manager = LoginManager()
csrf = CSRFProtect()


def utc_now():
    return datetime.now(timezone.utc)


class User(UserMixin, db.Model):
    __tablename__ = "users"

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    email = db.Column(
        db.String(255),
        unique=True,
        index=True,
        nullable=False,
    )

    password_hash = db.Column(
        db.String(255),
        nullable=False,
    )

    display_name = db.Column(
        db.String(100),
        nullable=False,
    )

    is_active_account = db.Column(
        db.Boolean,
        nullable=False,
        default=True,
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )

    memberships = db.relationship(
        "Membership",
        back_populates="user",
        cascade="all, delete-orphan",
    )

    @property
    def is_active(self):
        return self.is_active_account

    def set_password(self, password):
        self.password_hash = (
            generate_password_hash(
                password
            )
        )

    def check_password(self, password):
        return check_password_hash(
            self.password_hash,
            password,
        )


class Organization(db.Model):
    __tablename__ = "organizations"

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    tenant_id = db.Column(
        db.String(80),
        unique=True,
        index=True,
        nullable=False,
        default=lambda: (
            f"tenant_{secrets.token_hex(12)}"
        ),
    )

    name = db.Column(
        db.String(120),
        nullable=False,
    )

    slug = db.Column(
        db.String(140),
        unique=True,
        index=True,
        nullable=False,
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )

    memberships = db.relationship(
        "Membership",
        back_populates="organization",
        cascade="all, delete-orphan",
    )

    documents = db.relationship(
        "Document",
        back_populates="organization",
        cascade="all, delete-orphan",
    )


class Membership(db.Model):
    __tablename__ = "memberships"

    __table_args__ = (
        UniqueConstraint(
            "user_id",
            "organization_id",
            name=(
                "uq_membership_"
                "user_organization"
            ),
        ),
    )

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    user_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "users.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    organization_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "organizations.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    role = db.Column(
        db.String(20),
        nullable=False,
        default="member",
    )

    user = db.relationship(
        "User",
        back_populates="memberships",
    )

    organization = db.relationship(
        "Organization",
        back_populates="memberships",
    )


class Document(db.Model):
    __tablename__ = "documents"

    __table_args__ = (
        UniqueConstraint(
            "organization_id",
            "document_hash",
            name=(
                "uq_document_"
                "organization_hash"
            ),
        ),
    )

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    organization_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "organizations.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    original_filename = db.Column(
        db.String(255),
        nullable=False,
    )

    stored_filename = db.Column(
        db.String(255),
        nullable=False,
        unique=True,
    )

    storage_path = db.Column(
        db.String(500),
        nullable=False,
    )

    mime_type = db.Column(
        db.String(120),
        nullable=False,
    )

    size_bytes = db.Column(
        db.Integer,
        nullable=False,
    )

    document_hash = db.Column(
        db.String(64),
        nullable=False,
        index=True,
    )

    status = db.Column(
        db.String(20),
        nullable=False,
        default="uploaded",
    )

    chunk_count = db.Column(
        db.Integer,
        nullable=False,
        default=0,
    )

    error_message = db.Column(
        db.String(500),
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )

    indexed_at = db.Column(
        db.DateTime(timezone=True),
    )

    organization = db.relationship(
        "Organization",
        back_populates="documents",
    )


class QueryLog(db.Model):
    __tablename__ = "query_logs"

    id = db.Column(
        db.Integer,
        primary_key=True,
    )

    organization_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "organizations.id",
            ondelete="CASCADE",
        ),
        nullable=False,
        index=True,
    )

    user_id = db.Column(
        db.Integer,
        db.ForeignKey(
            "users.id",
            ondelete="SET NULL",
        ),
        nullable=True,
        index=True,
    )

    question = db.Column(
        db.Text,
        nullable=False,
    )

    answer = db.Column(
        db.Text,
        nullable=False,
    )

    source_count = db.Column(
        db.Integer,
        nullable=False,
        default=0,
    )

    created_at = db.Column(
        db.DateTime(timezone=True),
        nullable=False,
        default=utc_now,
    )


@login_manager.user_loader
def load_user(user_id):
    try:
        return db.session.get(
            User,
            int(user_id),
        )
    except (TypeError, ValueError):
        return None


def normalize_email(value):
    return (
        value or ""
    ).strip().lower()


def create_slug(value):
    normalized = unicodedata.normalize(
        "NFKD",
        value or "",
    )

    ascii_value = normalized.encode(
        "ascii",
        "ignore",
    ).decode("ascii")

    slug = re.sub(
        r"[^a-zA-Z0-9]+",
        "-",
        ascii_value,
    ).strip("-").lower()

    return slug or "organization"


def unique_organization_slug(name):
    base_slug = create_slug(name)
    candidate = base_slug
    counter = 1

    while Organization.query.filter_by(
        slug=candidate
    ).first():
        counter += 1
        candidate = (
            f"{base_slug}-{counter}"
        )

    return candidate


def password_is_strong(password):
    return (
        len(password or "") >= 10
        and any(
            character.isalpha()
            for character in password
        )
        and any(
            character.isdigit()
            for character in password
        )
    )


def get_memberships_for_user(user_id):
    return (
        Membership.query
        .filter_by(user_id=user_id)
        .order_by(Membership.id.asc())
        .all()
    )


def resolve_active_membership():
    if not current_user.is_authenticated:
        return None

    memberships = get_memberships_for_user(
        current_user.id
    )

    if not memberships:
        logout_user()
        session.pop(
            "organization_id",
            None,
        )
        return None

    selected_id = session.get(
        "organization_id"
    )

    for membership in memberships:
        if (
            membership.organization_id
            == selected_id
        ):
            return membership

    membership = memberships[0]

    session["organization_id"] = (
        membership.organization_id
    )

    return membership


def require_management_role(membership):
    if membership.role not in {
        "owner",
        "admin",
    }:
        abort(403)


def build_qdrant_document_filter(
    tenant_id,
    document_hash,
):
    return Filter(
        must=[
            FieldCondition(
                key="tenant_id",
                match=MatchValue(
                    value=tenant_id
                ),
            ),
            FieldCondition(
                key="document_hash",
                match=MatchValue(
                    value=document_hash
                ),
            ),
        ]
    )


def delete_document_vectors(
    services,
    tenant_id,
    document_hash,
):
    services["qdrant"].delete(
        collection_name=(
            services["collection_name"]
        ),
        points_selector=FilterSelector(
            filter=(
                build_qdrant_document_filter(
                    tenant_id,
                    document_hash,
                )
            )
        ),
        wait=True,
    )


def token_count(model, text):
    return len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
        )
    )


def split_text_by_tokens(
    model,
    text,
    max_tokens=220,
    overlap_tokens=30,
):
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    if not token_ids:
        return []

    pieces = []
    start = 0

    while start < len(token_ids):
        end = min(
            start + max_tokens,
            len(token_ids),
        )

        piece = model.tokenizer.decode(
            token_ids[start:end],
            skip_special_tokens=True,
        ).strip()

        if piece:
            pieces.append(piece)

        if end >= len(token_ids):
            break

        start = max(
            0,
            end - overlap_tokens,
        )

    return pieces


def split_large_table(
    model,
    header,
    table_lines,
    max_tokens=220,
):
    if len(table_lines) >= 2:
        prefix_lines = table_lines[:2]
        body_lines = table_lines[2:]
    else:
        prefix_lines = []
        body_lines = table_lines

    table_chunks = []
    current_lines = list(
        prefix_lines
    )

    for row in body_lines:
        candidate_lines = (
            current_lines + [row]
        )

        candidate = "\n".join(
            [
                header,
                "",
                *candidate_lines,
            ]
        ).strip()

        if (
            current_lines
            and token_count(
                model,
                candidate,
            )
            > max_tokens
        ):
            current_text = "\n".join(
                [
                    header,
                    "",
                    *current_lines,
                ]
            ).strip()

            if current_text:
                table_chunks.append(
                    current_text
                )

            current_lines = (
                list(prefix_lines)
                + [row]
            )
        else:
            current_lines.append(row)

    final_text = "\n".join(
        [
            header,
            "",
            *current_lines,
        ]
    ).strip()

    if final_text:
        table_chunks.append(
            final_text
        )

    return table_chunks


def smart_chunk_markdown(
    model,
    markdown_text,
):
    lines = markdown_text.splitlines()
    chunks = []
    text_buffer = []
    current_header = ""
    index = 0

    def flush_text():
        nonlocal text_buffer

        text = "\n".join(
            text_buffer
        ).strip()

        text_buffer = []

        if not text:
            return

        for piece in split_text_by_tokens(
            model,
            text,
        ):
            chunks.append(
                {
                    "text": piece,
                    "chunk_type": "text",
                    "section": (
                        current_header
                        or "Document text"
                    ),
                }
            )

    while index < len(lines):
        line = lines[index]

        if line.strip().startswith("#"):
            current_header = (
                line.strip()
            )

            text_buffer.append(line)
            index += 1
            continue

        if line.strip().startswith("|"):
            flush_text()

            table_lines = []

            while (
                index < len(lines)
                and lines[index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[index]
                )
                index += 1

            full_table = "\n".join(
                [
                    current_header,
                    "",
                    *table_lines,
                ]
            ).strip()

            if (
                token_count(
                    model,
                    full_table,
                )
                <= 220
            ):
                table_pieces = [
                    full_table
                ]
            else:
                table_pieces = (
                    split_large_table(
                        model,
                        current_header,
                        table_lines,
                    )
                )

            for table_piece in table_pieces:
                chunks.append(
                    {
                        "text": table_piece,
                        "chunk_type": "table",
                        "section": (
                            current_header
                            or "Document table"
                        ),
                    }
                )

            continue

        text_buffer.append(line)
        index += 1

    flush_text()

    return chunks


def ingest_document(
    document,
    services,
):
    document.status = "parsing"
    document.error_message = None
    db.session.commit()

    try:
        pipeline_options = (
            PdfPipelineOptions()
        )

        if torch.cuda.is_available():
            pipeline_options.accelerator_options = (
                AcceleratorOptions(
                    num_threads=4,
                    device=(
                        AcceleratorDevice.CUDA
                    ),
                )
            )

        converter = DocumentConverter(
            format_options={
                InputFormat.PDF:
                PdfFormatOption(
                    pipeline_options=(
                        pipeline_options
                    )
                )
            }
        )

        result = converter.convert(
            document.storage_path
        )

        markdown_text = (
            result.document
            .export_to_markdown()
        )

        chunks = smart_chunk_markdown(
            services["embedding_model"],
            markdown_text,
        )

        if not chunks:
            raise RuntimeError(
                "The PDF produced "
                "no indexable chunks."
            )

        document.status = "indexing"
        db.session.commit()

        texts = [
            chunk["text"]
            for chunk in chunks
        ]

        vectors = (
            services["embedding_model"]
            .encode(
                texts,
                batch_size=64,
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=True,
            )
        )

        delete_document_vectors(
            services,
            document.organization.tenant_id,
            document.document_hash,
        )

        points = []

        for (
            chunk_index,
            (chunk, vector),
        ) in enumerate(
            zip(
                chunks,
                vectors,
            )
        ):
            point_id = str(
                uuid.uuid5(
                    uuid.NAMESPACE_URL,
                    (
                        f"{document.organization.tenant_id}:"
                        f"{document.document_hash}:"
                        f"{chunk_index}"
                    ),
                )
            )

            points.append(
                PointStruct(
                    id=point_id,
                    vector=vector.tolist(),
                    payload={
                        "tenant_id": (
                            document.organization
                            .tenant_id
                        ),
                        "document_hash": (
                            document.document_hash
                        ),
                        "filename": (
                            document.original_filename
                        ),
                        "chunk_index": (
                            chunk_index
                        ),
                        "chunk_type": (
                            chunk["chunk_type"]
                        ),
                        "section": (
                            chunk["section"]
                        ),
                        "text": (
                            chunk["text"]
                        ),
                    },
                )
            )

        for batch_start in range(
            0,
            len(points),
            100,
        ):
            services["qdrant"].upsert(
                collection_name=(
                    services["collection_name"]
                ),
                points=points[
                    batch_start:
                    batch_start + 100
                ],
                wait=True,
            )

        document.status = "ready"
        document.chunk_count = len(
            points
        )
        document.indexed_at = utc_now()
        document.error_message = None

        db.session.commit()

        return len(points)

    except Exception as error:
        try:
            delete_document_vectors(
                services,
                (
                    document.organization
                    .tenant_id
                ),
                document.document_hash,
            )
        except Exception:
            pass

        document.status = "failed"
        document.chunk_count = 0

        document.error_message = (
            f"{type(error).__name__}: "
            f"{error}"
        )[:500]

        db.session.commit()
        raise


def serialize_sources(
    results,
    limit=5,
):
    sources = []

    for result in results[:limit]:
        payload = (
            result.payload or {}
        )

        sources.append(
            {
                "filename": payload.get(
                    "filename",
                    "Unknown file",
                ),
                "section": payload.get(
                    "section",
                    "Unknown section",
                ),
                "score": float(
                    result.score
                ),
            }
        )

    return sources


def create_app(
    services,
    root_path,
    secret_key,
):
    root_path = Path(root_path)

    app = Flask(
        __name__,
        template_folder=str(
            root_path / "templates"
        ),
        static_folder=str(
            root_path / "static"
        ),
    )

    app.config.update(
        SECRET_KEY=secret_key,
        SQLALCHEMY_DATABASE_URI=(
            "sqlite:///"
            + str(
                root_path
                / "instance"
                / "vaultify.db"
            )
        ),
        SQLALCHEMY_TRACK_MODIFICATIONS=False,
        MAX_CONTENT_LENGTH=(
            25 * 1024 * 1024
        ),
        UPLOAD_FOLDER=str(
            root_path / "uploads"
        ),
        SESSION_COOKIE_HTTPONLY=True,
        SESSION_COOKIE_SAMESITE="Lax",
        SESSION_COOKIE_SECURE=False,
        VAULTIFY_SERVICES=services,
    )

    db.init_app(app)
    login_manager.init_app(app)
    csrf.init_app(app)

    login_manager.login_view = "login"

    login_manager.login_message = (
        "Please sign in to continue."
    )

    login_manager.login_message_category = (
        "warning"
    )

    @app.context_processor
    def inject_navigation_context():
        if not current_user.is_authenticated:
            return {
                "memberships": [],
                "current_membership": None,
                "current_org": None,
            }

        memberships = (
            get_memberships_for_user(
                current_user.id
            )
        )

        current_membership = (
            resolve_active_membership()
        )

        return {
            "memberships": memberships,
            "current_membership": (
                current_membership
            ),
            "current_org": (
                current_membership.organization
                if current_membership
                else None
            ),
        }

    @app.errorhandler(413)
    def upload_too_large(_error):
        flash(
            (
                "The selected file exceeds "
                "the 25 MB limit."
            ),
            "error",
        )

        return redirect(
            url_for(
                "upload_document"
            )
        )

    @app.get("/health")
    def health():
        return {
            "status": "ok",
            "service": "Vaultify",
            "phase": 3,
        }

    @app.route(
        "/register",
        methods=["GET", "POST"],
    )
    def register():
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard")
            )

        if request.method == "POST":
            display_name = (
                request.form.get(
                    "display_name"
                )
                or ""
            ).strip()

            organization_name = (
                request.form.get(
                    "organization_name"
                )
                or ""
            ).strip()

            email = normalize_email(
                request.form.get("email")
            )

            password = (
                request.form.get(
                    "password"
                )
                or ""
            )

            confirm_password = (
                request.form.get(
                    "confirm_password"
                )
                or ""
            )

            if (
                not display_name
                or not organization_name
                or not email
            ):
                flash(
                    "All fields are required.",
                    "error",
                )

                return render_template(
                    "register.html"
                )

            if (
                password
                != confirm_password
            ):
                flash(
                    (
                        "Password confirmation "
                        "does not match."
                    ),
                    "error",
                )

                return render_template(
                    "register.html"
                )

            if not password_is_strong(
                password
            ):
                flash(
                    (
                        "Use at least 10 characters "
                        "with letters and numbers."
                    ),
                    "error",
                )

                return render_template(
                    "register.html"
                )

            if User.query.filter_by(
                email=email
            ).first():
                flash(
                    (
                        "An account with that "
                        "email already exists."
                    ),
                    "error",
                )

                return render_template(
                    "register.html"
                )

            user = User(
                email=email,
                display_name=display_name,
            )

            user.set_password(
                password
            )

            organization = Organization(
                name=organization_name,
                slug=(
                    unique_organization_slug(
                        organization_name
                    )
                ),
            )

            membership = Membership(
                user=user,
                organization=organization,
                role="owner",
            )

            db.session.add_all(
                [
                    user,
                    organization,
                    membership,
                ]
            )

            db.session.commit()

            login_user(user)

            session[
                "organization_id"
            ] = organization.id

            flash(
                (
                    "Your Vaultify account "
                    "was created successfully."
                ),
                "success",
            )

            return redirect(
                url_for("dashboard")
            )

        return render_template(
            "register.html",
            title=(
                "Create account — Vaultify"
            ),
        )

    @app.route(
        "/login",
        methods=["GET", "POST"],
    )
    def login():
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard")
            )

        if request.method == "POST":
            email = normalize_email(
                request.form.get("email")
            )

            password = (
                request.form.get(
                    "password"
                )
                or ""
            )

            user = User.query.filter_by(
                email=email
            ).first()

            if (
                user is None
                or not user.is_active
                or not user.check_password(
                    password
                )
            ):
                flash(
                    (
                        "Invalid email address "
                        "or password."
                    ),
                    "error",
                )

                return render_template(
                    "login.html"
                )

            login_user(user)

            membership = (
                Membership.query
                .filter_by(
                    user_id=user.id
                )
                .order_by(
                    Membership.id.asc()
                )
                .first()
            )

            if membership is None:
                logout_user()

                flash(
                    (
                        "This account has no "
                        "organization membership."
                    ),
                    "error",
                )

                return render_template(
                    "login.html"
                )

            session[
                "organization_id"
            ] = (
                membership.organization_id
            )

            return redirect(
                url_for("dashboard")
            )

        return render_template(
            "login.html",
            title=(
                "Sign in — Vaultify"
            ),
        )

    @app.post("/logout")
    @login_required
    def logout():
        logout_user()
        session.clear()

        flash(
            "You have been signed out.",
            "success",
        )

        return redirect(
            url_for("login")
        )

    @app.post(
        "/organizations/switch"
    )
    @login_required
    def switch_organization():
        try:
            organization_id = int(
                request.form.get(
                    "organization_id",
                    "",
                )
            )
        except ValueError:
            abort(400)

        membership = (
            Membership.query.filter_by(
                user_id=current_user.id,
                organization_id=(
                    organization_id
                ),
            ).first()
        )

        if membership is None:
            abort(403)

        session[
            "organization_id"
        ] = (
            membership.organization_id
        )

        return redirect(
            request.referrer
            or url_for("dashboard")
        )

    @app.get("/")
    def index():
        if current_user.is_authenticated:
            return redirect(
                url_for("dashboard")
            )

        return redirect(
            url_for("login")
        )

    @app.get("/dashboard")
    @login_required
    def dashboard():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            return redirect(
                url_for("login")
            )

        organization = (
            membership.organization
        )

        document_count = (
            Document.query.filter_by(
                organization_id=(
                    organization.id
                )
            ).count()
        )

        chunk_count = (
            db.session.query(
                db.func.coalesce(
                    db.func.sum(
                        Document.chunk_count
                    ),
                    0,
                )
            )
            .filter(
                Document.organization_id
                == organization.id,
                Document.status
                == "ready",
            )
            .scalar()
        )

        return render_template(
            "dashboard.html",
            title=(
                "Dashboard — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=organization,
            document_count=(
                document_count
            ),
            chunk_count=int(
                chunk_count or 0
            ),
            question=None,
            answer=None,
            sources=[],
        )

    @app.post("/ask")
    @login_required
    def ask():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        question = (
            request.form.get(
                "question"
            )
            or ""
        ).strip()

        if not question:
            flash(
                (
                    "Enter a question "
                    "first."
                ),
                "error",
            )

            return redirect(
                url_for("dashboard")
            )

        services = app.config[
            "VAULTIFY_SERVICES"
        ]

        result = services[
            "answer_tenant_question"
        ](
            question=question,
            tenant_id=(
                membership.organization
                .tenant_id
            ),
        )

        answer = result["answer"]

        sources = serialize_sources(
            result["results"]
        )

        db.session.add(
            QueryLog(
                organization_id=(
                    membership
                    .organization_id
                ),
                user_id=current_user.id,
                question=question,
                answer=answer,
                source_count=len(
                    sources
                ),
            )
        )

        db.session.commit()

        document_count = (
            Document.query.filter_by(
                organization_id=(
                    membership
                    .organization_id
                )
            ).count()
        )

        chunk_count = (
            db.session.query(
                db.func.coalesce(
                    db.func.sum(
                        Document.chunk_count
                    ),
                    0,
                )
            )
            .filter(
                Document.organization_id
                == membership.organization_id,
                Document.status
                == "ready",
            )
            .scalar()
        )

        return render_template(
            "dashboard.html",
            title=(
                "Dashboard — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
            document_count=(
                document_count
            ),
            chunk_count=int(
                chunk_count or 0
            ),
            question=question,
            answer=answer,
            sources=sources,
        )

    @app.get("/documents")
    @login_required
    def documents():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        organization_documents = (
            Document.query
            .filter_by(
                organization_id=(
                    membership
                    .organization_id
                )
            )
            .order_by(
                Document.created_at.desc()
            )
            .all()
        )

        return render_template(
            "documents.html",
            title=(
                "Documents — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
            documents=(
                organization_documents
            ),
        )

    @app.route(
        "/documents/upload",
        methods=["GET", "POST"],
    )
    @login_required
    def upload_document():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        require_management_role(
            membership
        )

        if request.method == "POST":
            uploaded_file = (
                request.files.get("file")
            )

            if (
                uploaded_file is None
                or not uploaded_file.filename
            ):
                flash(
                    (
                        "Please select "
                        "a PDF document."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            original_filename = (
                uploaded_file.filename
                .strip()
            )

            safe_filename = (
                secure_filename(
                    original_filename
                )
            )

            if (
                not safe_filename
                or not safe_filename
                .lower()
                .endswith(".pdf")
            ):
                flash(
                    (
                        "Only PDF documents "
                        "are supported."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            file_bytes = (
                uploaded_file.read()
            )

            if not file_bytes:
                flash(
                    (
                        "The selected file "
                        "is empty."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            if not file_bytes.startswith(
                b"%PDF-"
            ):
                flash(
                    (
                        "The selected file "
                        "is not a valid PDF."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            guessed_type = filetype.guess(
                file_bytes[:8192]
            )

            if (
                guessed_type
                and guessed_type.mime
                != "application/pdf"
            ):
                flash(
                    (
                        "The selected file "
                        "signature is not PDF."
                    ),
                    "error",
                )

                return render_template(
                    "upload.html",
                    current_membership=(
                        membership
                    ),
                    current_org=(
                        membership.organization
                    ),
                )

            document_hash = (
                hashlib.sha256(
                    file_bytes
                ).hexdigest()
            )

            existing_document = (
                Document.query.filter_by(
                    organization_id=(
                        membership
                        .organization_id
                    ),
                    document_hash=(
                        document_hash
                    ),
                ).first()
            )

            if existing_document:
                flash(
                    (
                        "This document already "
                        "exists in the organization."
                    ),
                    "warning",
                )

                return redirect(
                    url_for("documents")
                )

            stored_filename = (
                f"{uuid.uuid4().hex}.pdf"
            )

            storage_path = (
                Path(
                    app.config[
                        "UPLOAD_FOLDER"
                    ]
                )
                / stored_filename
            )

            storage_path.write_bytes(
                file_bytes
            )

            document = Document(
                organization_id=(
                    membership
                    .organization_id
                ),
                original_filename=(
                    original_filename
                ),
                stored_filename=(
                    stored_filename
                ),
                storage_path=str(
                    storage_path
                ),
                mime_type=(
                    "application/pdf"
                ),
                size_bytes=len(
                    file_bytes
                ),
                document_hash=(
                    document_hash
                ),
                status="uploaded",
            )

            db.session.add(document)
            db.session.commit()

            try:
                chunk_count = (
                    ingest_document(
                        document,
                        app.config[
                            "VAULTIFY_SERVICES"
                        ],
                    )
                )

                flash(
                    (
                        f"{original_filename} "
                        f"was indexed successfully "
                        f"with {chunk_count} chunks."
                    ),
                    "success",
                )

            except Exception:
                flash(
                    (
                        "The PDF was stored, "
                        "but indexing failed safely. "
                        "Use Retry after checking "
                        "the runtime logs."
                    ),
                    "error",
                )

            return redirect(
                url_for("documents")
            )

        return render_template(
            "upload.html",
            title=(
                "Upload PDF — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
        )

    @app.post(
        "/documents/<int:document_id>/retry"
    )
    @login_required
    def retry_document(document_id):
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        require_management_role(
            membership
        )

        document = (
            Document.query.filter_by(
                id=document_id,
                organization_id=(
                    membership
                    .organization_id
                ),
            ).first_or_404()
        )

        try:
            chunk_count = (
                ingest_document(
                    document,
                    app.config[
                        "VAULTIFY_SERVICES"
                    ],
                )
            )

            flash(
                (
                    "The document was indexed "
                    f"with {chunk_count} chunks."
                ),
                "success",
            )

        except Exception:
            flash(
                (
                    "The retry failed safely."
                ),
                "error",
            )

        return redirect(
            url_for("documents")
        )

    @app.post(
        "/documents/<int:document_id>/delete"
    )
    @login_required
    def delete_document(document_id):
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        require_management_role(
            membership
        )

        document = (
            Document.query.filter_by(
                id=document_id,
                organization_id=(
                    membership
                    .organization_id
                ),
            ).first_or_404()
        )

        services = app.config[
            "VAULTIFY_SERVICES"
        ]

        delete_document_vectors(
            services,
            (
                membership.organization
                .tenant_id
            ),
            document.document_hash,
        )

        storage_path = Path(
            document.storage_path
        )

        db.session.delete(document)
        db.session.commit()

        if storage_path.exists():
            storage_path.unlink()

        flash(
            (
                "The document was deleted "
                "from storage and Qdrant."
            ),
            "success",
        )

        return redirect(
            url_for("documents")
        )

    @app.get("/organization")
    @login_required
    def organization():
        membership = (
            resolve_active_membership()
        )

        if membership is None:
            abort(403)

        return render_template(
            "organization.html",
            title=(
                "Organization — Vaultify"
            ),
            current_membership=(
                membership
            ),
            current_org=(
                membership.organization
            ),
        )

    return app
"""

WEB_BACKEND_PATH.write_text(
    textwrap.dedent(
        WEB_BACKEND_CODE
    ).strip() + "\n",
    encoding="utf-8",
)

compile(
    WEB_BACKEND_PATH.read_text(
        encoding="utf-8"
    ),
    str(WEB_BACKEND_PATH),
    "exec",
)

print(
    "✅ Compact Flask backend created."
)
print(
    f"📄 Backend module: "
    f"{WEB_BACKEND_PATH}"
)

✅ Compact Flask backend created.
📄 Backend module: /content/vaultify_compact_web/vaultify_web_backend.py


In [56]:
# VAULTIFY COMPACT - Cell 13: Initialize Flask safely after every runtime restart

import hashlib
import importlib
import secrets
import sys
from pathlib import Path

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)

# ---------------------------------------------------------------------
# 1. Verify that Cell 12 created the backend module
# ---------------------------------------------------------------------

WEB_BACKEND_PATH = (
    WEB_PROJECT_ROOT
    / "vaultify_web_backend.py"
)

if not WEB_BACKEND_PATH.exists():
    raise RuntimeError(
        "The Flask backend file is missing. "
        "Run Cell 12 before Cell 13."
    )

# ---------------------------------------------------------------------
# 2. Add the generated backend folder to Python's import path
# ---------------------------------------------------------------------

web_project_path = str(
    WEB_PROJECT_ROOT.resolve()
)

if web_project_path not in sys.path:
    sys.path.insert(
        0,
        web_project_path,
    )

# Remove an old cached module after rerunning Cell 12
if "vaultify_web_backend" in sys.modules:
    del sys.modules[
        "vaultify_web_backend"
    ]

vaultify_web_backend = (
    importlib.import_module(
        "vaultify_web_backend"
    )
)

from vaultify_web_backend import (
    Document,
    Membership,
    Organization,
    User,
    create_app,
    db,
)

print(
    "✅ Flask backend module imported successfully."
)

# ---------------------------------------------------------------------
# 3. Load or create the Flask secret key
# ---------------------------------------------------------------------

try:
    FLASK_SECRET_KEY = userdata.get(
        "FLASK_SECRET_KEY"
    )
except Exception:
    FLASK_SECRET_KEY = None

if not FLASK_SECRET_KEY:
    FLASK_SECRET_KEY = (
        secrets.token_urlsafe(48)
    )

    print(
        "ℹ️ FLASK_SECRET_KEY is not configured. "
        "A temporary runtime key was generated."
    )
else:
    print(
        "✅ FLASK_SECRET_KEY loaded from Colab Secrets."
    )

# ---------------------------------------------------------------------
# 4. Connect Flask to the already initialized Vaultify services
# ---------------------------------------------------------------------

WEB_SERVICES = {
    "qdrant": qdrant,
    "embedding_model": embedding_model,
    "answer_tenant_question": (
        answer_tenant_question
    ),
    "collection_name": COLLECTION_NAME,
}

vaultify_web_app = create_app(
    services=WEB_SERVICES,
    root_path=WEB_PROJECT_ROOT,
    secret_key=FLASK_SECRET_KEY,
)

print(
    "✅ Compact Flask application created."
)

# ---------------------------------------------------------------------
# 5. Create the temporary Apple demo account safely
# ---------------------------------------------------------------------

DEMO_LOGIN_EMAIL = (
    "runtime-demo@vaultify.local"
)

DEMO_LOGIN_PASSWORD = (
    "Vaultify-"
    + secrets.token_urlsafe(9)
    + "-9"
)

DEMO_ORGANIZATION_NAME = (
    "Vaultify Apple Demo"
)

DEMO_ORGANIZATION_SLUG = (
    "vaultify-apple-demo"
)

DEMO_TENANT_ID = (
    DEMO_TENANTS[
        "company_a"
    ]["tenant_id"]
)

SEED_METADATA_FILENAME = (
    "seed-apple-demo-metadata-only.pdf"
)

SEED_METADATA_HASH = hashlib.sha256(
    b"vaultify-apple-demo-metadata-only"
).hexdigest()

with vaultify_web_app.app_context():
    # Clear a possible failed transaction from an earlier execution.
    db.session.rollback()

    db.create_all()

    demo_organization = (
        Organization.query.filter_by(
            tenant_id=DEMO_TENANT_ID
        ).first()
    )

    if demo_organization is None:
        demo_organization = Organization(
            tenant_id=DEMO_TENANT_ID,
            name=DEMO_ORGANIZATION_NAME,
            slug=DEMO_ORGANIZATION_SLUG,
        )

        db.session.add(
            demo_organization
        )

    demo_user = User.query.filter_by(
        email=DEMO_LOGIN_EMAIL
    ).first()

    if demo_user is None:
        demo_user = User(
            email=DEMO_LOGIN_EMAIL,
            display_name="Demo User",
        )

        # Password must exist before SQLAlchemy inserts the row.
        demo_user.set_password(
            DEMO_LOGIN_PASSWORD
        )

        db.session.add(
            demo_user
        )

    else:
        demo_user.display_name = (
            "Demo User"
        )

        demo_user.set_password(
            DEMO_LOGIN_PASSWORD
        )

    # Generate database IDs only after required fields are populated.
    db.session.flush()

    demo_membership = (
        Membership.query.filter_by(
            user_id=demo_user.id,
            organization_id=(
                demo_organization.id
            ),
        ).first()
    )

    if demo_membership is None:
        demo_membership = Membership(
            user_id=demo_user.id,
            organization_id=(
                demo_organization.id
            ),
            role="owner",
        )

        db.session.add(
            demo_membership
        )

    seed_document = (
        Document.query.filter_by(
            organization_id=(
                demo_organization.id
            ),
            stored_filename=(
                SEED_METADATA_FILENAME
            ),
        ).first()
    )

    if seed_document is None:
        seed_document = Document(
            organization_id=(
                demo_organization.id
            ),
            original_filename=(
                "apple_fy2025_10k.pdf"
            ),
            stored_filename=(
                SEED_METADATA_FILENAME
            ),
            storage_path=(
                "/content/"
                "vaultify_compact_web/"
                "seed/"
                "apple_fy2025_10k.pdf"
            ),
            mime_type=(
                "application/pdf"
            ),
            size_bytes=0,
            document_hash=(
                SEED_METADATA_HASH
            ),
            status="ready",
            chunk_count=(
                tenant_counts[
                    "company_a"
                ]
            ),
        )

        db.session.add(
            seed_document
        )

    else:
        seed_document.status = (
            "ready"
        )

        seed_document.chunk_count = (
            tenant_counts[
                "company_a"
            ]
        )

    db.session.commit()

print(
    "✅ Demo organization, user, "
    "membership, and metadata created."
)

# ---------------------------------------------------------------------
# 6. Run authentication and organization smoke tests
# ---------------------------------------------------------------------

original_testing_setting = (
    vaultify_web_app.config.get(
        "TESTING",
        False,
    )
)

original_csrf_setting = (
    vaultify_web_app.config.get(
        "WTF_CSRF_ENABLED",
        True,
    )
)

try:
    vaultify_web_app.config[
        "TESTING"
    ] = True

    vaultify_web_app.config[
        "WTF_CSRF_ENABLED"
    ] = False

    with vaultify_web_app.test_client() as client:
        health_response = client.get(
            "/health"
        )

        login_response = client.post(
            "/login",
            data={
                "email": (
                    DEMO_LOGIN_EMAIL
                ),
                "password": (
                    DEMO_LOGIN_PASSWORD
                ),
            },
            follow_redirects=True,
        )

        dashboard_reached = (
            login_response.status_code
            == 200
            and b"Vaultify Apple Demo"
            in login_response.data
        )

        documents_response = client.get(
            "/documents"
        )

        organization_response = client.get(
            "/organization"
        )

        logout_response = client.post(
            "/logout",
            follow_redirects=True,
        )

finally:
    vaultify_web_app.config[
        "TESTING"
    ] = original_testing_setting

    vaultify_web_app.config[
        "WTF_CSRF_ENABLED"
    ] = original_csrf_setting

if health_response.status_code != 200:
    raise RuntimeError(
        "The Flask health check failed."
    )

if not dashboard_reached:
    raise RuntimeError(
        "The demo login did not reach the dashboard."
    )

if documents_response.status_code != 200:
    raise RuntimeError(
        "The documents page failed."
    )

if organization_response.status_code != 200:
    raise RuntimeError(
        "The organization page failed."
    )

if logout_response.status_code != 200:
    raise RuntimeError(
        "The logout flow failed."
    )

print(
    "✅ Authentication smoke test passed."
)

print(
    "✅ Trusted organization resolution passed."
)

print(
    "✅ Documents and organization pages passed."
)

print(
    "\n🔐 Temporary demo account:"
)

print(
    f"   Email: {DEMO_LOGIN_EMAIL}"
)

print(
    f"   Password: {DEMO_LOGIN_PASSWORD}"
)

print(
    "\n⚠️ This password and SQLite database "
    "belong only to the current Colab runtime."
)

✅ Flask backend module imported successfully.
ℹ️ FLASK_SECRET_KEY is not configured. A temporary runtime key was generated.
✅ Compact Flask application created.
✅ Demo organization, user, membership, and metadata created.
✅ Authentication smoke test passed.
✅ Trusted organization resolution passed.
✅ Documents and organization pages passed.

🔐 Temporary demo account:
   Email: runtime-demo@vaultify.local
   Password: Vaultify-SqVNZpHtktgT-9

⚠️ This password and SQLite database belong only to the current Colab runtime.


In [57]:
# VAULTIFY COMPACT - Cell 14: Start and test the local Flask web server

import socket
import threading
import time

import requests
from werkzeug.serving import (
    make_server,
)

WEB_HOST = "127.0.0.1"

WEB_PORT_CANDIDATES = range(
    5000,
    5011,
)


def web_port_is_open(
    host,
    port,
):
    """
    Return True when a TCP server is listening on the target port.
    """
    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    ) as connection_socket:
        connection_socket.settimeout(
            0.5
        )

        return (
            connection_socket.connect_ex(
                (
                    host,
                    port,
                )
            )
            == 0
        )


def find_available_web_port():
    """
    Return the first available port for the Flask web server.
    """
    for candidate_port in (
        WEB_PORT_CANDIDATES
    ):
        if not web_port_is_open(
            WEB_HOST,
            candidate_port,
        ):
            return candidate_port

    raise RuntimeError(
        "No available Vaultify web port "
        "was found between 5000 and 5010."
    )


if (
    "vaultify_web_server"
    in globals()
    and vaultify_web_server
    is not None
):
    try:
        vaultify_web_server.shutdown()

        print(
            "ℹ️ Previous Vaultify "
            "web server stopped."
        )

    except Exception as error:
        print(
            "⚠️ Previous web server "
            "shutdown warning: "
            f"{error}"
        )


WEB_PORT = (
    find_available_web_port()
)

LOCAL_WEB_URL = (
    f"http://{WEB_HOST}:{WEB_PORT}"
)

vaultify_web_server = make_server(
    WEB_HOST,
    WEB_PORT,
    vaultify_web_app,
    threaded=True,
)

vaultify_web_server_thread = (
    threading.Thread(
        target=(
            vaultify_web_server
            .serve_forever
        ),
        name="vaultify-web-server",
        daemon=True,
    )
)

vaultify_web_server_thread.start()

startup_deadline = (
    time.time() + 15
)

while (
    time.time()
    < startup_deadline
    and not web_port_is_open(
        WEB_HOST,
        WEB_PORT,
    )
):
    time.sleep(0.25)

if not web_port_is_open(
    WEB_HOST,
    WEB_PORT,
):
    raise RuntimeError(
        "The Vaultify web server "
        f"did not start on port "
        f"{WEB_PORT}."
    )

local_health_response = requests.get(
    f"{LOCAL_WEB_URL}/health",
    timeout=10,
)

if (
    local_health_response.status_code
    != 200
    or local_health_response
    .json()
    .get("status")
    != "ok"
):
    raise RuntimeError(
        "The local Vaultify "
        "web health check failed."
    )

print(
    "✅ Vaultify web server "
    "started locally."
)
print(
    f"🌐 Local web URL: "
    f"{LOCAL_WEB_URL}"
)
print(
    f"🩺 Health: "
    f"{local_health_response.json()}"
)
print(
    "✅ The MCP server on port "
    "8003 was preserved."
)

INFO:werkzeug:127.0.0.1 - - [23/Jul/2026 11:54:17] "GET /health HTTP/1.1" 200 -


ℹ️ Previous Vaultify web server stopped.
✅ Vaultify web server started locally.
🌐 Local web URL: http://127.0.0.1:5000
🩺 Health: {'phase': 3, 'service': 'Vaultify', 'status': 'ok'}
✅ The MCP server on port 8003 was preserved.


In [58]:
# VAULTIFY COMPACT - Cell 15: Create, expose, and verify the Flask website after every runtime restart
import re
import shutil
import socket
import subprocess
import threading
import time

from urllib.parse import urlparse

import requests


# ---------------------------------------------------------------------
# 1. Verify that the local Flask server is running
# ---------------------------------------------------------------------

if "LOCAL_WEB_URL" not in globals():
    raise RuntimeError(
        "LOCAL_WEB_URL is missing. "
        "Run Cell 14 before Cell 15."
    )

try:
    local_health_response = requests.get(
        f"{LOCAL_WEB_URL}/health",
        timeout=10,
    )

    local_health_response.raise_for_status()

except Exception as error:
    raise RuntimeError(
        "The local Flask website is not reachable. "
        "Run Cell 14 again."
    ) from error


if (
    local_health_response
    .json()
    .get("status")
    != "ok"
):
    raise RuntimeError(
        "The local Flask health response is invalid."
    )


print("✅ Local Flask website is healthy.")


# ---------------------------------------------------------------------
# 2. Install cloudflared when missing
# ---------------------------------------------------------------------

def install_cloudflared():
    """
    Install the Cloudflare tunnel client
    inside the current Colab runtime.
    """
    if shutil.which("cloudflared"):
        print("ℹ️ cloudflared is already installed.")
        return

    install_command = """
set -e

wget -q \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
  -O /tmp/cloudflared.deb

dpkg -i /tmp/cloudflared.deb >/dev/null

rm -f /tmp/cloudflared.deb
"""

    subprocess.run(
        [
            "bash",
            "-lc",
            install_command,
        ],
        check=True,
    )

    print("✅ cloudflared installed.")


install_cloudflared()


# ---------------------------------------------------------------------
# 3. Stop a previous web tunnel when this cell is rerun
# ---------------------------------------------------------------------

if (
    "web_cloudflare_tunnel_process"
    in globals()
    and web_cloudflare_tunnel_process
    is not None
    and web_cloudflare_tunnel_process.poll()
    is None
):
    print("ℹ️ Stopping the previous web tunnel.")

    web_cloudflare_tunnel_process.terminate()

    try:
        web_cloudflare_tunnel_process.wait(
            timeout=5
        )

    except subprocess.TimeoutExpired:
        web_cloudflare_tunnel_process.kill()

        web_cloudflare_tunnel_process.wait(
            timeout=5
        )


# ---------------------------------------------------------------------
# 4. Start a new Cloudflare Quick Tunnel
# ---------------------------------------------------------------------

web_cloudflare_logs = []


def read_cloudflare_logs(
    process,
    storage,
):
    """
    Read Cloudflare logs continuously
    without blocking the notebook.
    """
    if process.stderr is None:
        return

    for line in iter(
        process.stderr.readline,
        "",
    ):
        cleaned_line = line.strip()

        if cleaned_line:
            storage.append(
                cleaned_line
            )

            print(
                f"[web-cloudflared] "
                f"{cleaned_line}"
            )

        if process.poll() is not None:
            break


def extract_public_url(
    log_lines,
):
    """
    Extract the generated TryCloudflare URL.
    """
    pattern = re.compile(
        r"https://[a-zA-Z0-9-]+"
        r"\.trycloudflare\.com"
    )

    for line in log_lines:
        match = pattern.search(
            line
        )

        if match:
            return match.group(0)

    return None


def tunnel_registered(
    log_lines,
):
    """
    Check whether Cloudflare registered
    the tunnel connection.
    """
    return any(
        "Registered tunnel connection"
        in line
        for line in log_lines
    )


web_cloudflare_tunnel_process = (
    subprocess.Popen(
        [
            "cloudflared",
            "tunnel",
            "--url",
            LOCAL_WEB_URL,
            "--no-autoupdate",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )
)


web_cloudflare_log_thread = (
    threading.Thread(
        target=read_cloudflare_logs,
        args=(
            web_cloudflare_tunnel_process,
            web_cloudflare_logs,
        ),
        name=(
            "vaultify-web-"
            "cloudflare-log-reader"
        ),
        daemon=True,
    )
)


web_cloudflare_log_thread.start()


TUNNEL_STARTUP_TIMEOUT = 60

startup_deadline = (
    time.time()
    + TUNNEL_STARTUP_TIMEOUT
)

VAULTIFY_WEB_PUBLIC_URL = None


while time.time() < startup_deadline:
    if (
        web_cloudflare_tunnel_process
        .poll()
        is not None
    ):
        raise RuntimeError(
            "cloudflared stopped unexpectedly. "
            f"Recent logs: "
            f"{web_cloudflare_logs[-10:]}"
        )

    VAULTIFY_WEB_PUBLIC_URL = (
        extract_public_url(
            web_cloudflare_logs
        )
    )

    if (
        VAULTIFY_WEB_PUBLIC_URL
        and tunnel_registered(
            web_cloudflare_logs
        )
    ):
        break

    time.sleep(0.25)


if not VAULTIFY_WEB_PUBLIC_URL:
    raise RuntimeError(
        "Cloudflare did not provide "
        "a public website URL."
    )


if not tunnel_registered(
    web_cloudflare_logs
):
    raise RuntimeError(
        "The public URL was created, "
        "but the tunnel connection "
        "was not registered."
    )


print("\n✅ Cloudflare web tunnel started.")
print(
    f"🌍 Public website: "
    f"{VAULTIFY_WEB_PUBLIC_URL}"
)


# ---------------------------------------------------------------------
# 5. Wait for the new hostname to appear in DNS
# ---------------------------------------------------------------------

def query_dns_over_https(
    hostname,
):
    """
    Query Google and Cloudflare DNS
    for the newly created tunnel hostname.
    """
    providers = [
        {
            "url": (
                "https://dns.google/resolve"
            ),
            "headers": {},
        },
        {
            "url": (
                "https://cloudflare-dns.com/"
                "dns-query"
            ),
            "headers": {
                "accept": (
                    "application/dns-json"
                )
            },
        },
    ]

    for provider in providers:
        try:
            response = requests.get(
                provider["url"],
                params={
                    "name": hostname,
                    "type": "A",
                },
                headers=(
                    provider["headers"]
                ),
                timeout=15,
            )

            response.raise_for_status()

            answers = (
                response.json()
                .get("Answer", [])
            )

            ipv4_addresses = [
                answer.get("data")
                for answer in answers
                if (
                    answer.get("type") == 1
                    and answer.get("data")
                )
            ]

            if ipv4_addresses:
                return ipv4_addresses[0]

        except Exception:
            continue

    return None


def add_hosts_entry(
    hostname,
    ipv4_address,
):
    """
    Add a temporary DNS mapping
    to the Colab virtual machine.
    """
    with open(
        "/etc/hosts",
        "r",
        encoding="utf-8",
    ) as hosts_file:
        existing_lines = (
            hosts_file.readlines()
        )

    filtered_lines = [
        line
        for line in existing_lines
        if hostname not in line
    ]

    filtered_lines.append(
        f"{ipv4_address} "
        f"{hostname}\n"
    )

    with open(
        "/etc/hosts",
        "w",
        encoding="utf-8",
    ) as hosts_file:
        hosts_file.writelines(
            filtered_lines
        )


hostname = urlparse(
    VAULTIFY_WEB_PUBLIC_URL
).hostname


if not hostname:
    raise RuntimeError(
        "The generated tunnel hostname "
        "is invalid."
    )


DNS_WAIT_SECONDS = 120
DNS_RETRY_SECONDS = 5

dns_deadline = (
    time.time()
    + DNS_WAIT_SECONDS
)

resolved_ip = None
dns_attempt = 0


while time.time() < dns_deadline:
    dns_attempt += 1

    if (
        web_cloudflare_tunnel_process
        .poll()
        is not None
    ):
        raise RuntimeError(
            "The Cloudflare tunnel stopped "
            "while waiting for DNS."
        )

    try:
        resolved_ip = (
            socket.gethostbyname(
                hostname
            )
        )

        print(
            f"✅ DNS resolved normally: "
            f"{hostname} → {resolved_ip}"
        )

        break

    except socket.gaierror:
        resolved_ip = (
            query_dns_over_https(
                hostname
            )
        )

        if resolved_ip:
            add_hosts_entry(
                hostname,
                resolved_ip,
            )

            print(
                f"✅ DNS-over-HTTPS resolved: "
                f"{hostname} → {resolved_ip}"
            )

            break

    print(
        f"⏳ Waiting for tunnel DNS "
        f"(attempt {dns_attempt})..."
    )

    time.sleep(
        DNS_RETRY_SECONDS
    )


if not resolved_ip:
    raise RuntimeError(
        "The Cloudflare hostname did not "
        "become available in DNS."
    )


# ---------------------------------------------------------------------
# 6. Verify the website through the public tunnel
# ---------------------------------------------------------------------

PUBLIC_TEST_ATTEMPTS = 20
PUBLIC_TEST_RETRY_SECONDS = 3

public_health_response = None
last_public_error = None


for attempt_number in range(
    1,
    PUBLIC_TEST_ATTEMPTS + 1,
):
    try:
        public_health_response = (
            requests.get(
                (
                    VAULTIFY_WEB_PUBLIC_URL
                    + "/health"
                ),
                timeout=15,
            )
        )

        if (
            public_health_response
            .status_code
            == 200
            and public_health_response
            .json()
            .get("status")
            == "ok"
        ):
            break

        last_public_error = RuntimeError(
            "Unexpected public health "
            "status: "
            f"{public_health_response.status_code}"
        )

    except Exception as error:
        last_public_error = error

    print(
        f"⏳ Public website test "
        f"{attempt_number}/"
        f"{PUBLIC_TEST_ATTEMPTS}..."
    )

    if (
        attempt_number
        < PUBLIC_TEST_ATTEMPTS
    ):
        time.sleep(
            PUBLIC_TEST_RETRY_SECONDS
        )


if (
    public_health_response is None
    or public_health_response.status_code
    != 200
):
    raise RuntimeError(
        "The public Vaultify website "
        "remained unreachable."
    ) from last_public_error


# ---------------------------------------------------------------------
# 7. Print the final runtime URLs
# ---------------------------------------------------------------------

print(
    "\n✅ Public Vaultify website "
    "health check passed."
)

print(
    f"🌍 Website: "
    f"{VAULTIFY_WEB_PUBLIC_URL}"
)

print(
    f"📝 Register: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/register"
)

print(
    f"🔐 Login: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/login"
)

print(
    f"📊 Dashboard: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/dashboard"
)

print(
    f"📄 Documents: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/documents"
)

print(
    f"🏢 Organization: "
    f"{VAULTIFY_WEB_PUBLIC_URL}/organization"
)


print(
    "\n🔐 Temporary demo account:"
)

print(
    f"   Email: "
    f"{DEMO_LOGIN_EMAIL}"
)

print(
    f"   Password: "
    f"{DEMO_LOGIN_PASSWORD}"
)


if "PUBLIC_MCP_URL" in globals():
    print(
        "\n🔗 MCP endpoint:"
    )

    print(
        PUBLIC_MCP_URL
    )


print(
    "\n⚠️ The website URL is temporary "
    "and remains active only while "
    "this Colab runtime is connected."
)

INFO:werkzeug:127.0.0.1 - - [23/Jul/2026 11:54:18] "GET /health HTTP/1.1" 200 -


✅ Local Flask website is healthy.
ℹ️ cloudflared is already installed.
ℹ️ Stopping the previous web tunnel.
[web-cloudflared] 2026-07-23T11:54:18Z INF Initiating graceful shutdown due to signal terminated ...
[web-cloudflared] 2026-07-23T11:54:18Z ERR failed to run the datagram handler error="context canceled" connIndex=0 event=0 ip=198.41.200.233
[web-cloudflared] 2026-07-23T11:54:18Z ERR failed to serve tunnel connection error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.233
[web-cloudflared] 2026-07-23T11:54:18Z ERR Serve tunnel error error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.233
[web-cloudflared] 2026-07-23T11:54:18Z INF Retrying connection in up to 1s connIndex=0 event=0 ip=198.41.200.233
[web-cloudflared] 2026-07-23T11:54:18Z ERR Connection terminated connIndex=0
[web-cloudflared] 2026-07-23T11:54:18Z ERR no more connections active and exiting
[web-cloudflared] 2026-07-23T

INFO:werkzeug:127.0.0.1 - - [23/Jul/2026 11:54:28] "GET /health HTTP/1.1" 200 -



✅ Public Vaultify website health check passed.
🌍 Website: https://hdtv-kills-variance-detail.trycloudflare.com
📝 Register: https://hdtv-kills-variance-detail.trycloudflare.com/register
🔐 Login: https://hdtv-kills-variance-detail.trycloudflare.com/login
📊 Dashboard: https://hdtv-kills-variance-detail.trycloudflare.com/dashboard
📄 Documents: https://hdtv-kills-variance-detail.trycloudflare.com/documents
🏢 Organization: https://hdtv-kills-variance-detail.trycloudflare.com/organization

🔐 Temporary demo account:
   Email: runtime-demo@vaultify.local
   Password: Vaultify-SqVNZpHtktgT-9

🔗 MCP endpoint:
https://sarah-banner-refugees-sydney.trycloudflare.com/mcp

⚠️ The website URL is temporary and remains active only while this Colab runtime is connected.


In [59]:
# VAULTIFY COMPACT - Cell 16:
# Audit old and web-upload Tesla chunks directly from Qdrant

from collections import Counter, defaultdict
import statistics

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)


OLD_TESLA_TENANT_ID = (
    DEMO_TENANTS["company_b"]["tenant_id"]
)


def load_filtered_points(
    qdrant_filter,
):
    """
    Load every Qdrant point matching one filter.
    """
    points = []
    next_offset = None

    while True:
        batch, next_offset = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=qdrant_filter,
            limit=256,
            offset=next_offset,
            with_payload=True,
            with_vectors=False,
        )

        points.extend(batch)

        if next_offset is None:
            break

    return points


def load_tenant_points(
    tenant_id,
):
    """
    Load every Qdrant point belonging to one tenant.
    """
    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant_id
                ),
            )
        ]
    )

    return load_filtered_points(
        tenant_filter
    )


def normalize_chunk_text(text):
    """
    Normalize whitespace for exact duplicate detection.
    """
    return " ".join(
        str(text or "").split()
    )


def count_tokens_without_truncation(text):
    """
    Count tokenizer IDs without allowing silent truncation.
    """
    tokenized = (
        embedding_model.tokenizer(
            str(text or ""),
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )
    )

    return len(
        tokenized["input_ids"]
    )


def percentile(
    values,
    percentage,
):
    """
    Calculate one percentile without external dependencies.
    """
    if not values:
        return 0

    sorted_values = sorted(values)

    index = int(
        round(
            (len(sorted_values) - 1)
            * percentage
        )
    )

    return sorted_values[index]


# ---------------------------------------------------------------------
# 1. Recover the Tesla document hash from the validated seed tenant
# ---------------------------------------------------------------------

old_tesla_points = load_tenant_points(
    OLD_TESLA_TENANT_ID
)

if not old_tesla_points:
    raise RuntimeError(
        "The original Tesla seed points were not found in Qdrant."
    )

old_hash_counts = Counter(
    (point.payload or {}).get(
        DOCUMENT_HASH_FIELD
    )
    for point in old_tesla_points
    if (point.payload or {}).get(
        DOCUMENT_HASH_FIELD
    )
)

if not old_hash_counts:
    raise RuntimeError(
        "The original Tesla points contain no document hash."
    )

TESLA_DOCUMENT_HASH = (
    old_hash_counts.most_common(1)[0][0]
)

print(
    "✅ Tesla document hash recovered "
    "from the original seed tenant."
)
print(
    f"🔑 Hash: {TESLA_DOCUMENT_HASH}"
)


# ---------------------------------------------------------------------
# 2. Find every tenant containing the same Tesla PDF
# ---------------------------------------------------------------------

tesla_hash_filter = Filter(
    must=[
        FieldCondition(
            key=DOCUMENT_HASH_FIELD,
            match=MatchValue(
                value=TESLA_DOCUMENT_HASH
            ),
        )
    ]
)

all_tesla_document_points = (
    load_filtered_points(
        tesla_hash_filter
    )
)

points_by_tenant = defaultdict(list)

for point in all_tesla_document_points:
    payload = point.payload or {}

    tenant_id = payload.get(
        TENANT_ID_FIELD
    )

    if tenant_id:
        points_by_tenant[
            tenant_id
        ].append(point)


print("\n📦 Tesla copies found in Qdrant:")

for tenant_id, points in sorted(
    points_by_tenant.items(),
    key=lambda item: len(item[1]),
):
    filenames = sorted(
        {
            (point.payload or {}).get(
                FILENAME_FIELD,
                "Unknown file",
            )
            for point in points
        }
    )

    print(
        f"   {tenant_id}: "
        f"{len(points)} points "
        f"— {filenames}"
    )


web_tenant_candidates = {
    tenant_id: points
    for tenant_id, points
    in points_by_tenant.items()
    if tenant_id
    != OLD_TESLA_TENANT_ID
}

if not web_tenant_candidates:
    raise RuntimeError(
        "No separate web-upload Tesla tenant was found in Qdrant. "
        "The runtime database was deleted, and the web-upload vectors "
        "may also have been removed or written to another collection."
    )

# Select the largest separate Tesla tenant.
NEW_TESLA_TENANT_ID = max(
    web_tenant_candidates,
    key=lambda tenant_id: len(
        web_tenant_candidates[
            tenant_id
        ]
    ),
)

print(
    "\n✅ Web-upload Tesla tenant "
    "recovered directly from Qdrant."
)
print(
    f"🏢 Tenant: "
    f"{NEW_TESLA_TENANT_ID}"
)


# ---------------------------------------------------------------------
# 3. Audit utility
# ---------------------------------------------------------------------

def summarize_tenant_chunks(
    label,
    tenant_id,
):
    """
    Print chunk types, token sizes, filenames,
    hashes, and duplicate counts.
    """
    points = load_tenant_points(
        tenant_id
    )

    payloads = [
        point.payload or {}
        for point in points
    ]

    texts = [
        normalize_chunk_text(
            payload.get(TEXT_FIELD)
        )
        for payload in payloads
    ]

    texts = [
        text
        for text in texts
        if text
    ]

    token_counts = [
        count_tokens_without_truncation(
            text
        )
        for text in texts
    ]

    chunk_types = Counter(
        payload.get(
            CHUNK_TYPE_FIELD,
            "unknown",
        )
        for payload in payloads
    )

    filenames = sorted(
        {
            payload.get(
                FILENAME_FIELD,
                "Unknown file",
            )
            for payload in payloads
        }
    )

    document_hashes = sorted(
        {
            payload.get(
                DOCUMENT_HASH_FIELD,
                "Missing hash",
            )
            for payload in payloads
        }
    )

    duplicate_count = (
        len(texts)
        - len(set(texts))
    )

    oversized_256_count = sum(
        token_count > 256
        for token_count
        in token_counts
    )

    oversized_240_count = sum(
        token_count > 240
        for token_count
        in token_counts
    )

    print("\n" + "=" * 88)
    print(f"📊 {label}")
    print(f"🏢 Tenant: {tenant_id}")
    print(f"📦 Total points: {len(points)}")
    print(f"📄 Files: {filenames}")
    print(f"🔑 Document hashes: {document_hashes}")
    print(
        f"🧩 Chunk types: "
        f"{dict(chunk_types)}"
    )
    print(
        f"♻️ Exact duplicate chunks: "
        f"{duplicate_count}"
    )

    if token_counts:
        print(
            "📐 Token counts:"
            f" min={min(token_counts)},"
            f" median="
            f"{statistics.median(token_counts):.1f},"
            f" p95="
            f"{percentile(token_counts, 0.95)},"
            f" max={max(token_counts)}"
        )

    print(
        f"⚠️ Chunks above 240 tokens: "
        f"{oversized_240_count}"
    )

    print(
        f"⚠️ Chunks above 256 tokens: "
        f"{oversized_256_count}"
    )

    return {
        "points": points,
        "texts": texts,
        "token_counts": token_counts,
        "chunk_types": chunk_types,
        "duplicates": duplicate_count,
        "oversized_240": (
            oversized_240_count
        ),
        "oversized_256": (
            oversized_256_count
        ),
    }


# ---------------------------------------------------------------------
# 4. Compare both pipelines
# ---------------------------------------------------------------------

old_tesla_audit = (
    summarize_tenant_chunks(
        label=(
            "Original seeded "
            "Tesla pipeline"
        ),
        tenant_id=(
            OLD_TESLA_TENANT_ID
        ),
    )
)

new_tesla_audit = (
    summarize_tenant_chunks(
        label=(
            "Web-upload "
            "Tesla pipeline"
        ),
        tenant_id=(
            NEW_TESLA_TENANT_ID
        ),
    )
)


old_count = len(
    old_tesla_audit["points"]
)

new_count = len(
    new_tesla_audit["points"]
)


print("\n" + "=" * 88)
print("🔍 Comparison")

print(
    f"Original seeded pipeline: "
    f"{old_count} chunks"
)

print(
    f"Web-upload pipeline: "
    f"{new_count} chunks"
)

if old_count:
    print(
        "Web/seed chunk ratio: "
        f"{new_count / old_count:.2f}x"
    )


new_pipeline_is_safe = (
    new_tesla_audit[
        "duplicates"
    ]
    == 0
    and new_tesla_audit[
        "oversized_240"
    ]
    == 0
)


if new_pipeline_is_safe:
    print(
        "✅ The web-upload chunks contain "
        "no exact duplicates and stay "
        "within the 240-token safety limit."
    )

else:
    print(
        "⚠️ The web-upload chunk set "
        "requires correction before becoming "
        "the final ingestion strategy."
    )

[web-cloudflared] 2026-07-23T11:54:28Z INF +-------------------------------------------------------------------------------------+
[web-cloudflared] 2026-07-23T11:54:28Z INF |                               CONNECTIVITY PRE-CHECKS                               |
[web-cloudflared] 2026-07-23T11:54:28Z INF +-------------------------------------------------------------------------------------+
[web-cloudflared] 2026-07-23T11:54:28Z INF |  COMPONENT         TARGET                     STATUS  DETAILS                       |
[web-cloudflared] 2026-07-23T11:54:28Z INF |  DNS Resolution    region1.v2.argotunnel.com  PASS    DNS Resolved successfully     |
[web-cloudflared] 2026-07-23T11:54:28Z INF |  DNS Resolution    region2.v2.argotunnel.com  PASS    DNS Resolved successfully     |
[web-cloudflared] 2026-07-23T11:54:28Z INF |  UDP Connectivity  region1.v2.argotunnel.com  PASS    QUIC connection successful    |
[web-cloudflared] 2026-07-23T11:54:28Z INF |  UDP Connectivity  region2.v2.argotunn

In [60]:
# VAULTIFY COMPACT - Cell 17:
# Install a token-safe, table-aware, duplicate-free chunker

import hashlib
import re

import vaultify_web_backend


# Leave a safety margin below the model's 256-token maximum.
MAX_SAFE_CHUNK_TOKENS = min(
    240,
    embedding_model.max_seq_length - 16,
)

TOKENIZER = embedding_model.tokenizer


def tokenize_ids(
    text,
    *,
    add_special_tokens,
):
    """
    Tokenize through the public tokenizer API
    without truncating the input.
    """
    encoded = TOKENIZER(
        str(text or ""),
        add_special_tokens=add_special_tokens,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return list(token_ids)


empty_raw_ids = tokenize_ids(
    "",
    add_special_tokens=False,
)

empty_complete_ids = tokenize_ids(
    "",
    add_special_tokens=True,
)

SPECIAL_TOKEN_COUNT = max(
    0,
    len(empty_complete_ids)
    - len(empty_raw_ids),
)

RAW_TOKEN_BUDGET = (
    MAX_SAFE_CHUNK_TOKENS
    - SPECIAL_TOKEN_COUNT
)

TEXT_OVERLAP_TOKENS = 30


if RAW_TOKEN_BUDGET < 32:
    raise RuntimeError(
        "The calculated raw token budget is unexpectedly small."
    )


def get_raw_token_ids(text):
    """
    Return token IDs without special tokens or truncation.
    """
    return tokenize_ids(
        text,
        add_special_tokens=False,
    )


def count_safe_tokens(text):
    """
    Count the complete tokenizer input,
    including special tokens and without truncation.
    """
    return len(
        tokenize_ids(
            text,
            add_special_tokens=True,
        )
    )


def decode_token_ids(token_ids):
    """
    Convert tokenizer IDs back into readable text.
    """
    return TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()


print(
    f"ℹ️ Tokenizer: "
    f"{TOKENIZER.__class__.__name__}"
)

print(
    f"ℹ️ Special-token overhead: "
    f"{SPECIAL_TOKEN_COUNT}"
)

print(
    f"ℹ️ Raw payload budget: "
    f"{RAW_TOKEN_BUDGET}"
)

def split_by_exact_tokens(
    text,
    overlap_tokens=0,
):
    """
    Split text into windows that can never exceed the embedding limit.
    """
    text = str(text or "").strip()

    if not text:
        return []

    raw_ids = get_raw_token_ids(text)

    if len(raw_ids) <= RAW_TOKEN_BUDGET:
        return [text]

    overlap_tokens = max(
        0,
        min(
            overlap_tokens,
            RAW_TOKEN_BUDGET - 1,
        ),
    )

    step_size = (
        RAW_TOKEN_BUDGET
        - overlap_tokens
    )

    pieces = []
    start_index = 0

    while start_index < len(raw_ids):
        end_index = min(
            start_index + RAW_TOKEN_BUDGET,
            len(raw_ids),
        )

        token_slice = raw_ids[
            start_index:end_index
        ]

        decoded_piece = decode_token_ids(
            token_slice
        )

        if decoded_piece:
            pieces.append(
                decoded_piece
            )

        if end_index >= len(raw_ids):
            break

        start_index += step_size

    return pieces


def normalize_chunk_for_dedup(text):
    """
    Normalize formatting differences before exact duplicate comparison.
    """
    return re.sub(
        r"\s+",
        " ",
        str(text or ""),
    ).strip()


def create_chunk_record(
    text,
    chunk_type,
    section,
):
    """
    Create one consistently structured chunk dictionary.
    """
    return {
        "text": str(text).strip(),
        "chunk_type": chunk_type,
        "section": (
            str(section).strip()
            or "Document content"
        ),
    }


def split_normal_text_safely(
    text,
    section,
):
    """
    Split ordinary document text with a small token overlap.
    """
    return [
        create_chunk_record(
            text=piece,
            chunk_type="text",
            section=section,
        )
        for piece in split_by_exact_tokens(
            text,
            overlap_tokens=(
                TEXT_OVERLAP_TOKENS
            ),
        )
        if piece.strip()
    ]


def identify_table_header_lines(
    table_lines,
):
    """
    Preserve the Markdown table header and separator when present.
    """
    if not table_lines:
        return [], []

    has_separator_row = (
        len(table_lines) >= 2
        and re.match(
            r"^\|\s*:?-{3,}",
            table_lines[1].strip(),
        )
    )

    header_line_count = (
        2 if has_separator_row else 1
    )

    return (
        table_lines[:header_line_count],
        table_lines[header_line_count:],
    )


def split_table_safely(
    table_lines,
    section,
):
    """
    Split a Markdown table by rows while repeating its header.

    A single oversized row is split directly by tokenizer IDs.
    """
    if not table_lines:
        return []

    header_lines, data_rows = (
        identify_table_header_lines(
            table_lines
        )
    )

    table_prefix = "\n".join(
        [
            str(section).strip(),
            "",
            *header_lines,
        ]
    ).strip()

    # Extremely large headers fall back to exact token windows.
    if (
        count_safe_tokens(table_prefix)
        >= MAX_SAFE_CHUNK_TOKENS
    ):
        entire_table = "\n".join(
            [
                table_prefix,
                *data_rows,
            ]
        ).strip()

        return [
            create_chunk_record(
                text=piece,
                chunk_type="table",
                section=section,
            )
            for piece in split_by_exact_tokens(
                entire_table,
                overlap_tokens=0,
            )
        ]

    if not data_rows:
        return [
            create_chunk_record(
                text=piece,
                chunk_type="table",
                section=section,
            )
            for piece in split_by_exact_tokens(
                table_prefix,
                overlap_tokens=0,
            )
        ]

    table_chunks = []
    current_rows = []

    def flush_current_rows():
        nonlocal current_rows

        if not current_rows:
            return

        table_text = "\n".join(
            [
                table_prefix,
                *current_rows,
            ]
        ).strip()

        for piece in split_by_exact_tokens(
            table_text,
            overlap_tokens=0,
        ):
            table_chunks.append(
                create_chunk_record(
                    text=piece,
                    chunk_type="table",
                    section=section,
                )
            )

        current_rows = []

    for row in data_rows:
        candidate_rows = (
            current_rows + [row]
        )

        candidate_text = "\n".join(
            [
                table_prefix,
                *candidate_rows,
            ]
        ).strip()

        if (
            count_safe_tokens(candidate_text)
            <= MAX_SAFE_CHUNK_TOKENS
        ):
            current_rows = candidate_rows
            continue

        flush_current_rows()

        single_row_text = "\n".join(
            [
                table_prefix,
                row,
            ]
        ).strip()

        if (
            count_safe_tokens(single_row_text)
            <= MAX_SAFE_CHUNK_TOKENS
        ):
            current_rows = [row]
            continue

        # The individual row itself is oversized.
        prefix_raw_ids = get_raw_token_ids(
            table_prefix
        )

        available_row_budget = (
            RAW_TOKEN_BUDGET
            - len(prefix_raw_ids)
            - 2
        )

        if available_row_budget < 16:
            for piece in split_by_exact_tokens(
                single_row_text,
                overlap_tokens=0,
            ):
                table_chunks.append(
                    create_chunk_record(
                        text=piece,
                        chunk_type="table",
                        section=section,
                    )
                )

            continue

        row_raw_ids = get_raw_token_ids(
            row
        )

        for start_index in range(
            0,
            len(row_raw_ids),
            available_row_budget,
        ):
            row_slice = row_raw_ids[
                start_index:
                start_index
                + available_row_budget
            ]

            decoded_row = decode_token_ids(
                row_slice
            )

            safe_row_text = "\n".join(
                [
                    table_prefix,
                    decoded_row,
                ]
            ).strip()

            table_chunks.append(
                create_chunk_record(
                    text=safe_row_text,
                    chunk_type="table",
                    section=section,
                )
            )

    flush_current_rows()

    return table_chunks


def enforce_final_token_safety(
    chunks,
):
    """
    Apply one final hard token check to every generated chunk.
    """
    safe_chunks = []

    for chunk in chunks:
        chunk_text = chunk["text"]

        if (
            count_safe_tokens(chunk_text)
            <= MAX_SAFE_CHUNK_TOKENS
        ):
            safe_chunks.append(chunk)
            continue

        for piece in split_by_exact_tokens(
            chunk_text,
            overlap_tokens=0,
        ):
            safe_chunks.append(
                create_chunk_record(
                    text=piece,
                    chunk_type=(
                        chunk["chunk_type"]
                    ),
                    section=(
                        chunk["section"]
                    ),
                )
            )

    return safe_chunks


def remove_exact_duplicate_chunks(
    chunks,
):
    """
    Remove repeated chunks before embeddings and Qdrant writes.
    """
    unique_chunks = []
    seen_hashes = set()

    for chunk in chunks:
        normalized_text = (
            normalize_chunk_for_dedup(
                chunk["text"]
            )
        )

        if not normalized_text:
            continue

        chunk_hash = hashlib.sha256(
            normalized_text.encode(
                "utf-8"
            )
        ).hexdigest()

        if chunk_hash in seen_hashes:
            continue

        seen_hashes.add(chunk_hash)

        unique_chunks.append(chunk)

    for chunk_index, chunk in enumerate(
        unique_chunks
    ):
        chunk["chunk_index"] = (
            chunk_index
        )

    return unique_chunks


def safe_smart_chunk_markdown(
    model,
    markdown_text,
):
    """
    Convert Markdown into table-aware, token-safe, duplicate-free chunks.
    """
    lines = str(
        markdown_text or ""
    ).splitlines()

    chunks = []
    current_section = (
        "Document introduction"
    )

    text_buffer = []

    def flush_text_buffer():
        nonlocal text_buffer

        buffered_text = "\n".join(
            text_buffer
        ).strip()

        if buffered_text:
            chunks.extend(
                split_normal_text_safely(
                    text=buffered_text,
                    section=current_section,
                )
            )

        text_buffer = []

    line_index = 0

    while line_index < len(lines):
        line = lines[line_index]
        stripped_line = line.strip()

        heading_match = re.match(
            r"^(#{1,6})\s+(.+)$",
            stripped_line,
        )

        if heading_match:
            flush_text_buffer()

            current_section = (
                heading_match
                .group(2)
                .strip()
            )

            text_buffer.append(
                stripped_line
            )

            line_index += 1
            continue

        if stripped_line.startswith("|"):
            flush_text_buffer()

            table_lines = []

            while (
                line_index < len(lines)
                and lines[line_index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[line_index]
                )

                line_index += 1

            chunks.extend(
                split_table_safely(
                    table_lines=table_lines,
                    section=current_section,
                )
            )

            continue

        text_buffer.append(line)
        line_index += 1

    flush_text_buffer()

    chunks = enforce_final_token_safety(
        chunks
    )

    chunks = remove_exact_duplicate_chunks(
        chunks
    )

    oversized_chunks = [
        chunk
        for chunk in chunks
        if (
            count_safe_tokens(
                chunk["text"]
            )
            > MAX_SAFE_CHUNK_TOKENS
        )
    ]

    if oversized_chunks:
        raise RuntimeError(
            "Token safety validation failed. "
            f"{len(oversized_chunks)} oversized "
            "chunks remain."
        )

    return chunks


# Patch the generated Flask backend for this runtime.
vaultify_web_backend.smart_chunk_markdown = (
    safe_smart_chunk_markdown
)

vaultify_web_backend.ingest_document.__globals__[
    "smart_chunk_markdown"
] = safe_smart_chunk_markdown


# ---------------------------------------------------------------------
# Synthetic safety test
# ---------------------------------------------------------------------

synthetic_markdown = "\n".join(
    [
        "# Synthetic Safety Test",
        "",
        " ".join(
            ["financial-data"] * 900
        ),
        "",
        "| Metric | Value |",
        "|---|---|",
        (
            "| Oversized row | "
            + " ".join(
                ["123456"] * 700
            )
            + " |"
        ),
        (
            "| Oversized row | "
            + " ".join(
                ["123456"] * 700
            )
            + " |"
        ),
    ]
)

synthetic_chunks = (
    safe_smart_chunk_markdown(
        embedding_model,
        synthetic_markdown,
    )
)

synthetic_token_counts = [
    count_safe_tokens(
        chunk["text"]
    )
    for chunk in synthetic_chunks
]

synthetic_normalized_texts = [
    normalize_chunk_for_dedup(
        chunk["text"]
    )
    for chunk in synthetic_chunks
]

synthetic_duplicate_count = (
    len(synthetic_normalized_texts)
    - len(set(synthetic_normalized_texts))
)

if max(synthetic_token_counts) > (
    MAX_SAFE_CHUNK_TOKENS
):
    raise RuntimeError(
        "Synthetic token safety test failed."
    )

if synthetic_duplicate_count != 0:
    raise RuntimeError(
        "Synthetic duplicate-removal test failed."
    )


print("✅ Safe web chunker installed.")
print(
    f"✅ Maximum allowed tokens: "
    f"{MAX_SAFE_CHUNK_TOKENS}"
)
print(
    f"✅ Synthetic chunks created: "
    f"{len(synthetic_chunks)}"
)
print(
    f"✅ Synthetic maximum tokens: "
    f"{max(synthetic_token_counts)}"
)
print(
    "✅ Synthetic oversized chunks: 0"
)
print(
    "✅ Synthetic exact duplicates: 0"
)
print(
    "\n⏭️ No real Qdrant vectors were changed yet."
)

ℹ️ Tokenizer: BertTokenizer
ℹ️ Special-token overhead: 2
ℹ️ Raw payload budget: 238
✅ Safe web chunker installed.
✅ Maximum allowed tokens: 240
✅ Synthetic chunks created: 8
✅ Synthetic maximum tokens: 240
✅ Synthetic oversized chunks: 0
✅ Synthetic exact duplicates: 0

⏭️ No real Qdrant vectors were changed yet.


In [61]:
# VAULTIFY COMPACT - Cell 18:
# Generic PDF ingestion dry-run with the safe chunker

from collections import Counter
from pathlib import Path

import hashlib
import json
import statistics
import time

import filetype
import torch

from google.colab import files
from werkzeug.utils import secure_filename

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)


# ---------------------------------------------------------------------
# 1. Verify required runtime components
# ---------------------------------------------------------------------

required_runtime_names = [
    "safe_smart_chunk_markdown",
    "count_safe_tokens",
    "normalize_chunk_for_dedup",
    "MAX_SAFE_CHUNK_TOKENS",
    "embedding_model",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

if missing_runtime_names:
    raise RuntimeError(
        "Required components are missing: "
        + ", ".join(missing_runtime_names)
        + ". Run Cell 17 before Cell 18."
    )


# ---------------------------------------------------------------------
# 2. Upload one PDF
# ---------------------------------------------------------------------

print("📄 Select one PDF for the generic ingestion dry-run.")

uploaded_files = files.upload()

if len(uploaded_files) != 1:
    raise RuntimeError(
        "Upload exactly one PDF for this dry-run."
    )

original_filename, pdf_bytes = next(
    iter(uploaded_files.items())
)

safe_filename = secure_filename(
    original_filename
)

if (
    not safe_filename
    or not safe_filename.lower().endswith(".pdf")
):
    raise RuntimeError(
        "The selected file must have a .pdf extension."
    )

if not pdf_bytes:
    raise RuntimeError(
        "The selected PDF is empty."
    )

MAX_DRY_RUN_FILE_SIZE = (
    25 * 1024 * 1024
)

if len(pdf_bytes) > MAX_DRY_RUN_FILE_SIZE:
    raise RuntimeError(
        "The PDF exceeds the current 25 MB application limit."
    )

if not pdf_bytes.startswith(
    b"%PDF-"
):
    raise RuntimeError(
        "The selected file does not contain a valid PDF signature."
    )

detected_type = filetype.guess(
    pdf_bytes[:8192]
)

if (
    detected_type is not None
    and detected_type.mime
    != "application/pdf"
):
    raise RuntimeError(
        "The selected file MIME signature is not PDF."
    )


# ---------------------------------------------------------------------
# 3. Store the PDF temporarily
# ---------------------------------------------------------------------

DRY_RUN_ROOT = Path(
    "/content/vaultify_dry_run"
)

DRY_RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DRY_RUN_PDF_PATH = (
    DRY_RUN_ROOT
    / safe_filename
)

DRY_RUN_PDF_PATH.write_bytes(
    pdf_bytes
)

DRY_RUN_DOCUMENT_HASH = hashlib.sha256(
    pdf_bytes
).hexdigest()

DRY_RUN_ORIGINAL_FILENAME = (
    original_filename
)

print("\n✅ PDF validation passed.")
print(
    f"📄 Filename: "
    f"{DRY_RUN_ORIGINAL_FILENAME}"
)
print(
    f"💾 Size: "
    f"{len(pdf_bytes) / (1024 * 1024):.2f} MB"
)
print(
    f"🔑 SHA-256: "
    f"{DRY_RUN_DOCUMENT_HASH}"
)


# ---------------------------------------------------------------------
# 4. Parse with Docling
# ---------------------------------------------------------------------

print("\n⏳ Stage 1/3 — Parsing PDF with Docling...")

pipeline_options = PdfPipelineOptions()

if torch.cuda.is_available():
    pipeline_options.accelerator_options = (
        AcceleratorOptions(
            num_threads=4,
            device=(
                AcceleratorDevice.CUDA
            ),
        )
    )

document_converter = DocumentConverter(
    format_options={
        InputFormat.PDF:
        PdfFormatOption(
            pipeline_options=(
                pipeline_options
            )
        )
    }
)

parse_started_at = time.perf_counter()

conversion_result = (
    document_converter.convert(
        str(DRY_RUN_PDF_PATH)
    )
)

DRY_RUN_MARKDOWN = (
    conversion_result.document
    .export_to_markdown()
)

parse_duration_seconds = (
    time.perf_counter()
    - parse_started_at
)

if not DRY_RUN_MARKDOWN.strip():
    raise RuntimeError(
        "Docling produced no Markdown content."
    )

markdown_output_path = (
    DRY_RUN_ROOT
    / (
        DRY_RUN_PDF_PATH.stem
        + ".md"
    )
)

markdown_output_path.write_text(
    DRY_RUN_MARKDOWN,
    encoding="utf-8",
)

print(
    f"✅ Docling parsing completed "
    f"in {parse_duration_seconds:.1f} seconds."
)
print(
    f"📝 Markdown characters: "
    f"{len(DRY_RUN_MARKDOWN):,}"
)


# ---------------------------------------------------------------------
# 5. Apply the safe generic chunker
# ---------------------------------------------------------------------

print(
    "\n⏳ Stage 2/3 — "
    "Creating token-safe chunks..."
)

chunking_started_at = (
    time.perf_counter()
)

DRY_RUN_CHUNKS = (
    safe_smart_chunk_markdown(
        embedding_model,
        DRY_RUN_MARKDOWN,
    )
)

chunking_duration_seconds = (
    time.perf_counter()
    - chunking_started_at
)

if not DRY_RUN_CHUNKS:
    raise RuntimeError(
        "The safe chunker produced no chunks."
    )

print(
    f"✅ Chunking completed "
    f"in {chunking_duration_seconds:.1f} seconds."
)


# ---------------------------------------------------------------------
# 6. Build the quality report
# ---------------------------------------------------------------------

print(
    "\n⏳ Stage 3/3 — "
    "Auditing chunk quality..."
)


def dry_run_percentile(
    values,
    percentile,
):
    if not values:
        return 0

    ordered_values = sorted(
        values
    )

    index = round(
        (
            len(ordered_values)
            - 1
        )
        * percentile
    )

    return ordered_values[
        index
    ]


chunk_token_counts = [
    count_safe_tokens(
        chunk["text"]
    )
    for chunk in DRY_RUN_CHUNKS
]

normalized_chunk_texts = [
    normalize_chunk_for_dedup(
        chunk["text"]
    )
    for chunk in DRY_RUN_CHUNKS
]

exact_duplicate_count = (
    len(normalized_chunk_texts)
    - len(
        set(
            normalized_chunk_texts
        )
    )
)

empty_chunk_count = sum(
    not text
    for text in normalized_chunk_texts
)

oversized_240_count = sum(
    token_count
    > MAX_SAFE_CHUNK_TOKENS
    for token_count
    in chunk_token_counts
)

oversized_256_count = sum(
    token_count > 256
    for token_count
    in chunk_token_counts
)

tiny_chunk_count = sum(
    token_count < 20
    for token_count
    in chunk_token_counts
)

chunk_type_counts = Counter(
    chunk.get(
        "chunk_type",
        "unknown",
    )
    for chunk in DRY_RUN_CHUNKS
)

section_counts = Counter(
    chunk.get(
        "section",
        "Unknown section",
    )
    for chunk in DRY_RUN_CHUNKS
)

DRY_RUN_REPORT = {
    "filename": (
        DRY_RUN_ORIGINAL_FILENAME
    ),
    "document_hash": (
        DRY_RUN_DOCUMENT_HASH
    ),
    "file_size_bytes": len(
        pdf_bytes
    ),
    "parse_seconds": round(
        parse_duration_seconds,
        2,
    ),
    "chunking_seconds": round(
        chunking_duration_seconds,
        2,
    ),
    "markdown_characters": len(
        DRY_RUN_MARKDOWN
    ),
    "total_chunks": len(
        DRY_RUN_CHUNKS
    ),
    "chunk_types": dict(
        chunk_type_counts
    ),
    "token_min": min(
        chunk_token_counts
    ),
    "token_median": (
        statistics.median(
            chunk_token_counts
        )
    ),
    "token_p95": (
        dry_run_percentile(
            chunk_token_counts,
            0.95,
        )
    ),
    "token_max": max(
        chunk_token_counts
    ),
    "chunks_above_240": (
        oversized_240_count
    ),
    "chunks_above_256": (
        oversized_256_count
    ),
    "exact_duplicates": (
        exact_duplicate_count
    ),
    "empty_chunks": (
        empty_chunk_count
    ),
    "tiny_chunks_below_20": (
        tiny_chunk_count
    ),
    "unique_sections": len(
        section_counts
    ),
}

DRY_RUN_READY_FOR_EVALUATION = (
    oversized_240_count == 0
    and oversized_256_count == 0
    and exact_duplicate_count == 0
    and empty_chunk_count == 0
)


# ---------------------------------------------------------------------
# 7. Save temporary audit artifacts
# ---------------------------------------------------------------------

chunks_output_path = (
    DRY_RUN_ROOT
    / (
        DRY_RUN_PDF_PATH.stem
        + "_safe_chunks.json"
    )
)

report_output_path = (
    DRY_RUN_ROOT
    / (
        DRY_RUN_PDF_PATH.stem
        + "_dry_run_report.json"
    )
)

chunks_output_path.write_text(
    json.dumps(
        DRY_RUN_CHUNKS,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

report_output_path.write_text(
    json.dumps(
        DRY_RUN_REPORT,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# 8. Print the final report
# ---------------------------------------------------------------------

print("\n" + "=" * 88)
print("📊 GENERIC INGESTION DRY-RUN REPORT")

print(
    f"📄 File: "
    f"{DRY_RUN_REPORT['filename']}"
)

print(
    f"📦 Total chunks: "
    f"{DRY_RUN_REPORT['total_chunks']}"
)

print(
    f"🧩 Chunk types: "
    f"{DRY_RUN_REPORT['chunk_types']}"
)

print(
    "📐 Token distribution:"
    f" min={DRY_RUN_REPORT['token_min']},"
    f" median="
    f"{DRY_RUN_REPORT['token_median']:.1f},"
    f" p95="
    f"{DRY_RUN_REPORT['token_p95']},"
    f" max="
    f"{DRY_RUN_REPORT['token_max']}"
)

print(
    f"⚠️ Chunks above 240 tokens: "
    f"{DRY_RUN_REPORT['chunks_above_240']}"
)

print(
    f"⚠️ Chunks above 256 tokens: "
    f"{DRY_RUN_REPORT['chunks_above_256']}"
)

print(
    f"♻️ Exact duplicates: "
    f"{DRY_RUN_REPORT['exact_duplicates']}"
)

print(
    f"🕳️ Empty chunks: "
    f"{DRY_RUN_REPORT['empty_chunks']}"
)

print(
    f"🔹 Tiny chunks below 20 tokens: "
    f"{DRY_RUN_REPORT['tiny_chunks_below_20']}"
)

print(
    f"📚 Unique sections: "
    f"{DRY_RUN_REPORT['unique_sections']}"
)

print("\nMost frequent sections:")

for section, count in (
    section_counts.most_common(10)
):
    print(
        f"   {count:>4} chunks — "
        f"{section}"
    )


print("\nLongest generated chunks:")

longest_chunk_indexes = sorted(
    range(
        len(DRY_RUN_CHUNKS)
    ),
    key=lambda index: (
        chunk_token_counts[index]
    ),
    reverse=True,
)[:5]

for chunk_index in (
    longest_chunk_indexes
):
    chunk = DRY_RUN_CHUNKS[
        chunk_index
    ]

    preview = " ".join(
        chunk["text"].split()
    )[:180]

    print(
        f"   Chunk {chunk_index}"
        f" — {chunk_token_counts[chunk_index]} tokens"
        f" — {chunk['chunk_type']}"
        f" — {chunk['section']}"
    )

    print(
        f"      {preview}..."
    )


print("\n" + "=" * 88)

if DRY_RUN_READY_FOR_EVALUATION:
    print(
        "✅ Dry-run passed all hard safety checks."
    )

    print(
        "➡️ The chunks are ready for "
        "retrieval-quality evaluation in Cell 19."
    )

else:
    print(
        "❌ Dry-run failed one or more hard checks."
    )

    print(
        "⛔ Do not generate embeddings "
        "or write these chunks to Qdrant."
    )


print(
    f"\n📁 Temporary artifacts: "
    f"{DRY_RUN_ROOT}"
)

print(
    "⏭️ No embeddings were generated."
)

print(
    "⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

📄 Select one PDF for the generic ingestion dry-run.


[INFO] 2026-07-23 11:57:37,043 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-23 11:57:37,059 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-07-23 11:57:37,060 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-07-23 11:57:37,184 [RapidOCR] base.py:22: Using engine_name: onnxruntime


Saving nist.ai.100-1.pdf to nist.ai.100-1 (1).pdf

✅ PDF validation passed.
📄 Filename: nist.ai.100-1 (1).pdf
💾 Size: 1.86 MB
🔑 SHA-256: 7576edb531d9848825814ee88e28b1795d3a84b435b4b797d3670eafdc4a89f1

⏳ Stage 1/3 — Parsing PDF with Docling...


[INFO] 2026-07-23 11:57:37,189 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-23 11:57:37,190 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-23 11:57:37,242 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-23 11:57:37,274 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-07-23 11:57:37,275 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ Docling parsing completed in 72.6 seconds.
📝 Markdown characters: 149,427

⏳ Stage 2/3 — Creating token-safe chunks...
✅ Chunking completed in 0.4 seconds.

⏳ Stage 3/3 — Auditing chunk quality...

📊 GENERIC INGESTION DRY-RUN REPORT
📄 File: nist.ai.100-1 (1).pdf
📦 Total chunks: 153
🧩 Chunk types: {'text': 107, 'table': 46}
📐 Token distribution: min=4, median=240.0, p95=240, max=240
⚠️ Chunks above 240 tokens: 0
⚠️ Chunks above 256 tokens: 0
♻️ Exact duplicates: 0
🕳️ Empty chunks: 0
🔹 Tiny chunks below 20 tokens: 9
📚 Unique sections: 37

Most frequent sections:
     15 chunks — 5.1 Govern
     15 chunks — 5.2 Map
     15 chunks — 5.3 Measure
     10 chunks — 2. Audience
     10 chunks — 5.4 Manage
      9 chunks — Executive Summary
      6 chunks — Table of Contents
      6 chunks — Appendix A: Descriptions of AI Actor Tasks from Figures 2 and 3
      5 chunks — 3. AI Risks and Trustworthiness
      5 chunks — Appendix B: How AI Risks Differ from Traditional Software Risks

Longest ge

In [62]:
# VAULTIFY COMPACT - Cell 18A:
# Preserve each document dry-run result for later comparison

if "DRY_RUN_RESULTS" not in globals():
    DRY_RUN_RESULTS = {}

DRY_RUN_RESULTS[
    DRY_RUN_DOCUMENT_HASH
] = {
    "filename": DRY_RUN_ORIGINAL_FILENAME,
    "document_hash": DRY_RUN_DOCUMENT_HASH,
    "report": dict(DRY_RUN_REPORT),
    "chunks": list(DRY_RUN_CHUNKS),
    "markdown": DRY_RUN_MARKDOWN,
}

print(
    "✅ Dry-run result preserved:"
    f" {DRY_RUN_ORIGINAL_FILENAME}"
)

print(
    f"📦 Stored documents: "
    f"{len(DRY_RUN_RESULTS)}"
)

for stored_result in (
    DRY_RUN_RESULTS.values()
):
    report = stored_result["report"]

    print(
        f"   {stored_result['filename']}"
        f" — {report['total_chunks']} chunks"
        f" — max {report['token_max']} tokens"
        f" — duplicates {report['exact_duplicates']}"
    )

✅ Dry-run result preserved: nist.ai.100-1 (1).pdf
📦 Stored documents: 3
   _10-K-2025-As-Filed.pdf — 565 chunks — max 240 tokens — duplicates 0
   TSLA-Q4-2025-Update.pdf — 258 chunks — max 240 tokens — duplicates 0
   nist.ai.100-1 (1).pdf — 153 chunks — max 240 tokens — duplicates 0


In [63]:
# VAULTIFY COMPACT - Cell 19:
# Generic in-memory retrieval benchmark for preserved dry-run documents

from collections import defaultdict

import numpy as np


# ---------------------------------------------------------------------
# 1. Validate preserved dry-run results
# ---------------------------------------------------------------------

if (
    "DRY_RUN_RESULTS" not in globals()
    or len(DRY_RUN_RESULTS) < 3
):
    raise RuntimeError(
        "Three preserved dry-run documents are required. "
        "Run Cell 18 and Cell 18A for Apple, Tesla, and NIST first."
    )


def find_preserved_document(filename_fragment):
    """
    Find one preserved dry-run document by a case-insensitive
    filename fragment.
    """
    filename_fragment = (
        filename_fragment.lower()
    )

    matches = [
        result
        for result in DRY_RUN_RESULTS.values()
        if filename_fragment
        in result["filename"].lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one document matching "
            f"'{filename_fragment}', found {len(matches)}."
        )

    return matches[0]


apple_result = find_preserved_document(
    "10-k"
)

tesla_result = find_preserved_document(
    "tsla"
)

nist_result = find_preserved_document(
    "nist"
)


DOCUMENT_BENCHMARKS = {
    "Apple 10-K": {
        "result": apple_result,
        "positive_queries": [
            {
                "question": (
                    "What were Apple's total net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["416,161"],
                    [
                        "total net sales",
                        "net sales",
                    ],
                ],
            },
        ],
        "negative_queries": [
            (
                "What is the capital city "
                "of Australia?"
            ),
        ],
    },

    "Tesla Q4 2025": {
        "result": tesla_result,
        "positive_queries": [
            {
                "question": (
                    "What was Tesla's total revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["24,901"],
                    [
                        "total revenue",
                        "total revenues",
                        "revenues",
                    ],
                ],
            },
        ],
        "negative_queries": [
            (
                "Who wrote the novel "
                "Pride and Prejudice?"
            ),
        ],
    },

    "NIST AI RMF": {
        "result": nist_result,
        "positive_queries": [
            {
                "question": (
                    "What are the four core functions "
                    "of the NIST AI Risk Management Framework?"
                ),
                "expected_groups": [
                    ["govern"],
                    ["map"],
                    ["measure"],
                    ["manage"],
                ],
            },
            {
                "question": (
                    "What characteristics are associated "
                    "with trustworthy AI systems?"
                ),
                "expected_groups": [
                    [
                        "valid and reliable",
                        "validity and reliability",
                    ],
                    ["safe"],
                    [
                        "secure and resilient",
                        "security and resilience",
                    ],
                    [
                        "accountable and transparent",
                        "accountability and transparency",
                    ],
                    [
                        "explainable and interpretable",
                        "explainability and interpretability",
                    ],
                    ["privacy"],
                    ["fair"],
                ],
            },
        ],
        "negative_queries": [
            (
                "How many goals did the winner "
                "of the 2018 World Cup final score?"
            ),
        ],
    },
}


# ---------------------------------------------------------------------
# 2. Create and cache document embeddings
# ---------------------------------------------------------------------

print("⏳ Creating in-memory embeddings...")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    result = benchmark["result"]

    chunks = result["chunks"]

    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    if "embeddings" not in result:
        print(
            f"   Embedding {document_label}: "
            f"{len(chunk_texts)} chunks"
        )

        result["embeddings"] = (
            embedding_model.encode(
                chunk_texts,
                batch_size=64,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
        )

    embeddings = result[
        "embeddings"
    ]

    if len(embeddings) != len(chunks):
        raise RuntimeError(
            f"Embedding count mismatch "
            f"for {document_label}."
        )

print("✅ In-memory embeddings are ready.")


# ---------------------------------------------------------------------
# 3. Generic semantic retrieval
# ---------------------------------------------------------------------

def retrieve_from_preserved_document(
    result,
    question,
    top_k=6,
):
    """
    Retrieve the most similar dry-run chunks using cosine similarity.
    Embeddings are normalized, so dot product equals cosine similarity.
    """
    query_embedding = (
        embedding_model.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )[0]
    )

    document_embeddings = (
        result["embeddings"]
    )

    similarity_scores = (
        document_embeddings
        @ query_embedding
    )

    top_indexes = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    retrieved = []

    for rank, chunk_index in enumerate(
        top_indexes,
        start=1,
    ):
        chunk = result["chunks"][
            int(chunk_index)
        ]

        retrieved.append(
            {
                "rank": rank,
                "chunk_index": int(
                    chunk_index
                ),
                "score": float(
                    similarity_scores[
                        chunk_index
                    ]
                ),
                "text": chunk["text"],
                "section": chunk.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": chunk.get(
                    "chunk_type",
                    "unknown",
                ),
            }
        )

    return retrieved


def expected_groups_found(
    retrieved_chunks,
    expected_groups,
):
    """
    Every expected group must have at least one matching alternative
    somewhere in the retrieved context.
    """
    combined_context = " ".join(
        item["text"]
        for item in retrieved_chunks
    ).lower()

    group_results = []

    for alternatives in expected_groups:
        matched_alternative = next(
            (
                alternative
                for alternative
                in alternatives
                if alternative.lower()
                in combined_context
            ),
            None,
        )

        group_results.append(
            {
                "alternatives": alternatives,
                "matched": (
                    matched_alternative
                ),
            }
        )

    passed = all(
        group["matched"]
        is not None
        for group in group_results
    )

    return passed, group_results


# ---------------------------------------------------------------------
# 4. Print chunk-density diagnostics
# ---------------------------------------------------------------------

print("\n" + "=" * 92)
print("📐 CHUNK DENSITY DIAGNOSTICS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    result = benchmark["result"]
    chunks = result["chunks"]

    token_counts = [
        count_safe_tokens(
            chunk["text"]
        )
        for chunk in chunks
    ]

    saturated_count = sum(
        token_count
        == MAX_SAFE_CHUNK_TOKENS
        for token_count
        in token_counts
    )

    tiny_count = sum(
        token_count < 20
        for token_count
        in token_counts
    )

    saturated_ratio = (
        saturated_count
        / len(token_counts)
        * 100
    )

    tiny_ratio = (
        tiny_count
        / len(token_counts)
        * 100
    )

    print(
        f"\n{document_label}"
    )

    print(
        f"   Total chunks: "
        f"{len(chunks)}"
    )

    print(
        f"   Exactly 240 tokens: "
        f"{saturated_count} "
        f"({saturated_ratio:.1f}%)"
    )

    print(
        f"   Below 20 tokens: "
        f"{tiny_count} "
        f"({tiny_ratio:.1f}%)"
    )


# ---------------------------------------------------------------------
# 5. Positive retrieval benchmarks
# ---------------------------------------------------------------------

positive_results = []

print("\n" + "=" * 92)
print("🔎 POSITIVE RETRIEVAL BENCHMARKS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for test_case in benchmark[
        "positive_queries"
    ]:
        question = test_case[
            "question"
        ]

        retrieved = (
            retrieve_from_preserved_document(
                result=benchmark["result"],
                question=question,
                top_k=6,
            )
        )

        passed, group_results = (
            expected_groups_found(
                retrieved,
                test_case[
                    "expected_groups"
                ],
            )
        )

        positive_results.append(
            {
                "document": (
                    document_label
                ),
                "question": question,
                "passed": passed,
                "top_score": (
                    retrieved[0]["score"]
                ),
                "retrieved": retrieved,
                "group_results": (
                    group_results
                ),
            }
        )

        status = (
            "✅ PASS"
            if passed
            else "❌ FAIL"
        )

        print(
            f"\n{status} — "
            f"{document_label}"
        )

        print(
            f"Question: {question}"
        )

        print(
            f"Top score: "
            f"{retrieved[0]['score']:.4f}"
        )

        print(
            "Expected evidence:"
        )

        for group in group_results:
            print(
                f"   "
                f"{'✅' if group['matched'] else '❌'} "
                f"{group['alternatives']}"
                f" → "
                f"{group['matched']}"
            )

        print(
            "Top retrieved chunks:"
        )

        for item in retrieved[:3]:
            preview = " ".join(
                item["text"].split()
            )[:180]

            print(
                f"   #{item['rank']} "
                f"score={item['score']:.4f} "
                f"type={item['chunk_type']} "
                f"section={item['section']}"
            )

            print(
                f"      {preview}..."
            )


# ---------------------------------------------------------------------
# 6. Negative-query score diagnostics
# ---------------------------------------------------------------------

negative_scores = defaultdict(
    list
)

print("\n" + "=" * 92)
print("🚫 NEGATIVE QUERY SCORE DIAGNOSTICS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for question in benchmark[
        "negative_queries"
    ]:
        retrieved = (
            retrieve_from_preserved_document(
                result=benchmark["result"],
                question=question,
                top_k=3,
            )
        )

        top_score = retrieved[0][
            "score"
        ]

        negative_scores[
            document_label
        ].append(top_score)

        print(
            f"\n{document_label}"
        )

        print(
            f"Question: {question}"
        )

        print(
            f"Highest irrelevant similarity: "
            f"{top_score:.4f}"
        )

        print(
            f"Top unrelated section: "
            f"{retrieved[0]['section']}"
        )


# ---------------------------------------------------------------------
# 7. Final benchmark summary
# ---------------------------------------------------------------------

passed_positive_tests = sum(
    result["passed"]
    for result in positive_results
)

total_positive_tests = len(
    positive_results
)

CELL_19_RETRIEVAL_PASSED = (
    passed_positive_tests
    == total_positive_tests
)

CELL_19_RESULTS = {
    "positive_results": (
        positive_results
    ),
    "negative_scores": dict(
        negative_scores
    ),
    "passed": (
        CELL_19_RETRIEVAL_PASSED
    ),
}

print("\n" + "=" * 92)
print("📊 CELL 19 SUMMARY")

print(
    f"Positive retrieval tests passed: "
    f"{passed_positive_tests}"
    f"/{total_positive_tests}"
)

if CELL_19_RETRIEVAL_PASSED:
    print(
        "✅ Every expected answer signal "
        "was found inside the top-6 retrieved chunks."
    )

    print(
        "➡️ The new chunk sets are ready "
        "for old-vs-new retrieval comparison."
    )

else:
    print(
        "❌ At least one expected answer signal "
        "was not found in the top-6 results."
    )

    print(
        "⛔ Do not replace existing "
        "Qdrant vectors yet."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

⏳ Creating in-memory embeddings...
   Embedding NIST AI RMF: 153 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✅ In-memory embeddings are ready.

📐 CHUNK DENSITY DIAGNOSTICS

Apple 10-K
   Total chunks: 565
   Exactly 240 tokens: 147 (26.0%)
   Below 20 tokens: 55 (9.7%)

Tesla Q4 2025
   Total chunks: 258
   Exactly 240 tokens: 30 (11.6%)
   Below 20 tokens: 18 (7.0%)

NIST AI RMF
   Total chunks: 153
   Exactly 240 tokens: 93 (60.8%)
   Below 20 tokens: 9 (5.9%)

🔎 POSITIVE RETRIEVAL BENCHMARKS

❌ FAIL — Apple 10-K
Question: What were Apple's total net sales in fiscal year 2025?
Top score: 0.6956
Expected evidence:
   ❌ ['416,161'] → None
   ✅ ['total net sales', 'net sales'] → net sales
Top retrieved chunks:
   #1 score=0.6956 type=text section=Mac
      ## Mac Mac net sales increased during 2025 compared to 2024 primarily due to higher net sales of laptops and desktops....
   #2 score=0.6613 type=text section=iPhone
      ## iPhone iPhone net sales increased during 2025 compared to 2024 due to higher net sales of Pro models....
   #3 score=0.6350 type=text section=Products and Services Perf

In [64]:
# VAULTIFY COMPACT - Cell 19A:
# Diagnose where known evidence ranks inside a dry-run document

import re

import numpy as np


def normalize_evidence_text(text):
    """
    Normalize text so values such as 416,161 and 416161 can match.
    """
    normalized = str(text or "").lower()

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    compact = re.sub(
        r"[^a-z0-9]+",
        "",
        normalized,
    )

    return normalized, compact


def diagnose_evidence_rank(
    result,
    question,
    evidence_terms,
    label,
    preview_neighbors=True,
):
    """
    Find every chunk containing expected evidence and report
    its semantic retrieval rank.
    """
    chunks = result["chunks"]
    embeddings = result["embeddings"]

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]

    similarity_scores = (
        embeddings
        @ query_embedding
    )

    ranked_indexes = np.argsort(
        similarity_scores
    )[::-1]

    rank_by_index = {
        int(chunk_index): rank
        for rank, chunk_index
        in enumerate(
            ranked_indexes,
            start=1,
        )
    }

    normalized_terms = [
        normalize_evidence_text(term)
        for term in evidence_terms
    ]

    matching_indexes = []

    for chunk_index, chunk in enumerate(
        chunks
    ):
        normalized_text, compact_text = (
            normalize_evidence_text(
                chunk["text"]
            )
        )

        has_match = any(
            (
                normalized_term
                in normalized_text
            )
            or (
                compact_term
                and compact_term
                in compact_text
            )
            for (
                normalized_term,
                compact_term,
            )
            in normalized_terms
        )

        if has_match:
            matching_indexes.append(
                chunk_index
            )

    print("\n" + "=" * 92)
    print(f"🔬 EVIDENCE RANK DIAGNOSTIC — {label}")
    print(f"Question: {question}")
    print(f"Evidence terms: {evidence_terms}")
    print(
        f"Matching chunks found: "
        f"{len(matching_indexes)}"
    )

    if not matching_indexes:
        print(
            "❌ Expected evidence does not appear "
            "inside any generated chunk."
        )

        print(
            "This suggests a Docling extraction "
            "or chunk-construction issue."
        )

        return []

    diagnostic_results = []

    for chunk_index in matching_indexes:
        chunk = chunks[chunk_index]

        rank = rank_by_index[
            chunk_index
        ]

        score = float(
            similarity_scores[
                chunk_index
            ]
        )

        diagnostic_results.append(
            {
                "chunk_index": chunk_index,
                "rank": rank,
                "score": score,
                "section": chunk.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": chunk.get(
                    "chunk_type",
                    "unknown",
                ),
            }
        )

    diagnostic_results.sort(
        key=lambda item: item["rank"]
    )

    for result_item in (
        diagnostic_results[:10]
    ):
        chunk_index = result_item[
            "chunk_index"
        ]

        chunk = chunks[
            chunk_index
        ]

        preview = " ".join(
            chunk["text"].split()
        )[:500]

        print(
            f"\nRank #{result_item['rank']}"
            f" — score={result_item['score']:.4f}"
            f" — chunk={chunk_index}"
            f" — type={result_item['chunk_type']}"
            f" — section={result_item['section']}"
        )

        print(
            f"   {preview}..."
        )

        if preview_neighbors:
            for neighbor_index in [
                chunk_index - 1,
                chunk_index + 1,
            ]:
                if (
                    neighbor_index < 0
                    or neighbor_index
                    >= len(chunks)
                ):
                    continue

                neighbor = chunks[
                    neighbor_index
                ]

                neighbor_preview = " ".join(
                    neighbor["text"].split()
                )[:260]

                print(
                    f"   ↳ Neighbor {neighbor_index}"
                    f" — {neighbor.get('chunk_type')}"
                    f" — {neighbor.get('section')}"
                )

                print(
                    f"      {neighbor_preview}..."
                )

    best_rank = diagnostic_results[
        0
    ]["rank"]

    print("\nInterpretation:")

    if best_rank <= 6:
        print(
            "✅ Evidence is already within top-6. "
            "The original benchmark matching logic "
            "may need inspection."
        )

    elif best_rank <= 20:
        print(
            "⚠️ Evidence exists and is reasonably close, "
            "but dense retrieval needs reranking."
        )

    else:
        print(
            "⚠️ Evidence exists but dense semantic retrieval "
            "ranks it too low. Hybrid lexical + vector "
            "retrieval is recommended."
        )

    return diagnostic_results


APPLE_EVIDENCE_DIAGNOSTIC = (
    diagnose_evidence_rank(
        result=apple_result,
        question=(
            "What were Apple's total net sales "
            "in fiscal year 2025?"
        ),
        evidence_terms=[
            "416,161",
            "416161",
        ],
        label="Apple FY2025 total net sales",
    )
)


🔬 EVIDENCE RANK DIAGNOSTIC — Apple FY2025 total net sales
Question: What were Apple's total net sales in fiscal year 2025?
Evidence terms: ['416,161', '416161']
Matching chunks found: 6

Rank #27 — score=0.5195 — chunk=289 — type=table — section=Note 2 - Revenue
   Note 2 - Revenue | | 2025 | 2024 | 2023 | |----------------------------------------------------------------------------------------------------|-----------|-----------|-----------| | Services (1) | 109,158 | 96,169 | 85,200 | | Total net sales | $ 416,161 | $ 391,035 | $ 383,285 | | Portion of total net sales that was included in deferred revenue as of the beginning of the period | $ 8,229 | $ 7,728 | $ 8,169 |...
   ↳ Neighbor 288 — table — Note 2 - Revenue
      Note 2 - Revenue | | 2025 | 2024 | 2023 | |----------------------------------------------------------------------------------------------------|-----------|-----------|-----------| | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357...


In [65]:
# VAULTIFY COMPACT - Cell 19B:
# Generic hybrid retrieval using Dense + BM25 + Reciprocal Rank Fusion

from collections import Counter, defaultdict
import html
import math
import re

import numpy as np


if "DOCUMENT_BENCHMARKS" not in globals():
    raise RuntimeError(
        "Run Cell 19 before Cell 19B."
    )


HYBRID_STOPWORDS = {
    "a", "an", "and", "are", "as", "at",
    "be", "been", "by", "for", "from",
    "had", "has", "have", "how", "in",
    "is", "it", "its", "of", "on", "or",
    "s", "that", "the", "their", "this",
    "to", "was", "were", "what", "which",
    "who", "with",
}


QUANTITATIVE_TERMS = {
    "assets",
    "capacity",
    "cash",
    "cost",
    "costs",
    "earnings",
    "expense",
    "expenses",
    "income",
    "liabilities",
    "margin",
    "margins",
    "percent",
    "percentage",
    "revenue",
    "revenues",
    "sales",
    "total",
}


def hybrid_tokenize(text):
    """
    Normalize text into lexical tokens while preserving
    years and financial values.
    """
    normalized = html.unescape(
        str(text or "")
    ).lower()

    # Convert 416,161 into 416161.
    normalized = re.sub(
        r"(?<=\d),(?=\d)",
        "",
        normalized,
    )

    tokens = re.findall(
        r"[a-z0-9]+",
        normalized,
    )

    return [
        token
        for token in tokens
        if token not in HYBRID_STOPWORDS
    ]


def build_bm25_index(chunks):
    """
    Build a lightweight in-memory BM25 index.
    """
    document_tokens = [
        hybrid_tokenize(
            chunk["text"]
        )
        for chunk in chunks
    ]

    term_frequencies = [
        Counter(tokens)
        for tokens in document_tokens
    ]

    document_lengths = np.array(
        [
            len(tokens)
            for tokens in document_tokens
        ],
        dtype=np.float32,
    )

    average_document_length = float(
        document_lengths.mean()
    ) if len(document_lengths) else 0.0

    postings = defaultdict(list)

    for document_index, frequencies in enumerate(
        term_frequencies
    ):
        for term, frequency in frequencies.items():
            postings[term].append(
                (
                    document_index,
                    frequency,
                )
            )

    document_frequency = {
        term: len(entries)
        for term, entries in postings.items()
    }

    return {
        "document_tokens": document_tokens,
        "document_lengths": document_lengths,
        "average_document_length": (
            average_document_length
        ),
        "postings": dict(postings),
        "document_frequency": (
            document_frequency
        ),
        "document_count": len(chunks),
    }


def calculate_bm25_scores(
    index,
    question,
    k1=1.5,
    b=0.75,
):
    """
    Calculate BM25 lexical relevance scores.
    """
    query_tokens = hybrid_tokenize(
        question
    )

    query_term_counts = Counter(
        query_tokens
    )

    scores = np.zeros(
        index["document_count"],
        dtype=np.float32,
    )

    average_length = max(
        index["average_document_length"],
        1.0,
    )

    for term, query_frequency in (
        query_term_counts.items()
    ):
        postings = index[
            "postings"
        ].get(term, [])

        document_frequency = index[
            "document_frequency"
        ].get(term, 0)

        if document_frequency == 0:
            continue

        inverse_document_frequency = math.log(
            1.0
            + (
                (
                    index["document_count"]
                    - document_frequency
                    + 0.5
                )
                / (
                    document_frequency
                    + 0.5
                )
            )
        )

        for document_index, term_frequency in postings:
            document_length = index[
                "document_lengths"
            ][document_index]

            denominator = (
                term_frequency
                + k1
                * (
                    1.0
                    - b
                    + b
                    * document_length
                    / average_length
                )
            )

            score = (
                inverse_document_frequency
                * (
                    term_frequency
                    * (k1 + 1.0)
                )
                / denominator
            )

            scores[document_index] += (
                score
                * query_frequency
            )

    return scores


def create_rank_array(scores):
    """
    Convert scores into one-based ranks.
    """
    order = np.argsort(
        scores
    )[::-1]

    ranks = np.empty(
        len(scores),
        dtype=np.int32,
    )

    ranks[order] = np.arange(
        1,
        len(scores) + 1,
    )

    return ranks


def build_query_phrases(
    query_tokens,
):
    """
    Create meaningful query bigrams and trigrams.
    """
    phrases = []

    for phrase_size in (3, 2):
        for start_index in range(
            0,
            len(query_tokens)
            - phrase_size
            + 1,
        ):
            phrase = " ".join(
                query_tokens[
                    start_index:
                    start_index
                    + phrase_size
                ]
            )

            phrases.append(
                phrase
            )

    return phrases


def retrieve_hybrid(
    result,
    question,
    top_k=6,
    rrf_constant=60,
):
    """
    Combine dense and BM25 rankings using Reciprocal Rank Fusion,
    then apply generic lexical, year, phrase, and table signals.
    """
    chunks = result["chunks"]

    if "bm25_index" not in result:
        result["bm25_index"] = (
            build_bm25_index(
                chunks
            )
        )

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]

    dense_scores = (
        result["embeddings"]
        @ query_embedding
    )

    lexical_scores = calculate_bm25_scores(
        result["bm25_index"],
        question,
    )

    dense_ranks = create_rank_array(
        dense_scores
    )

    lexical_ranks = create_rank_array(
        lexical_scores
    )

    query_tokens = hybrid_tokenize(
        question
    )

    unique_query_tokens = set(
        query_tokens
    )

    query_phrases = build_query_phrases(
        query_tokens
    )

    query_years = {
        token
        for token in query_tokens
        if re.fullmatch(
            r"(?:19|20)\d{2}",
            token,
        )
    }

    is_quantitative_query = bool(
        unique_query_tokens
        & QUANTITATIVE_TERMS
    )

    hybrid_scores = np.zeros(
        len(chunks),
        dtype=np.float32,
    )

    feature_details = []

    for chunk_index, chunk in enumerate(
        chunks
    ):
        chunk_tokens = set(
            result["bm25_index"][
                "document_tokens"
            ][chunk_index]
        )

        chunk_normalized = " ".join(
            result["bm25_index"][
                "document_tokens"
            ][chunk_index]
        )

        reciprocal_rank_score = (
            1.0
            / (
                rrf_constant
                + dense_ranks[
                    chunk_index
                ]
            )
            + 1.0
            / (
                rrf_constant
                + lexical_ranks[
                    chunk_index
                ]
            )
        )

        if unique_query_tokens:
            lexical_coverage = (
                len(
                    unique_query_tokens
                    & chunk_tokens
                )
                / len(
                    unique_query_tokens
                )
            )
        else:
            lexical_coverage = 0.0

        phrase_hits = sum(
            phrase in chunk_normalized
            for phrase in query_phrases
        )

        phrase_bonus = min(
            phrase_hits,
            2,
        ) * 0.004

        year_bonus = 0.0

        if (
            query_years
            and query_years.issubset(
                chunk_tokens
            )
        ):
            year_bonus = 0.004

        table_bonus = 0.0

        if (
            is_quantitative_query
            and chunk.get(
                "chunk_type"
            ) == "table"
        ):
            table_bonus = 0.002

        coverage_bonus = (
            lexical_coverage
            * 0.010
        )

        final_score = (
            reciprocal_rank_score
            + coverage_bonus
            + phrase_bonus
            + year_bonus
            + table_bonus
        )

        hybrid_scores[
            chunk_index
        ] = final_score

        feature_details.append(
            {
                "dense_rank": int(
                    dense_ranks[
                        chunk_index
                    ]
                ),
                "lexical_rank": int(
                    lexical_ranks[
                        chunk_index
                    ]
                ),
                "dense_score": float(
                    dense_scores[
                        chunk_index
                    ]
                ),
                "lexical_score": float(
                    lexical_scores[
                        chunk_index
                    ]
                ),
                "coverage": float(
                    lexical_coverage
                ),
                "phrase_hits": int(
                    phrase_hits
                ),
                "year_bonus": (
                    year_bonus
                ),
                "table_bonus": (
                    table_bonus
                ),
            }
        )

    top_indexes = np.argsort(
        hybrid_scores
    )[::-1][:top_k]

    results = []

    for rank, chunk_index in enumerate(
        top_indexes,
        start=1,
    ):
        chunk_index = int(
            chunk_index
        )

        chunk = chunks[
            chunk_index
        ]

        features = feature_details[
            chunk_index
        ]

        results.append(
            {
                "rank": rank,
                "chunk_index": (
                    chunk_index
                ),
                "hybrid_score": float(
                    hybrid_scores[
                        chunk_index
                    ]
                ),
                "dense_rank": features[
                    "dense_rank"
                ],
                "lexical_rank": features[
                    "lexical_rank"
                ],
                "dense_score": features[
                    "dense_score"
                ],
                "lexical_score": features[
                    "lexical_score"
                ],
                "coverage": features[
                    "coverage"
                ],
                "text": chunk["text"],
                "section": chunk.get(
                    "section",
                    "Unknown section",
                ),
                "chunk_type": chunk.get(
                    "chunk_type",
                    "unknown",
                ),
            }
        )

    return results


def hybrid_expected_groups_found(
    retrieved_chunks,
    expected_groups,
):
    combined_context = " ".join(
        item["text"]
        for item in retrieved_chunks
    ).lower()

    group_results = []

    for alternatives in expected_groups:
        matched = next(
            (
                alternative
                for alternative
                in alternatives
                if alternative.lower()
                in combined_context
            ),
            None,
        )

        group_results.append(
            {
                "alternatives": (
                    alternatives
                ),
                "matched": matched,
            }
        )

    passed = all(
        result["matched"]
        is not None
        for result in group_results
    )

    return passed, group_results


# ---------------------------------------------------------------------
# Build reusable indexes
# ---------------------------------------------------------------------

print("⏳ Building generic BM25 indexes...")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    result = benchmark["result"]

    result["bm25_index"] = (
        build_bm25_index(
            result["chunks"]
        )
    )

    print(
        f"   ✅ {document_label}: "
        f"{len(result['chunks'])} chunks"
    )


# ---------------------------------------------------------------------
# Run positive hybrid benchmarks
# ---------------------------------------------------------------------

HYBRID_POSITIVE_RESULTS = []

print("\n" + "=" * 92)
print("🔀 HYBRID RETRIEVAL BENCHMARKS")

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for test_case in benchmark[
        "positive_queries"
    ]:
        retrieved = retrieve_hybrid(
            result=benchmark["result"],
            question=test_case[
                "question"
            ],
            top_k=6,
        )

        passed, group_results = (
            hybrid_expected_groups_found(
                retrieved,
                test_case[
                    "expected_groups"
                ],
            )
        )

        HYBRID_POSITIVE_RESULTS.append(
            {
                "document": (
                    document_label
                ),
                "question": test_case[
                    "question"
                ],
                "passed": passed,
                "retrieved": retrieved,
                "groups": group_results,
            }
        )

        status = (
            "✅ PASS"
            if passed
            else "❌ FAIL"
        )

        print(
            f"\n{status} — "
            f"{document_label}"
        )

        print(
            f"Question: "
            f"{test_case['question']}"
        )

        for group in group_results:
            print(
                f"   "
                f"{'✅' if group['matched'] else '❌'} "
                f"{group['alternatives']}"
                f" → {group['matched']}"
            )

        print(
            "Top hybrid chunks:"
        )

        for item in retrieved[:3]:
            preview = " ".join(
                item["text"].split()
            )[:200]

            print(
                f"   #{item['rank']} "
                f"hybrid={item['hybrid_score']:.4f} "
                f"dense_rank={item['dense_rank']} "
                f"bm25_rank={item['lexical_rank']} "
                f"type={item['chunk_type']} "
                f"section={item['section']}"
            )

            print(
                f"      {preview}..."
            )


# ---------------------------------------------------------------------
# Negative-query diagnostics
# ---------------------------------------------------------------------

print("\n" + "=" * 92)
print("🚫 HYBRID NEGATIVE QUERY DIAGNOSTICS")

HYBRID_NEGATIVE_RESULTS = []

for document_label, benchmark in (
    DOCUMENT_BENCHMARKS.items()
):
    for question in benchmark[
        "negative_queries"
    ]:
        retrieved = retrieve_hybrid(
            result=benchmark["result"],
            question=question,
            top_k=3,
        )

        top_result = retrieved[0]

        HYBRID_NEGATIVE_RESULTS.append(
            {
                "document": document_label,
                "question": question,
                "top_result": top_result,
            }
        )

        print(
            f"\n{document_label}"
        )

        print(
            f"Question: {question}"
        )

        print(
            f"Hybrid score: "
            f"{top_result['hybrid_score']:.4f}"
        )

        print(
            f"Dense score: "
            f"{top_result['dense_score']:.4f}"
        )

        print(
            f"BM25 score: "
            f"{top_result['lexical_score']:.4f}"
        )

        print(
            f"Query coverage: "
            f"{top_result['coverage']:.2f}"
        )

        print(
            f"Top unrelated section: "
            f"{top_result['section']}"
        )


# ---------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------

hybrid_passed_count = sum(
    result["passed"]
    for result in HYBRID_POSITIVE_RESULTS
)

hybrid_total_count = len(
    HYBRID_POSITIVE_RESULTS
)

CELL_19B_HYBRID_PASSED = (
    hybrid_passed_count
    == hybrid_total_count
)

CELL_19B_RESULTS = {
    "positive_results": (
        HYBRID_POSITIVE_RESULTS
    ),
    "negative_results": (
        HYBRID_NEGATIVE_RESULTS
    ),
    "passed": (
        CELL_19B_HYBRID_PASSED
    ),
}

print("\n" + "=" * 92)
print("📊 CELL 19B SUMMARY")

print(
    f"Hybrid positive tests passed: "
    f"{hybrid_passed_count}"
    f"/{hybrid_total_count}"
)

if CELL_19B_HYBRID_PASSED:
    print(
        "✅ Hybrid retrieval found every expected "
        "answer signal inside the top-6 results."
    )

    print(
        "➡️ Continue to old-vs-new retrieval "
        "comparison before replacing vectors."
    )

else:
    print(
        "❌ At least one hybrid retrieval "
        "test still failed."
    )

    print(
        "⛔ Do not replace existing "
        "Qdrant vectors."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

⏳ Building generic BM25 indexes...
   ✅ Apple 10-K: 565 chunks
   ✅ Tesla Q4 2025: 258 chunks
   ✅ NIST AI RMF: 153 chunks

🔀 HYBRID RETRIEVAL BENCHMARKS

✅ PASS — Apple 10-K
Question: What were Apple's total net sales in fiscal year 2025?
   ✅ ['416,161'] → 416,161
   ✅ ['total net sales', 'net sales'] → total net sales
Top hybrid chunks:
   #1 hybrid=0.0469 dense_rank=4 bm25_rank=2 type=text section=Note 2 - Revenue
      (1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. The Company's proportion of net sales by disaggregated revenue source was ...
   #2 hybrid=0.0468 dense_rank=27 bm25_rank=4 type=table section=Note 2 - Revenue
      Note 2 - Revenue | | 2025 | 2024 | 2023 | |----------------------------------------------------------------------------------------------------|-----------|-----------|-----------| | Services (1) | 10...
   #3 hybrid=0.0451 dense_rank=32 bm25_rank=9 type=table section=Operating E

In [66]:
# VAULTIFY COMPACT - Cell 19C:
# Fair old-vs-new chunking comparison using the same hybrid retriever

from collections import Counter
import statistics

import numpy as np

from qdrant_client.models import (
    FieldCondition,
    Filter,
    MatchValue,
)


# ---------------------------------------------------------------------
# 1. Verify previous benchmark components
# ---------------------------------------------------------------------

required_names = [
    "retrieve_hybrid",
    "build_bm25_index",
    "hybrid_expected_groups_found",
    "apple_result",
    "tesla_result",
    "embedding_model",
    "qdrant",
    "COLLECTION_NAME",
    "DEMO_TENANTS",
    "TENANT_ID_FIELD",
    "TEXT_FIELD",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 16–19B first."
    )


# ---------------------------------------------------------------------
# 2. Load one complete seed tenant from Qdrant
# ---------------------------------------------------------------------

def load_seed_chunks_from_qdrant(
    tenant_id,
):
    """
    Load every text payload for one validated seed tenant.
    """
    points = []
    next_offset = None

    tenant_filter = Filter(
        must=[
            FieldCondition(
                key=TENANT_ID_FIELD,
                match=MatchValue(
                    value=tenant_id
                ),
            )
        ]
    )

    while True:
        batch, next_offset = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=tenant_filter,
            limit=256,
            offset=next_offset,
            with_payload=True,
            with_vectors=False,
        )

        points.extend(batch)

        if next_offset is None:
            break

    chunks = []

    for point in points:
        payload = point.payload or {}

        chunk_text = str(
            payload.get(
                TEXT_FIELD,
                "",
            )
        ).strip()

        if not chunk_text:
            continue

        chunks.append(
            {
                "text": chunk_text,
                "chunk_type": payload.get(
                    CHUNK_TYPE_FIELD,
                    "unknown",
                ),
               "section": payload.get(
    "section",
    "Unknown section",
),
"chunk_index": payload.get(
    CHUNK_INDEX_FIELD,
    len(chunks),
),
            }
        )

    if not chunks:
        raise RuntimeError(
            f"No seed chunks found for tenant: "
            f"{tenant_id}"
        )

    return chunks


# ---------------------------------------------------------------------
# 3. Convert chunks into the same in-memory retrieval format
# ---------------------------------------------------------------------

def prepare_comparison_result(
    label,
    chunks,
):
    """
    Create normalized embeddings and BM25 index using exactly
    the same settings for old and new chunk sets.
    """
    print(
        f"⏳ Preparing {label}: "
        f"{len(chunks)} chunks"
    )

    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    embeddings = embedding_model.encode(
        chunk_texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    result = {
        "filename": label,
        "chunks": chunks,
        "embeddings": embeddings,
    }

    result["bm25_index"] = (
        build_bm25_index(
            chunks
        )
    )

    return result


print("📦 Loading validated seed chunks from Qdrant...")

old_apple_chunks = (
    load_seed_chunks_from_qdrant(
        DEMO_TENANTS[
            "company_a"
        ]["tenant_id"]
    )
)

old_tesla_chunks = (
    load_seed_chunks_from_qdrant(
        DEMO_TENANTS[
            "company_b"
        ]["tenant_id"]
    )
)

print(
    f"✅ Old Apple chunks: "
    f"{len(old_apple_chunks)}"
)

print(
    f"✅ Old Tesla chunks: "
    f"{len(old_tesla_chunks)}"
)


OLD_APPLE_RESULT = (
    prepare_comparison_result(
        "Old Apple seed",
        old_apple_chunks,
    )
)

NEW_APPLE_RESULT = (
    prepare_comparison_result(
        "New Apple safe",
        apple_result["chunks"],
    )
)

OLD_TESLA_RESULT = (
    prepare_comparison_result(
        "Old Tesla seed",
        old_tesla_chunks,
    )
)

NEW_TESLA_RESULT = (
    prepare_comparison_result(
        "New Tesla safe",
        tesla_result["chunks"],
    )
)


# ---------------------------------------------------------------------
# 4. Define comparison questions
# ---------------------------------------------------------------------

COMPARISON_TESTS = {
    "Apple": {
        "old": OLD_APPLE_RESULT,
        "new": NEW_APPLE_RESULT,
        "queries": [
            {
                "question": (
                    "What were Apple's total net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["416,161", "416161"],
                    ["total net sales"],
                ],
            },
            {
                "question": (
                    "What were Apple's services net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["109,158", "109158"],
                    ["services"],
                ],
            },
            {
                "question": (
                    "What were Apple's iPhone net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    ["209,586", "209586"],
                    ["iphone"],
                ],
            },
        ],
    },

    "Tesla": {
        "old": OLD_TESLA_RESULT,
        "new": NEW_TESLA_RESULT,
        "queries": [
            {
                "question": (
                    "What was Tesla's total revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["24,901", "24901"],
                    [
                        "total revenue",
                        "total revenues",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's automotive revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["automotive revenues"],
                    ["q4-2025", "q4 2025"],
                ],
            },
            {
                "question": (
                    "What was Tesla's energy generation "
                    "and storage revenue in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "energy generation and storage",
                        "energy generation & storage",
                    ],
                    ["q4-2025", "q4 2025"],
                ],
            },
            {
                "question": (
                    "What was Tesla's GAAP operating income "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    ["operating income"],
                    ["q4-2025", "q4 2025"],
                ],
            },
        ],
    },
}


# ---------------------------------------------------------------------
# 5. Find the best rank containing all expected evidence
# ---------------------------------------------------------------------

def normalize_comparison_text(text):
    normalized = " ".join(
        str(text or "").lower().split()
    )

    compact = "".join(
        character
        for character in normalized
        if character.isalnum()
    )

    return normalized, compact


def chunk_contains_expected_groups(
    text,
    expected_groups,
):
    normalized_text, compact_text = (
        normalize_comparison_text(
            text
        )
    )

    for alternatives in expected_groups:
        group_found = False

        for alternative in alternatives:
            (
                normalized_alternative,
                compact_alternative,
            ) = normalize_comparison_text(
                alternative
            )

            if (
                normalized_alternative
                in normalized_text
                or (
                    compact_alternative
                    and compact_alternative
                    in compact_text
                )
            ):
                group_found = True
                break

        if not group_found:
            return False

    return True


def evaluate_one_chunk_set(
    result,
    question,
    expected_groups,
    retrieval_depth=20,
):
    retrieved = retrieve_hybrid(
        result=result,
        question=question,
        top_k=retrieval_depth,
    )

    best_complete_rank = None

    for item in retrieved:
        if chunk_contains_expected_groups(
            item["text"],
            expected_groups,
        ):
            best_complete_rank = item[
                "rank"
            ]
            break

    context_passed, group_results = (
        hybrid_expected_groups_found(
            retrieved[:6],
            expected_groups,
        )
    )

    return {
        "retrieved": retrieved,
        "best_complete_rank": (
            best_complete_rank
        ),
        "top_6_context_passed": (
            context_passed
        ),
        "group_results": (
            group_results
        ),
    }


# ---------------------------------------------------------------------
# 6. Run fair old-vs-new comparison
# ---------------------------------------------------------------------

OLD_VS_NEW_RESULTS = []

print("\n" + "=" * 96)
print("⚖️ OLD VS NEW CHUNKING — SAME HYBRID RETRIEVER")

for document_label, test_config in (
    COMPARISON_TESTS.items()
):
    print(
        f"\n{'=' * 96}"
    )

    print(
        f"📄 {document_label}"
    )

    for test_case in test_config[
        "queries"
    ]:
        question = test_case[
            "question"
        ]

        expected_groups = test_case[
            "expected_groups"
        ]

        old_evaluation = (
            evaluate_one_chunk_set(
                result=test_config["old"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        new_evaluation = (
            evaluate_one_chunk_set(
                result=test_config["new"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        old_rank = old_evaluation[
            "best_complete_rank"
        ]

        new_rank = new_evaluation[
            "best_complete_rank"
        ]

        if old_rank is None:
            old_rank_label = ">20"
        else:
            old_rank_label = str(
                old_rank
            )

        if new_rank is None:
            new_rank_label = ">20"
        else:
            new_rank_label = str(
                new_rank
            )

        if (
            new_rank is not None
            and (
                old_rank is None
                or new_rank < old_rank
            )
        ):
            winner = "NEW"

        elif (
            old_rank is not None
            and (
                new_rank is None
                or old_rank < new_rank
            )
        ):
            winner = "OLD"

        else:
            winner = "TIE"

        OLD_VS_NEW_RESULTS.append(
            {
                "document": (
                    document_label
                ),
                "question": question,
                "old_rank": old_rank,
                "new_rank": new_rank,
                "winner": winner,
                "old_top_6_passed": (
                    old_evaluation[
                        "top_6_context_passed"
                    ]
                ),
                "new_top_6_passed": (
                    new_evaluation[
                        "top_6_context_passed"
                    ]
                ),
            }
        )

        print(
            f"\nQuestion: {question}"
        )

        print(
            f"   Old complete-evidence rank: "
            f"{old_rank_label}"
        )

        print(
            f"   New complete-evidence rank: "
            f"{new_rank_label}"
        )

        print(
            f"   Old top-6 context pass: "
            f"{old_evaluation['top_6_context_passed']}"
        )

        print(
            f"   New top-6 context pass: "
            f"{new_evaluation['top_6_context_passed']}"
        )

        print(
            f"   Winner: {winner}"
        )

        print(
            "   Old top result:"
        )

        old_top = old_evaluation[
            "retrieved"
        ][0]

        print(
            f"      rank=1 "
            f"dense={old_top['dense_rank']} "
            f"bm25={old_top['lexical_rank']} "
            f"section={old_top['section']}"
        )

        print(
            "      "
            + " ".join(
                old_top["text"].split()
            )[:180]
            + "..."
        )

        print(
            "   New top result:"
        )

        new_top = new_evaluation[
            "retrieved"
        ][0]

        print(
            f"      rank=1 "
            f"dense={new_top['dense_rank']} "
            f"bm25={new_top['lexical_rank']} "
            f"section={new_top['section']}"
        )

        print(
            "      "
            + " ".join(
                new_top["text"].split()
            )[:180]
            + "..."
        )


# ---------------------------------------------------------------------
# 7. Final comparison summary
# ---------------------------------------------------------------------

new_wins = sum(
    result["winner"] == "NEW"
    for result in OLD_VS_NEW_RESULTS
)

old_wins = sum(
    result["winner"] == "OLD"
    for result in OLD_VS_NEW_RESULTS
)

ties = sum(
    result["winner"] == "TIE"
    for result in OLD_VS_NEW_RESULTS
)

new_top_6_passes = sum(
    result["new_top_6_passed"]
    for result in OLD_VS_NEW_RESULTS
)

old_top_6_passes = sum(
    result["old_top_6_passed"]
    for result in OLD_VS_NEW_RESULTS
)

total_tests = len(
    OLD_VS_NEW_RESULTS
)

CELL_19C_RESULTS = {
    "results": OLD_VS_NEW_RESULTS,
    "new_wins": new_wins,
    "old_wins": old_wins,
    "ties": ties,
    "new_top_6_passes": (
        new_top_6_passes
    ),
    "old_top_6_passes": (
        old_top_6_passes
    ),
    "total_tests": total_tests,
}

print("\n" + "=" * 96)
print("📊 CELL 19C SUMMARY")

print(
    f"New chunker wins: "
    f"{new_wins}"
)

print(
    f"Old chunker wins: "
    f"{old_wins}"
)

print(
    f"Ties: "
    f"{ties}"
)

print(
    f"Old top-6 passes: "
    f"{old_top_6_passes}"
    f"/{total_tests}"
)

print(
    f"New top-6 passes: "
    f"{new_top_6_passes}"
    f"/{total_tests}"
)

CELL_19C_NEW_CHUNKER_APPROVED = (
    new_top_6_passes
    >= old_top_6_passes
    and new_wins >= old_wins
)

if CELL_19C_NEW_CHUNKER_APPROVED:
    print(
        "✅ The new safe chunker is at least as reliable "
        "as the old chunker under the same hybrid retriever."
    )

    print(
        "➡️ It is eligible for a staged Qdrant migration."
    )

else:
    print(
        "❌ The new safe chunker did not match "
        "the old retrieval quality."
    )

    print(
        "⛔ Do not replace existing vectors."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

📦 Loading validated seed chunks from Qdrant...
✅ Old Apple chunks: 609
✅ Old Tesla chunks: 140
⏳ Preparing Old Apple seed: 609 chunks


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

⏳ Preparing New Apple safe: 565 chunks


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

⏳ Preparing Old Tesla seed: 140 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

⏳ Preparing New Tesla safe: 258 chunks


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


⚖️ OLD VS NEW CHUNKING — SAME HYBRID RETRIEVER

📄 Apple

Question: What were Apple's total net sales in fiscal year 2025?
   Old complete-evidence rank: 1
   New complete-evidence rank: 2
   Old top-6 context pass: True
   New top-6 context pass: True
   Winner: OLD
   Old top result:
      rank=1 dense=3 bm25=11 section=Note 2 - Revenue
      Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 2023 | | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357 | | iPad | 28,023 | 26,694 | 28,30...
   New top result:
      rank=1 dense=4 bm25=2 section=Note 2 - Revenue
      (1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. The Company's proportion of net sales by disaggregated...

Question: What were Apple's services net sales in fiscal year 2025?
   Old complete-evidence rank: 7
   New complete-evidence rank: 13
   Old top-6 context pass: False
   New top-6 context pass: False
   Wi

In [67]:
# VAULTIFY COMPACT - Cell 20A:
# Build the canonical seed-style, token-safe chunker V2
# and rechunk the preserved Markdown documents.

from collections import Counter
import hashlib
import re
import statistics


# ---------------------------------------------------------------------
# 1. Runtime validation
# ---------------------------------------------------------------------

if (
    "DRY_RUN_RESULTS" not in globals()
    or len(DRY_RUN_RESULTS) < 3
):
    raise RuntimeError(
        "Preserved Apple, Tesla, and NIST dry-run results "
        "were not found. Run Cell 18 and Cell 18A first."
    )


V2_TOKENIZER = embedding_model.tokenizer

V2_MAX_CHUNK_TOKENS = min(
    240,
    embedding_model.max_seq_length - 16,
)

V2_SPECIAL_TOKEN_BUFFER = 2

V2_TOKEN_PAYLOAD_LIMIT = (
    V2_MAX_CHUNK_TOKENS
    - V2_SPECIAL_TOKEN_BUFFER
)

# Preserve section and column information without
# allowing the prefix to consume the whole chunk.
V2_MAX_TABLE_PREFIX_TOKENS = min(
    96,
    V2_TOKEN_PAYLOAD_LIMIT // 2,
)

# These are the validated seed-pipeline text settings.
V2_TEXT_CHUNK_SIZE = 800
V2_TEXT_CHUNK_OVERLAP = 120


# ---------------------------------------------------------------------
# 2. Token utilities
# ---------------------------------------------------------------------

def v2_get_raw_token_ids(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return list(token_ids)


def v2_count_raw_tokens(text):
    return len(
        v2_get_raw_token_ids(text)
    )


def v2_count_embedding_tokens(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return len(token_ids)


def v2_decode_token_slice(token_ids):
    return V2_TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()


def v2_truncate_to_token_limit(
    text,
    token_limit,
):
    text = str(text or "").strip()

    if not text or token_limit <= 0:
        return ""

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return text

    return v2_decode_token_slice(
        token_ids[:token_limit]
    )


def v2_split_to_token_safe_text(
    text,
    token_limit=V2_TOKEN_PAYLOAD_LIMIT,
):
    text = str(text or "").strip()

    if not text:
        return []

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return [text]

    safe_pieces = []

    for start_index in range(
        0,
        len(token_ids),
        token_limit,
    ):
        token_slice = token_ids[
            start_index:
            start_index + token_limit
        ]

        decoded_piece = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if decoded_piece:
            safe_pieces.append(
                decoded_piece
            )

    return safe_pieces


# ---------------------------------------------------------------------
# 3. Table helpers
# ---------------------------------------------------------------------

def v2_is_markdown_separator_row(line):
    stripped_line = str(
        line or ""
    ).strip()

    if not stripped_line.startswith("|"):
        return False

    cells = [
        cell.strip()
        for cell in stripped_line
        .strip("|")
        .split("|")
    ]

    if not cells:
        return False

    return all(
        bool(
            re.fullmatch(
                r":?-{3,}:?",
                cell,
            )
        )
        for cell in cells
    )


def v2_build_safe_table_prefix(
    section,
    header_line,
):
    """
    Give tables explicit semantic labels and remove
    token-expensive Markdown separator rows.
    """
    section = (
        str(section or "").strip()
        or "Unknown section"
    )

    header_line = str(
        header_line or ""
    ).strip()

    section_label = (
        f"Section: {section}\n"
        "Table columns:\n"
    )

    section_token_count = (
        v2_count_raw_tokens(
            section_label
        )
    )

    if (
        section_token_count
        >= V2_MAX_TABLE_PREFIX_TOKENS
    ):
        return v2_truncate_to_token_limit(
            section_label,
            V2_MAX_TABLE_PREFIX_TOKENS,
        )

    available_header_tokens = (
        V2_MAX_TABLE_PREFIX_TOKENS
        - section_token_count
    )

    compact_header = (
        v2_truncate_to_token_limit(
            header_line,
            available_header_tokens,
        )
    )

    return (
        section_label
        + compact_header
    ).strip()


def v2_create_chunk(
    text,
    chunk_type,
    section,
):
    return {
        "text": str(text or "").strip(),
        "chunk_type": chunk_type,
        "section": (
            str(section or "").strip()
            or "Document content"
        ),
    }


# ---------------------------------------------------------------------
# 4. Text chunking
# ---------------------------------------------------------------------

def v2_split_normal_text(
    text,
    section,
):
    text = str(text or "").strip()

    if not text:
        return []

    chunks = []
    start_index = 0

    while start_index < len(text):
        end_index = min(
            start_index
            + V2_TEXT_CHUNK_SIZE,
            len(text),
        )

        character_piece = text[
            start_index:end_index
        ].strip()

        for safe_piece in (
            v2_split_to_token_safe_text(
                character_piece
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="text",
                    section=section,
                )
            )

        if end_index >= len(text):
            break

        next_start_index = (
            end_index
            - V2_TEXT_CHUNK_OVERLAP
        )

        start_index = max(
            next_start_index,
            start_index + 1,
        )

    return chunks


# ---------------------------------------------------------------------
# 5. Table chunking
# ---------------------------------------------------------------------

def v2_split_oversized_table_row(
    row,
    table_prefix,
    section,
):
    prefix_with_newline = (
        table_prefix.rstrip()
        + "\n"
    )

    prefix_token_count = (
        v2_count_raw_tokens(
            prefix_with_newline
        )
    )

    available_row_tokens = (
        V2_TOKEN_PAYLOAD_LIMIT
        - prefix_token_count
    )

    if available_row_tokens <= 0:
        raise RuntimeError(
            "The table prefix consumed the complete token budget. "
            f"Section: {section}"
        )

    row_token_ids = (
        v2_get_raw_token_ids(
            row
        )
    )

    chunks = []

    for start_index in range(
        0,
        len(row_token_ids),
        available_row_tokens,
    ):
        token_slice = row_token_ids[
            start_index:
            start_index
            + available_row_tokens
        ]

        decoded_row = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if not decoded_row:
            continue

        chunk_text = (
            prefix_with_newline
            + decoded_row
        ).strip()

        if (
            v2_count_embedding_tokens(
                chunk_text
            )
            > V2_MAX_CHUNK_TOKENS
        ):
            raise RuntimeError(
                "An oversized table-row piece "
                "was generated unexpectedly."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

    return chunks


def v2_split_table(
    table_lines,
    section,
):
    if not table_lines:
        return []

    cleaned_lines = [
        line.strip()
        for line in table_lines
        if line.strip()
    ]

    if not cleaned_lines:
        return []

    header_line = cleaned_lines[0]

    if (
        len(cleaned_lines) > 1
        and v2_is_markdown_separator_row(
            cleaned_lines[1]
        )
    ):
        data_rows = cleaned_lines[2:]
    else:
        data_rows = cleaned_lines[1:]

    table_prefix = (
        v2_build_safe_table_prefix(
            section=section,
            header_line=header_line,
        )
    )

    chunks = []
    current_rows = []

    def build_table_text(rows):
        return (
            table_prefix.rstrip()
            + "\n"
            + "\n".join(rows)
        ).strip()

    def flush_current_rows():
        nonlocal current_rows

        if not current_rows:
            return

        chunk_text = build_table_text(
            current_rows
        )

        token_count = (
            v2_count_embedding_tokens(
                chunk_text
            )
        )

        if token_count > V2_MAX_CHUNK_TOKENS:
            raise RuntimeError(
                "An internal table chunk exceeded "
                "the 240-token limit."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

        current_rows = []

    if not data_rows:
        for safe_piece in (
            v2_split_to_token_safe_text(
                table_prefix
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="table",
                    section=section,
                )
            )

        return chunks

    for row in data_rows:
        candidate_rows = (
            current_rows + [row]
        )

        candidate_text = (
            build_table_text(
                candidate_rows
            )
        )

        if (
            v2_count_embedding_tokens(
                candidate_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = candidate_rows
            continue

        flush_current_rows()

        single_row_text = (
            build_table_text([row])
        )

        if (
            v2_count_embedding_tokens(
                single_row_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = [row]

        else:
            chunks.extend(
                v2_split_oversized_table_row(
                    row=row,
                    table_prefix=table_prefix,
                    section=section,
                )
            )

    flush_current_rows()

    return chunks


# ---------------------------------------------------------------------
# 6. Canonical Markdown chunker
# ---------------------------------------------------------------------

def canonical_chunk_markdown_v2(
    model,
    markdown_text,
):
    """
    Canonical Vaultify chunker:

    - validated seed-style table serialization
    - explicit section and column labels
    - separator-row removal
    - strict 240-token maximum
    - oversized-row splitting
    - exact duplicate removal
    """
    lines = str(
        markdown_text or ""
    ).splitlines()

    chunks = []
    current_section = (
        "Document introduction"
    )

    text_buffer = []

    def flush_text_buffer():
        nonlocal text_buffer

        buffered_text = "\n".join(
            text_buffer
        ).strip()

        if buffered_text:
            chunks.extend(
                v2_split_normal_text(
                    text=buffered_text,
                    section=current_section,
                )
            )

        text_buffer = []

    line_index = 0

    while line_index < len(lines):
        line = lines[line_index]
        stripped_line = line.strip()

        heading_match = re.match(
            r"^(#{1,6})\s+(.+)$",
            stripped_line,
        )

        if heading_match:
            flush_text_buffer()

            current_section = (
                heading_match
                .group(2)
                .strip()
            )

            text_buffer.append(
                stripped_line
            )

            line_index += 1
            continue

        if stripped_line.startswith("|"):
            flush_text_buffer()

            table_lines = []

            while (
                line_index < len(lines)
                and lines[line_index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[line_index]
                )

                line_index += 1

            chunks.extend(
                v2_split_table(
                    table_lines=table_lines,
                    section=current_section,
                )
            )

            continue

        text_buffer.append(line)
        line_index += 1

    flush_text_buffer()

    # Exact deduplication.
    unique_chunks = []
    seen_hashes = set()

    for chunk in chunks:
        normalized_text = re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()

        if not normalized_text:
            continue

        chunk_hash = hashlib.sha256(
            normalized_text.encode(
                "utf-8"
            )
        ).hexdigest()

        if chunk_hash in seen_hashes:
            continue

        seen_hashes.add(
            chunk_hash
        )

        chunk["chunk_index"] = len(
            unique_chunks
        )

        unique_chunks.append(
            chunk
        )

    oversized_chunks = [
        chunk
        for chunk in unique_chunks
        if (
            v2_count_embedding_tokens(
                chunk["text"]
            )
            > V2_MAX_CHUNK_TOKENS
        )
    ]

    if oversized_chunks:
        raise RuntimeError(
            f"Canonical chunker generated "
            f"{len(oversized_chunks)} oversized chunks."
        )

    return unique_chunks


# ---------------------------------------------------------------------
# 7. Rechunk preserved documents without reparsing
# ---------------------------------------------------------------------

DRY_RUN_RESULTS_V2 = {}

print(
    "⏳ Rechunking preserved Markdown "
    "with Canonical Chunker V2..."
)

for (
    document_hash,
    preserved_result,
) in DRY_RUN_RESULTS.items():

    filename = preserved_result[
        "filename"
    ]

    markdown_text = preserved_result[
        "markdown"
    ]

    chunks = canonical_chunk_markdown_v2(
        embedding_model,
        markdown_text,
    )

    token_counts = [
        v2_count_embedding_tokens(
            chunk["text"]
        )
        for chunk in chunks
    ]

    chunk_types = Counter(
        chunk["chunk_type"]
        for chunk in chunks
    )

    normalized_texts = [
        re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()
        for chunk in chunks
    ]

    duplicate_count = (
        len(normalized_texts)
        - len(set(normalized_texts))
    )

    oversized_count = sum(
        token_count
        > V2_MAX_CHUNK_TOKENS
        for token_count
        in token_counts
    )

    report = {
        "filename": filename,
        "total_chunks": len(chunks),
        "chunk_types": dict(
            chunk_types
        ),
        "token_min": min(
            token_counts
        ),
        "token_median": (
            statistics.median(
                token_counts
            )
        ),
        "token_p95": sorted(
            token_counts
        )[
            round(
                (len(token_counts) - 1)
                * 0.95
            )
        ],
        "token_max": max(
            token_counts
        ),
        "exact_duplicates": (
            duplicate_count
        ),
        "oversized": (
            oversized_count
        ),
    }

    DRY_RUN_RESULTS_V2[
        document_hash
    ] = {
        "filename": filename,
        "document_hash": (
            document_hash
        ),
        "markdown": markdown_text,
        "chunks": chunks,
        "report": report,
    }

    print(
        f"\n✅ {filename}"
    )

    print(
        f"   Total chunks: "
        f"{report['total_chunks']}"
    )

    print(
        f"   Types: "
        f"{report['chunk_types']}"
    )

    print(
        "   Tokens:"
        f" median={report['token_median']:.1f},"
        f" p95={report['token_p95']},"
        f" max={report['token_max']}"
    )

    print(
        f"   Oversized: "
        f"{report['oversized']}"
    )

    print(
        f"   Exact duplicates: "
        f"{report['exact_duplicates']}"
    )


print("\n" + "=" * 88)
print("✅ Canonical Chunker V2 dry-run completed.")
print(
    f"📦 Documents processed: "
    f"{len(DRY_RUN_RESULTS_V2)}"
)
print("⏭️ Existing web backend was not patched.")
print("⏭️ No embeddings were generated.")
print("⏭️ No Qdrant points were changed.")

⏳ Rechunking preserved Markdown with Canonical Chunker V2...

✅ _10-K-2025-As-Filed.pdf
   Total chunks: 599
   Types: {'text': 512, 'table': 87}
   Tokens: median=133.0, p95=215, max=240
   Oversized: 0
   Exact duplicates: 0

✅ TSLA-Q4-2025-Update.pdf
   Total chunks: 136
   Types: {'text': 67, 'table': 69}
   Tokens: median=156.5, p95=238, max=240
   Oversized: 0
   Exact duplicates: 0

✅ nist.ai.100-1 (1).pdf
   Total chunks: 183
   Types: {'text': 151, 'table': 32}
   Tokens: median=148.0, p95=207, max=240
   Oversized: 0
   Exact duplicates: 0

✅ Canonical Chunker V2 dry-run completed.
📦 Documents processed: 3
⏭️ Existing web backend was not patched.
⏭️ No embeddings were generated.
⏭️ No Qdrant points were changed.


In [68]:
# VAULTIFY COMPACT - Cell 20A:
# Build the canonical seed-style, token-safe chunker V2
# and rechunk the preserved Markdown documents.

from collections import Counter
import hashlib
import re
import statistics


# ---------------------------------------------------------------------
# 1. Runtime validation
# ---------------------------------------------------------------------

if (
    "DRY_RUN_RESULTS" not in globals()
    or len(DRY_RUN_RESULTS) < 3
):
    raise RuntimeError(
        "Preserved Apple, Tesla, and NIST dry-run results "
        "were not found. Run Cell 18 and Cell 18A first."
    )


V2_TOKENIZER = embedding_model.tokenizer

V2_MAX_CHUNK_TOKENS = min(
    240,
    embedding_model.max_seq_length - 16,
)

V2_SPECIAL_TOKEN_BUFFER = 2

V2_TOKEN_PAYLOAD_LIMIT = (
    V2_MAX_CHUNK_TOKENS
    - V2_SPECIAL_TOKEN_BUFFER
)

# Preserve section and column information without
# allowing the prefix to consume the whole chunk.
V2_MAX_TABLE_PREFIX_TOKENS = min(
    96,
    V2_TOKEN_PAYLOAD_LIMIT // 2,
)

# These are the validated seed-pipeline text settings.
V2_TEXT_CHUNK_SIZE = 800
V2_TEXT_CHUNK_OVERLAP = 120


# ---------------------------------------------------------------------
# 2. Token utilities
# ---------------------------------------------------------------------

def v2_get_raw_token_ids(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return list(token_ids)


def v2_count_raw_tokens(text):
    return len(
        v2_get_raw_token_ids(text)
    )


def v2_count_embedding_tokens(text):
    encoded = V2_TOKENIZER(
        str(text or ""),
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        token_ids = token_ids[0]

    return len(token_ids)


def v2_decode_token_slice(token_ids):
    return V2_TOKENIZER.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()


def v2_truncate_to_token_limit(
    text,
    token_limit,
):
    text = str(text or "").strip()

    if not text or token_limit <= 0:
        return ""

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return text

    return v2_decode_token_slice(
        token_ids[:token_limit]
    )


def v2_split_to_token_safe_text(
    text,
    token_limit=V2_TOKEN_PAYLOAD_LIMIT,
):
    text = str(text or "").strip()

    if not text:
        return []

    token_ids = v2_get_raw_token_ids(
        text
    )

    if len(token_ids) <= token_limit:
        return [text]

    safe_pieces = []

    for start_index in range(
        0,
        len(token_ids),
        token_limit,
    ):
        token_slice = token_ids[
            start_index:
            start_index + token_limit
        ]

        decoded_piece = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if decoded_piece:
            safe_pieces.append(
                decoded_piece
            )

    return safe_pieces


# ---------------------------------------------------------------------
# 3. Table helpers
# ---------------------------------------------------------------------

def v2_is_markdown_separator_row(line):
    stripped_line = str(
        line or ""
    ).strip()

    if not stripped_line.startswith("|"):
        return False

    cells = [
        cell.strip()
        for cell in stripped_line
        .strip("|")
        .split("|")
    ]

    if not cells:
        return False

    return all(
        bool(
            re.fullmatch(
                r":?-{3,}:?",
                cell,
            )
        )
        for cell in cells
    )


def v2_build_safe_table_prefix(
    section,
    header_line,
):
    """
    Give tables explicit semantic labels and remove
    token-expensive Markdown separator rows.
    """
    section = (
        str(section or "").strip()
        or "Unknown section"
    )

    header_line = str(
        header_line or ""
    ).strip()

    section_label = (
        f"Section: {section}\n"
        "Table columns:\n"
    )

    section_token_count = (
        v2_count_raw_tokens(
            section_label
        )
    )

    if (
        section_token_count
        >= V2_MAX_TABLE_PREFIX_TOKENS
    ):
        return v2_truncate_to_token_limit(
            section_label,
            V2_MAX_TABLE_PREFIX_TOKENS,
        )

    available_header_tokens = (
        V2_MAX_TABLE_PREFIX_TOKENS
        - section_token_count
    )

    compact_header = (
        v2_truncate_to_token_limit(
            header_line,
            available_header_tokens,
        )
    )

    return (
        section_label
        + compact_header
    ).strip()


def v2_create_chunk(
    text,
    chunk_type,
    section,
):
    return {
        "text": str(text or "").strip(),
        "chunk_type": chunk_type,
        "section": (
            str(section or "").strip()
            or "Document content"
        ),
    }


# ---------------------------------------------------------------------
# 4. Text chunking
# ---------------------------------------------------------------------

def v2_split_normal_text(
    text,
    section,
):
    text = str(text or "").strip()

    if not text:
        return []

    chunks = []
    start_index = 0

    while start_index < len(text):
        end_index = min(
            start_index
            + V2_TEXT_CHUNK_SIZE,
            len(text),
        )

        character_piece = text[
            start_index:end_index
        ].strip()

        for safe_piece in (
            v2_split_to_token_safe_text(
                character_piece
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="text",
                    section=section,
                )
            )

        if end_index >= len(text):
            break

        next_start_index = (
            end_index
            - V2_TEXT_CHUNK_OVERLAP
        )

        start_index = max(
            next_start_index,
            start_index + 1,
        )

    return chunks


# ---------------------------------------------------------------------
# 5. Table chunking
# ---------------------------------------------------------------------

def v2_split_oversized_table_row(
    row,
    table_prefix,
    section,
):
    prefix_with_newline = (
        table_prefix.rstrip()
        + "\n"
    )

    prefix_token_count = (
        v2_count_raw_tokens(
            prefix_with_newline
        )
    )

    available_row_tokens = (
        V2_TOKEN_PAYLOAD_LIMIT
        - prefix_token_count
    )

    if available_row_tokens <= 0:
        raise RuntimeError(
            "The table prefix consumed the complete token budget. "
            f"Section: {section}"
        )

    row_token_ids = (
        v2_get_raw_token_ids(
            row
        )
    )

    chunks = []

    for start_index in range(
        0,
        len(row_token_ids),
        available_row_tokens,
    ):
        token_slice = row_token_ids[
            start_index:
            start_index
            + available_row_tokens
        ]

        decoded_row = (
            v2_decode_token_slice(
                token_slice
            )
        )

        if not decoded_row:
            continue

        chunk_text = (
            prefix_with_newline
            + decoded_row
        ).strip()

        if (
            v2_count_embedding_tokens(
                chunk_text
            )
            > V2_MAX_CHUNK_TOKENS
        ):
            raise RuntimeError(
                "An oversized table-row piece "
                "was generated unexpectedly."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

    return chunks


def v2_split_table(
    table_lines,
    section,
):
    if not table_lines:
        return []

    cleaned_lines = [
        line.strip()
        for line in table_lines
        if line.strip()
    ]

    if not cleaned_lines:
        return []

    header_line = cleaned_lines[0]

    if (
        len(cleaned_lines) > 1
        and v2_is_markdown_separator_row(
            cleaned_lines[1]
        )
    ):
        data_rows = cleaned_lines[2:]
    else:
        data_rows = cleaned_lines[1:]

    table_prefix = (
        v2_build_safe_table_prefix(
            section=section,
            header_line=header_line,
        )
    )

    chunks = []
    current_rows = []

    def build_table_text(rows):
        return (
            table_prefix.rstrip()
            + "\n"
            + "\n".join(rows)
        ).strip()

    def flush_current_rows():
        nonlocal current_rows

        if not current_rows:
            return

        chunk_text = build_table_text(
            current_rows
        )

        token_count = (
            v2_count_embedding_tokens(
                chunk_text
            )
        )

        if token_count > V2_MAX_CHUNK_TOKENS:
            raise RuntimeError(
                "An internal table chunk exceeded "
                "the 240-token limit."
            )

        chunks.append(
            v2_create_chunk(
                text=chunk_text,
                chunk_type="table",
                section=section,
            )
        )

        current_rows = []

    if not data_rows:
        for safe_piece in (
            v2_split_to_token_safe_text(
                table_prefix
            )
        ):
            chunks.append(
                v2_create_chunk(
                    text=safe_piece,
                    chunk_type="table",
                    section=section,
                )
            )

        return chunks

    for row in data_rows:
        candidate_rows = (
            current_rows + [row]
        )

        candidate_text = (
            build_table_text(
                candidate_rows
            )
        )

        if (
            v2_count_embedding_tokens(
                candidate_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = candidate_rows
            continue

        flush_current_rows()

        single_row_text = (
            build_table_text([row])
        )

        if (
            v2_count_embedding_tokens(
                single_row_text
            )
            <= V2_MAX_CHUNK_TOKENS
        ):
            current_rows = [row]

        else:
            chunks.extend(
                v2_split_oversized_table_row(
                    row=row,
                    table_prefix=table_prefix,
                    section=section,
                )
            )

    flush_current_rows()

    return chunks


# ---------------------------------------------------------------------
# 6. Canonical Markdown chunker
# ---------------------------------------------------------------------

def canonical_chunk_markdown_v2(
    model,
    markdown_text,
):
    """
    Canonical Vaultify chunker:

    - validated seed-style table serialization
    - explicit section and column labels
    - separator-row removal
    - strict 240-token maximum
    - oversized-row splitting
    - exact duplicate removal
    """
    lines = str(
        markdown_text or ""
    ).splitlines()

    chunks = []
    current_section = (
        "Document introduction"
    )

    text_buffer = []

    def flush_text_buffer():
        nonlocal text_buffer

        buffered_text = "\n".join(
            text_buffer
        ).strip()

        if buffered_text:
            chunks.extend(
                v2_split_normal_text(
                    text=buffered_text,
                    section=current_section,
                )
            )

        text_buffer = []

    line_index = 0

    while line_index < len(lines):
        line = lines[line_index]
        stripped_line = line.strip()

        heading_match = re.match(
            r"^(#{1,6})\s+(.+)$",
            stripped_line,
        )

        if heading_match:
            flush_text_buffer()

            current_section = (
                heading_match
                .group(2)
                .strip()
            )

            text_buffer.append(
                stripped_line
            )

            line_index += 1
            continue

        if stripped_line.startswith("|"):
            flush_text_buffer()

            table_lines = []

            while (
                line_index < len(lines)
                and lines[line_index]
                .strip()
                .startswith("|")
            ):
                table_lines.append(
                    lines[line_index]
                )

                line_index += 1

            chunks.extend(
                v2_split_table(
                    table_lines=table_lines,
                    section=current_section,
                )
            )

            continue

        text_buffer.append(line)
        line_index += 1

    flush_text_buffer()

    # Exact deduplication.
    unique_chunks = []
    seen_hashes = set()

    for chunk in chunks:
        normalized_text = re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()

        if not normalized_text:
            continue

        chunk_hash = hashlib.sha256(
            normalized_text.encode(
                "utf-8"
            )
        ).hexdigest()

        if chunk_hash in seen_hashes:
            continue

        seen_hashes.add(
            chunk_hash
        )

        chunk["chunk_index"] = len(
            unique_chunks
        )

        unique_chunks.append(
            chunk
        )

    oversized_chunks = [
        chunk
        for chunk in unique_chunks
        if (
            v2_count_embedding_tokens(
                chunk["text"]
            )
            > V2_MAX_CHUNK_TOKENS
        )
    ]

    if oversized_chunks:
        raise RuntimeError(
            f"Canonical chunker generated "
            f"{len(oversized_chunks)} oversized chunks."
        )

    return unique_chunks


# ---------------------------------------------------------------------
# 7. Rechunk preserved documents without reparsing
# ---------------------------------------------------------------------

DRY_RUN_RESULTS_V2 = {}

print(
    "⏳ Rechunking preserved Markdown "
    "with Canonical Chunker V2..."
)

for (
    document_hash,
    preserved_result,
) in DRY_RUN_RESULTS.items():

    filename = preserved_result[
        "filename"
    ]

    markdown_text = preserved_result[
        "markdown"
    ]

    chunks = canonical_chunk_markdown_v2(
        embedding_model,
        markdown_text,
    )

    token_counts = [
        v2_count_embedding_tokens(
            chunk["text"]
        )
        for chunk in chunks
    ]

    chunk_types = Counter(
        chunk["chunk_type"]
        for chunk in chunks
    )

    normalized_texts = [
        re.sub(
            r"\s+",
            " ",
            chunk["text"],
        ).strip()
        for chunk in chunks
    ]

    duplicate_count = (
        len(normalized_texts)
        - len(set(normalized_texts))
    )

    oversized_count = sum(
        token_count
        > V2_MAX_CHUNK_TOKENS
        for token_count
        in token_counts
    )

    report = {
        "filename": filename,
        "total_chunks": len(chunks),
        "chunk_types": dict(
            chunk_types
        ),
        "token_min": min(
            token_counts
        ),
        "token_median": (
            statistics.median(
                token_counts
            )
        ),
        "token_p95": sorted(
            token_counts
        )[
            round(
                (len(token_counts) - 1)
                * 0.95
            )
        ],
        "token_max": max(
            token_counts
        ),
        "exact_duplicates": (
            duplicate_count
        ),
        "oversized": (
            oversized_count
        ),
    }

    DRY_RUN_RESULTS_V2[
        document_hash
    ] = {
        "filename": filename,
        "document_hash": (
            document_hash
        ),
        "markdown": markdown_text,
        "chunks": chunks,
        "report": report,
    }

    print(
        f"\n✅ {filename}"
    )

    print(
        f"   Total chunks: "
        f"{report['total_chunks']}"
    )

    print(
        f"   Types: "
        f"{report['chunk_types']}"
    )

    print(
        "   Tokens:"
        f" median={report['token_median']:.1f},"
        f" p95={report['token_p95']},"
        f" max={report['token_max']}"
    )

    print(
        f"   Oversized: "
        f"{report['oversized']}"
    )

    print(
        f"   Exact duplicates: "
        f"{report['exact_duplicates']}"
    )


print("\n" + "=" * 88)
print("✅ Canonical Chunker V2 dry-run completed.")
print(
    f"📦 Documents processed: "
    f"{len(DRY_RUN_RESULTS_V2)}"
)
print("⏭️ Existing web backend was not patched.")
print("⏭️ No embeddings were generated.")
print("⏭️ No Qdrant points were changed.")

⏳ Rechunking preserved Markdown with Canonical Chunker V2...

✅ _10-K-2025-As-Filed.pdf
   Total chunks: 599
   Types: {'text': 512, 'table': 87}
   Tokens: median=133.0, p95=215, max=240
   Oversized: 0
   Exact duplicates: 0

✅ TSLA-Q4-2025-Update.pdf
   Total chunks: 136
   Types: {'text': 67, 'table': 69}
   Tokens: median=156.5, p95=238, max=240
   Oversized: 0
   Exact duplicates: 0

✅ nist.ai.100-1 (1).pdf
   Total chunks: 183
   Types: {'text': 151, 'table': 32}
   Tokens: median=148.0, p95=207, max=240
   Oversized: 0
   Exact duplicates: 0

✅ Canonical Chunker V2 dry-run completed.
📦 Documents processed: 3
⏭️ Existing web backend was not patched.
⏭️ No embeddings were generated.
⏭️ No Qdrant points were changed.


In [69]:
# VAULTIFY COMPACT - Cell 20B:
# Compare Canonical Chunker V2 against the validated seed pipeline
# using the same hybrid retrieval system.

import re

import numpy as np


# ---------------------------------------------------------------------
# 1. Validate runtime dependencies
# ---------------------------------------------------------------------

required_names = [
    "DRY_RUN_RESULTS_V2",
    "OLD_APPLE_RESULT",
    "OLD_TESLA_RESULT",
    "embedding_model",
    "build_bm25_index",
    "retrieve_hybrid",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 19B, 19C, and 20A first."
    )


# ---------------------------------------------------------------------
# 2. Locate Canonical V2 documents
# ---------------------------------------------------------------------

def find_v2_document(filename_fragment):
    filename_fragment = (
        filename_fragment.lower()
    )

    matches = [
        result
        for result
        in DRY_RUN_RESULTS_V2.values()
        if filename_fragment
        in result["filename"].lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one Canonical V2 document matching "
            f"'{filename_fragment}', found {len(matches)}."
        )

    return matches[0]


v2_apple_document = find_v2_document(
    "10-k"
)

v2_tesla_document = find_v2_document(
    "tsla"
)


# ---------------------------------------------------------------------
# 3. Prepare Canonical V2 retrieval objects
# ---------------------------------------------------------------------

def prepare_v2_retrieval_result(
    label,
    preserved_document,
):
    chunks = preserved_document[
        "chunks"
    ]

    print(
        f"⏳ Preparing {label}: "
        f"{len(chunks)} chunks"
    )

    embeddings = embedding_model.encode(
        [
            chunk["text"]
            for chunk in chunks
        ],
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    result = {
        "filename": label,
        "chunks": chunks,
        "embeddings": embeddings,
    }

    result["bm25_index"] = (
        build_bm25_index(
            chunks
        )
    )

    return result


V2_APPLE_RESULT = (
    prepare_v2_retrieval_result(
        label="Canonical V2 Apple",
        preserved_document=(
            v2_apple_document
        ),
    )
)

V2_TESLA_RESULT = (
    prepare_v2_retrieval_result(
        label="Canonical V2 Tesla",
        preserved_document=(
            v2_tesla_document
        ),
    )
)


# ---------------------------------------------------------------------
# 4. Benchmark questions
# ---------------------------------------------------------------------

V2_COMPARISON_TESTS = {
    "Apple": {
        "old": OLD_APPLE_RESULT,
        "v2": V2_APPLE_RESULT,
        "queries": [
            {
                "question": (
                    "What were Apple's total net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    [
                        "416,161",
                        "416161",
                    ],
                    [
                        "total net sales",
                    ],
                ],
            },
            {
                "question": (
                    "What were Apple's services net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    [
                        "109,158",
                        "109158",
                    ],
                    [
                        "services",
                    ],
                ],
            },
            {
                "question": (
                    "What were Apple's iPhone net sales "
                    "in fiscal year 2025?"
                ),
                "expected_groups": [
                    [
                        "209,586",
                        "209586",
                    ],
                    [
                        "iphone",
                    ],
                ],
            },
        ],
    },

    "Tesla": {
        "old": OLD_TESLA_RESULT,
        "v2": V2_TESLA_RESULT,
        "queries": [
            {
                "question": (
                    "What was Tesla's total revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "24,901",
                        "24901",
                    ],
                    [
                        "total revenue",
                        "total revenues",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's automotive revenue "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "automotive revenues",
                        "automotive revenue",
                    ],
                    [
                        "q4-2025",
                        "q4 2025",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's energy generation "
                    "and storage revenue in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "energy generation and storage",
                        "energy generation & storage",
                    ],
                    [
                        "q4-2025",
                        "q4 2025",
                    ],
                ],
            },
            {
                "question": (
                    "What was Tesla's GAAP operating income "
                    "in Q4 2025?"
                ),
                "expected_groups": [
                    [
                        "gaap operating income",
                        "operating income",
                    ],
                    [
                        "1.4b",
                        "$1.4b",
                        "1.4 b",
                    ],
                    [
                        "q4",
                    ],
                ],
            },
        ],
    },
}


# ---------------------------------------------------------------------
# 5. Evidence matching
# ---------------------------------------------------------------------

def normalize_v2_evidence(text):
    normalized = " ".join(
        str(text or "")
        .lower()
        .split()
    )

    compact = re.sub(
        r"[^a-z0-9]+",
        "",
        normalized,
    )

    return normalized, compact


def v2_text_contains_groups(
    text,
    expected_groups,
):
    normalized_text, compact_text = (
        normalize_v2_evidence(
            text
        )
    )

    for alternatives in expected_groups:
        group_found = False

        for alternative in alternatives:
            (
                normalized_alternative,
                compact_alternative,
            ) = normalize_v2_evidence(
                alternative
            )

            if (
                normalized_alternative
                in normalized_text
                or (
                    compact_alternative
                    and compact_alternative
                    in compact_text
                )
            ):
                group_found = True
                break

        if not group_found:
            return False

    return True


def evaluate_v2_chunk_set(
    result,
    question,
    expected_groups,
    retrieval_depth=20,
):
    retrieved = retrieve_hybrid(
        result=result,
        question=question,
        top_k=retrieval_depth,
    )

    complete_evidence_rank = None

    for item in retrieved:
        if v2_text_contains_groups(
            item["text"],
            expected_groups,
        ):
            complete_evidence_rank = (
                item["rank"]
            )
            break

    combined_top_6_context = " ".join(
        item["text"]
        for item in retrieved[:6]
    )

    top_6_context_passed = (
        v2_text_contains_groups(
            combined_top_6_context,
            expected_groups,
        )
    )

    return {
        "retrieved": retrieved,
        "complete_evidence_rank": (
            complete_evidence_rank
        ),
        "top_6_context_passed": (
            top_6_context_passed
        ),
    }


# ---------------------------------------------------------------------
# 6. Run comparison
# ---------------------------------------------------------------------

CELL_20B_RESULTS = []

print("\n" + "=" * 96)
print(
    "⚖️ VALIDATED SEED VS CANONICAL V2 "
    "— SAME HYBRID RETRIEVER"
)

for document_label, config in (
    V2_COMPARISON_TESTS.items()
):
    print("\n" + "=" * 96)
    print(f"📄 {document_label}")

    for test_case in config[
        "queries"
    ]:
        question = test_case[
            "question"
        ]

        expected_groups = test_case[
            "expected_groups"
        ]

        old_evaluation = (
            evaluate_v2_chunk_set(
                result=config["old"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        v2_evaluation = (
            evaluate_v2_chunk_set(
                result=config["v2"],
                question=question,
                expected_groups=(
                    expected_groups
                ),
            )
        )

        old_rank = old_evaluation[
            "complete_evidence_rank"
        ]

        v2_rank = v2_evaluation[
            "complete_evidence_rank"
        ]

        if (
            v2_rank is not None
            and (
                old_rank is None
                or v2_rank < old_rank
            )
        ):
            winner = "V2"

        elif (
            old_rank is not None
            and (
                v2_rank is None
                or old_rank < v2_rank
            )
        ):
            winner = "SEED"

        else:
            winner = "TIE"

        CELL_20B_RESULTS.append(
            {
                "document": (
                    document_label
                ),
                "question": question,
                "seed_rank": old_rank,
                "v2_rank": v2_rank,
                "winner": winner,
                "seed_top_6": (
                    old_evaluation[
                        "top_6_context_passed"
                    ]
                ),
                "v2_top_6": (
                    v2_evaluation[
                        "top_6_context_passed"
                    ]
                ),
            }
        )

        old_rank_label = (
            str(old_rank)
            if old_rank is not None
            else ">20"
        )

        v2_rank_label = (
            str(v2_rank)
            if v2_rank is not None
            else ">20"
        )

        print(
            f"\nQuestion: {question}"
        )

        print(
            f"   Seed complete-evidence rank: "
            f"{old_rank_label}"
        )

        print(
            f"   V2 complete-evidence rank: "
            f"{v2_rank_label}"
        )

        print(
            f"   Seed top-6 context pass: "
            f"{old_evaluation['top_6_context_passed']}"
        )

        print(
            f"   V2 top-6 context pass: "
            f"{v2_evaluation['top_6_context_passed']}"
        )

        print(
            f"   Winner: {winner}"
        )

        seed_top = old_evaluation[
            "retrieved"
        ][0]

        v2_top = v2_evaluation[
            "retrieved"
        ][0]

        print(
            "   Seed top result:"
        )

        print(
            f"      dense={seed_top['dense_rank']} "
            f"bm25={seed_top['lexical_rank']} "
            f"section={seed_top['section']}"
        )

        print(
            "      "
            + " ".join(
                seed_top["text"].split()
            )[:170]
            + "..."
        )

        print(
            "   V2 top result:"
        )

        print(
            f"      dense={v2_top['dense_rank']} "
            f"bm25={v2_top['lexical_rank']} "
            f"section={v2_top['section']}"
        )

        print(
            "      "
            + " ".join(
                v2_top["text"].split()
            )[:170]
            + "..."
        )


# ---------------------------------------------------------------------
# 7. Summary and approval gate
# ---------------------------------------------------------------------

v2_wins = sum(
    result["winner"] == "V2"
    for result in CELL_20B_RESULTS
)

seed_wins = sum(
    result["winner"] == "SEED"
    for result in CELL_20B_RESULTS
)

ties = sum(
    result["winner"] == "TIE"
    for result in CELL_20B_RESULTS
)

seed_top_6_passes = sum(
    result["seed_top_6"]
    for result in CELL_20B_RESULTS
)

v2_top_6_passes = sum(
    result["v2_top_6"]
    for result in CELL_20B_RESULTS
)

total_tests = len(
    CELL_20B_RESULTS
)

CELL_20B_V2_APPROVED = (
    v2_top_6_passes
    >= seed_top_6_passes
    and v2_wins >= seed_wins
)

print("\n" + "=" * 96)
print("📊 CELL 20B SUMMARY")

print(
    f"Canonical V2 wins: "
    f"{v2_wins}"
)

print(
    f"Validated seed wins: "
    f"{seed_wins}"
)

print(
    f"Ties: "
    f"{ties}"
)

print(
    f"Seed top-6 passes: "
    f"{seed_top_6_passes}"
    f"/{total_tests}"
)

print(
    f"Canonical V2 top-6 passes: "
    f"{v2_top_6_passes}"
    f"/{total_tests}"
)

if CELL_20B_V2_APPROVED:
    print(
        "✅ Canonical V2 matched or exceeded "
        "the validated seed retrieval quality."
    )

    print(
        "➡️ Canonical V2 is eligible to become "
        "the web ingestion chunker."
    )

else:
    print(
        "❌ Canonical V2 still introduced "
        "a measurable retrieval regression."
    )

    print(
        "⛔ Do not patch the web backend yet."
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

⏳ Preparing Canonical V2 Apple: 599 chunks


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

⏳ Preparing Canonical V2 Tesla: 136 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


⚖️ VALIDATED SEED VS CANONICAL V2 — SAME HYBRID RETRIEVER

📄 Apple

Question: What were Apple's total net sales in fiscal year 2025?
   Seed complete-evidence rank: 1
   V2 complete-evidence rank: 1
   Seed top-6 context pass: True
   V2 top-6 context pass: True
   Winner: TIE
   Seed top result:
      dense=3 bm25=11 section=Note 2 - Revenue
      Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 2023 | | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357 | | iPad | 28,023 | 26,6...
   V2 top result:
      dense=3 bm25=11 section=Note 2 - Revenue
      Section: Note 2 - Revenue Table columns: | | 2025 | 2024 | 2023 | | iPhone | $ 209,586 | $ 201,183 | $ 200,583 | | Mac | 33,708 | 29,984 | 29,357 | | iPad | 28,023 | 26,6...

Question: What were Apple's services net sales in fiscal year 2025?
   Seed complete-evidence rank: 7
   V2 complete-evidence rank: 7
   Seed top-6 context pass: False
   V2 top-6 context pass: False
   Winner: TIE
   Seed top r

In [70]:
# VAULTIFY COMPACT - Cell 20C:
# Activate Canonical Chunker V2 for the current web runtime

import vaultify_web_backend


required_names = [
    "canonical_chunk_markdown_v2",
    "CELL_20B_V2_APPROVED",
    "DRY_RUN_RESULTS_V2",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_names)
        + ". Run Cells 20A and 20B first."
    )


if not CELL_20B_V2_APPROVED:
    raise RuntimeError(
        "Canonical V2 did not pass the retrieval approval gate."
    )


# Keep references to the previous runtime chunker
# so that the patch can be inspected or reverted.
PREVIOUS_WEB_CHUNKER = getattr(
    vaultify_web_backend,
    "smart_chunk_markdown",
    None,
)


# Patch the module-level function.
vaultify_web_backend.smart_chunk_markdown = (
    canonical_chunk_markdown_v2
)


# Patch the exact global reference used by ingest_document().
vaultify_web_backend.ingest_document.__globals__[
    "smart_chunk_markdown"
] = canonical_chunk_markdown_v2


# ---------------------------------------------------------------------
# Verify that every backend reference points to Canonical V2
# ---------------------------------------------------------------------

module_patch_ok = (
    vaultify_web_backend.smart_chunk_markdown
    is canonical_chunk_markdown_v2
)

ingestion_patch_ok = (
    vaultify_web_backend
    .ingest_document
    .__globals__[
        "smart_chunk_markdown"
    ]
    is canonical_chunk_markdown_v2
)


if not module_patch_ok:
    raise RuntimeError(
        "The backend module chunker was not patched correctly."
    )

if not ingestion_patch_ok:
    raise RuntimeError(
        "The ingestion function still references the old chunker."
    )


# ---------------------------------------------------------------------
# Lightweight regression smoke test using preserved Markdown
# ---------------------------------------------------------------------

smoke_test_results = []

for preserved_result in (
    DRY_RUN_RESULTS_V2.values()
):
    report = preserved_result[
        "report"
    ]

    smoke_test_results.append(
        {
            "filename": preserved_result[
                "filename"
            ],
            "chunks": report[
                "total_chunks"
            ],
            "maximum_tokens": report[
                "token_max"
            ],
            "oversized": report[
                "oversized"
            ],
            "duplicates": report[
                "exact_duplicates"
            ],
        }
    )


for result in smoke_test_results:
    if result["maximum_tokens"] > 240:
        raise RuntimeError(
            f"Oversized regression in "
            f"{result['filename']}."
        )

    if result["oversized"] != 0:
        raise RuntimeError(
            f"Oversized chunks detected in "
            f"{result['filename']}."
        )

    if result["duplicates"] != 0:
        raise RuntimeError(
            f"Duplicate chunks detected in "
            f"{result['filename']}."
        )


CANONICAL_V2_WEB_PATCH_ACTIVE = True


print("✅ Canonical Chunker V2 installed in the web backend.")
print("✅ ingest_document now uses Canonical V2.")
print("✅ Maximum chunk limit: 240 tokens.")
print("✅ Exact duplicate removal: active.")
print("✅ Table-aware seed-style serialization: active.")

print("\nValidated document results:")

for result in smoke_test_results:
    print(
        f"   {result['filename']}"
        f" — {result['chunks']} chunks"
        f" — max {result['maximum_tokens']} tokens"
        f" — oversized {result['oversized']}"
        f" — duplicates {result['duplicates']}"
    )

print(
    "\n⏭️ No Qdrant points were created, "
    "updated, or deleted."
)

print(
    "⏭️ The next uploaded PDF will use "
    "Canonical Chunker V2."
)

✅ Canonical Chunker V2 installed in the web backend.
✅ ingest_document now uses Canonical V2.
✅ Maximum chunk limit: 240 tokens.
✅ Exact duplicate removal: active.
✅ Table-aware seed-style serialization: active.

Validated document results:
   _10-K-2025-As-Filed.pdf — 599 chunks — max 240 tokens — oversized 0 — duplicates 0
   TSLA-Q4-2025-Update.pdf — 136 chunks — max 240 tokens — oversized 0 — duplicates 0
   nist.ai.100-1 (1).pdf — 183 chunks — max 240 tokens — oversized 0 — duplicates 0

⏭️ No Qdrant points were created, updated, or deleted.
⏭️ The next uploaded PDF will use Canonical Chunker V2.
